# Task 3 — standalone starter notebook

Use this notebook in lab Jupyter, VS Code, or Colab connected to the machine's local runtime. For a fresh folder, the first code cell installs the supporting source files. There is no Git dependency. Restore the permitted Monet/Photo dataset separately.

For an existing trained run, preserve its source, split manifest and runtime configuration. Do not rerun bootstrap, preparation or benchmark cells to update a running experiment. Use the existing notebook's runtime variables and the evaluation/package cells instead.

- Default model: two ResNet-9 generators and two PatchGAN discriminators, trained from scratch, with raw and EMA checkpoints.
- Sections 1–7 set up and benchmark a NEW run. Freeze the full schedule before training.
- Section 8 starts/resumes a background phase. `SESSION_HOURS` controls that phase only; it does not change the frozen learning-rate schedule.
- After a phase finishes, rerun Section 9's FIRST cell to freeze the latest checkpoint and update `SNAPSHOT`.
- Section 15 calculates a quick official FID/MiFID and expected displayed Kaggle score for that frozen snapshot, using only 300 predictions per direction.
- Section 16 creates a full ZIP of the completed snapshot, matching score/image evidence and completed training-history segments. It defaults to the latest completed step.
- Save the executed notebook, checkpoint backup and Kaggle screenshot. A clean source notebook is not executed-run evidence.

Run the selected cells deliberately; do not use Run All during background training. Complete the full local metrics, written analysis and independent human audit before final submission. Synthetic smoke-test outputs are not assignment results. Verify the actual GPU model before training.


In [ ]:
from pathlib import Path
import json
candidates = [Path.cwd(), *Path.cwd().parents]
existing_root = next((p for p in candidates if (p/"task3_gan/Pramod_Dindukurthi/src/train.py").is_file()), None)
DEFAULT_ROOT = existing_root or (Path.cwd() if (Path.cwd()/"task3_gan/data/monet_jpg").exists() else Path.cwd()/"data266_task3_runtime")
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", DEFAULT_ROOT))
FILES = json.loads('{"README.md": "# DATA266 Lab 1 \\u2014 Pair 17\\n\\nThe repository preserves separate Task 1, Task 2, Task 3, report and reproducibility folders.\\nTask 3 now contains a portable PyTorch CycleGAN implementation for Pramod Dindukurthi.\\nTask 1 and Task 2 are not implemented by this change.\\n\\nStart with [Task 3 instructions](task3_gan/Pramod_Dindukurthi/README.md) and\\n[the execution notebook](task3_gan/Pramod_Dindukurthi/src/task3_lab.ipynb).\\nThe notebook runs in lab Jupyter, VS Code, Google-hosted Colab, or Colab connected\\nto the GPU-lab machine. Git is needed only on the laptop used for version control.\\n\\nAfter installing a compatible PyTorch/torchvision pair and the Task 3 requirements,\\nthe one-command CPU integration smoke test is:\\n\\n```bash\\npython task3_gan/Pramod_Dindukurthi/src/smoke_test.py\\n```\\n\\nThis creates only synthetic temporary test artifacts. It tests data splitting,\\nboth architectures, cycle gradients, training, exact CPU checkpoint resume,\\nimage export, metric mathematics and the blinded audit workflow.\\nAdd `--full-metrics` to also test Inception/LPIPS extraction and report generation;\\nthis downloads pretrained evaluation weights on first use (never generation weights).\\n\\nTraining results, human ratings and Kaggle scores must come from real assignment\\nruns. The instructor evaluator is retained under Task 3 reference/ and adapted to\\n`src/evaluate_official.py`; it produces the confirmed ID,FID,MiFID CSV schema.\\n", ".gitignore": "__pycache__/\\n*.py[cod]\\n.ipynb_checkpoints/\\n.DS_Store\\n.venv/\\n.env\\n*.pt\\n*.pth\\n*.zip\\ntask*/data/**\\n!task*/data/.gitkeep\\ntask*/**/data_processed/**\\n!task*/**/data_processed/.gitkeep\\ntask*/**/checkpoints/**\\n!task*/**/checkpoints/.gitkeep\\ntask3_gan/**/outputs/**\\n!task3_gan/**/outputs/.gitkeep\\n!task3_gan/**/outputs/plots/\\n!task3_gan/**/outputs/plots/*.png\\nartifacts/\\nruntime_paths.json\\n", "task3_gan/Pramod_Dindukurthi/README.md": "# Task 3 \\u2014 portable CycleGAN\\n\\n**A = Monet; B = Photo.** A2B means Monet-to-Photo; B2A means Photo-to-Monet.\\nBoth networks are trained from scratch. Baseline: 9-block ResNet generators and\\n70x70 PatchGAN discriminators. Experimental configuration: resize-convolution\\nupsampling. Both configurations track raw and EMA generator weights.\\n\\n## Start in the GPU lab\\n\\n1. Extract the supplied `task3-code.zip` into a project folder on the lab machine.\\n   There is no Git dependency. Keep `CODE_BUNDLE_MANIFEST.json` with the extracted code.\\n2. Use the lab\'s working CUDA PyTorch kernel. Python >=3.10 is required;\\n   the trainer supports both PyTorch 2.1 CUDA AMP and newer AMP namespaces. Verify `torch.cuda.is_available()` before full training.\\n3. Open `src/task3_lab.ipynb` in Jupyter or Colab connected to that lab machine.\\n   For a local runtime, the Colab browser and Jupyter server run on the lab machine;\\n   no Google GPU is involved. The notebook includes a setup check and instructions.\\n   `src/task3_start_here.ipynb` is the standalone starter with embedded supporting\\n   source files. Its bootstrap is for a fresh folder and refuses to replace\\n   different existing code. For an existing training run, use its evaluation/package\\n   cells with the existing runtime paths; do not rerun setup or benchmark cells.\\n4. Set the project root, data root and persistent backup directory in the first cells.\\n   Put raw images in `data_root/monet_jpg/` and `data_root/photo_jpg/`.\\n   Use only instructor-permitted development images; do not mix a hidden test set in.\\n5. Run CPU smoke test, prepare splits, and run a short GPU benchmark. Use its measured\\n   seconds/step to set a realistic full-run step budget and decay start BEFORE training.\\n6. Start training. The notebook defaults to a configurable 12-hour overnight budget\\n   and a four-hour first phase in a detached background job. Use its status cell\\n   to check logs and checkpoints. After evaluating and preserving the first-phase\\n   snapshot, set SESSION_HOURS=8.0 and rerun the launch cell to resume.\\n   TRAIN_HOURS controls the full schedule; SESSION_HOURS only controls this phase.\\n   Do not run evaluation until training has stopped. Machine sleep, shutdown, or lab\\n   session cleanup can still stop the job; verify overnight access before leaving.\\n   Set BACKGROUND_TRAINING=False for foreground training with notebook interrupts.\\n7. Evaluate immutable checkpoint snapshots, compare raw/EMA on the SAME validation\\n   set, select a checkpoint with a written reason, and complete the human audit.\\n\\nFor Colab **hosted** runtimes, copy the whole project into the runtime and mount\\nDrive for backups; opening just the notebook does not install sibling Python files.\\nWith a **local** runtime, `google.colab.drive.mount()` is unavailable. Use a lab\\npersistent disk, external drive, approved network storage, or explicit downloads.\\nSaving the notebook does NOT back up model checkpoints or generated files.\\n\\n## Dependencies and offline preparation\\n\\nPreserve the lab\'s matching torch/torchvision CUDA installation. If absent, install\\nthe matching pair using the lab\'s instructions or https://pytorch.org/get-started/locally/.\\nThen, using the same Python interpreter as the notebook:\\n\\n```bash\\npython -m pip install -r task3_gan/Pramod_Dindukurthi/requirements.txt\\n```\\n\\nFull metrics download the torch-fidelity Inception weights and LPIPS AlexNet weights\\non first use. Pre-download them during setup if lab network access is limited. Set\\n`TORCH_HOME` to an approved persistent cache. These weights are evaluation-only;\\nthey never modify submitted images. The run records installed package versions and\\n`pip freeze`. CUDA mixed precision is optional; CPU smoke tests use float32.\\n\\n## File layout\\n\\n```text\\ntask3_gan/data/{monet_jpg,photo_jpg}/     # shared raw data, not in Git\\ntask3_gan/Pramod_Dindukurthi/\\n  src/task3_lab.ipynb                    # execution notebook; save lab outputs\\n  src/task3_start_here.ipynb             # standalone starter, tracked without outputs\\n  src/Quick_Kaggle_Score.ipynb           # single-cell official score\\n  src/{models,data,train,infer,metrics,checkpointing}.py\\n  src/{preflight,smoke_test,audit,report,select_checkpoint}.py\\n  src/{package_code,record_submission,collect_evidence}.py\\n  src/{quick_kaggle_score,package_completed_run}.py\\n  configs/{baseline,resizeconv}.yaml\\n  requirements.txt\\n  data_processed/splits.json             # runtime-created split manifest\\n  checkpoints/latest_checkpoint.pt       # full recovery checkpoint\\n  checkpoints/periodic/step_*.pt\\n  checkpoints/best_model.pt              # explicitly selected checkpoint\\n  outputs/evaluations/<checkpoint-weight>/  # immutable candidate image exports\\n  outputs/{pred_A2B,pred_B2A}/            # complete selected export\\n  outputs/plots/\\n  outputs/human_audit/\\n  evaluate_local.py\\n  full_metrics_report.csv                # created from actual evaluation\\n  metrics_report.csv                     # same source for general lab rubric\\n  submission.csv                         # only from official evaluator output\\n  reproducibility/{raw_logs,manifests}/   # self-contained run evidence\\n  results.md\\n  failure_analysis.md\\n```\\n\\nDefault `RUN_DIR` is your member folder, matching the requested structure. For a\\nsecond experiment choose a new run directory (e.g. `experiments/resizeconv_seed42`)\\nand keep both runs. `collect_evidence.py` copies the selected run\'s logs/manifests\\ninto the team\'s top-level `reproducibility/` folders without changing originals.\\n\\n## Quick scores and completed-snapshot ZIPs\\n\\nAfter a training segment finishes, rerun only Section 9\'s first snapshot cell,\\nthen run Section 15 to score that frozen snapshot. This uses the instructor\\nevaluator with 300 fixed predictions per direction and saves the actual JPGs and\\nCSV. Existing results are reused only when their checkpoint, weights, image count,\\nexport-manifest hash and evaluator source match. The expected displayed Kaggle\\nscore is `-(FID + MiFID)/2`; an actual upload remains separate.\\n\\nSection 16 creates a full snapshot ZIP under `task3_gan/transfer/`. From the member\\nfolder, the equivalent command is:\\n\\n```bash\\npython src/package_completed_run.py --run-dir . --full --weights ema\\n```\\n\\nThe default selects the highest **completed** step. Add `--step NUMBER` to choose\\nan exact completed checkpoint. Freeze that checkpoint first; this packager never\\nsubstitutes the active `latest_checkpoint.pt`. It includes earlier completed logs\\nwith the same configuration and split, the selected model\'s exports, exact score\\nCSV/provenance, notebooks and checksums. Add `--notebook PATH` to include a saved\\nexecuted notebook outside the member folder. A matching evaluator CSV is also\\ncopied unchanged to the member\'s `submission.csv` inside the ZIP.\\n\\nThe ZIP excludes the shared original dataset and active training files. Restore\\nthe dataset separately on another computer. Its `ARCHIVE_EVIDENCE.json` lists\\nwhether official scores, local validation metrics and a human-audit summary are\\npresent. Missing metrics, ratings and written analysis still need completion;\\npackaging does not complete the assignment. Save the Kaggle result screenshot too.\\n\\n## Terminal equivalents\\n\\nRun from the member folder; all paths may be overridden. Replace `BACKUP_PATH`\\nwith your actual persistent directory; it is a runtime argument, not a committed path.\\n\\n```bash\\npython src/preflight.py --data-root ../data\\npython src/smoke_test.py\\npython src/data.py --data-root ../data --output data_processed/splits.json\\npython src/train.py --config configs/baseline.yaml --data-root ../data --splits data_processed/splits.json --run-dir . --backup-dir BACKUP_PATH\\n```\\n\\nResume the same run with the same config and data:\\n\\n```bash\\npython src/train.py --config configs/baseline.yaml --data-root ../data --splits data_processed/splits.json --run-dir . --backup-dir BACKUP_PATH --resume checkpoints/latest_checkpoint.pt\\n```\\n\\nThe notebook creates a runtime config with benchmark-adjusted step counts. When\\nusing it, resume with that exact saved runtime config, not the original YAML.\\n`--stop-after N` ends cleanly after N additional steps without restarting/changing\\nthe LR schedule. `max_hours` is the wall-time limit PER INVOCATION, not total lifetime.\\n\\n## Recovery and evidence\\n\\nCheckpoints contain four networks, two generator EMA copies, both optimizers,\\nschedulers, AMP scaler, replay pools, Python/NumPy/CPU/CUDA RNG states, completed\\nstep, dataset/config identity and throughput counters. Data sampling/augmentation\\nis keyed to sample index so DataLoader prefetch does not change resumed samples.\\nExact CPU recovery is tested; exact bitwise reproduction across GPU types, package\\nversions or nondeterministic CUDA kernels is not promised.\\n\\nAtomic writes protect against partial files. Every resume creates NEW log segments.\\nFailed/interrupted runs keep the last complete checkpoint; non-finite events are\\nprinted explicitly. Do not overwrite a run by changing its config and resuming.\\nIf a problem needs a changed objective/precision setting, preserve the failed run\\nand deliberately create/document a new experiment.\\n\\nUse an immutable periodic checkpoint for comparisons. `latest_checkpoint.pt` can\\nchange during training. The notebook creates a separate evaluation snapshot before\\ninference. `best_model.pt` is created only after explicit validation selection;\\nthe script never assumes the latest or lowest-cycle-loss model is best.\\n\\n## Metrics and interpretation\\n\\n`evaluate_local.py` computes BOTH directions: local FID, KID, density/coverage,\\ncycle L1, LPIPS(input, translation), LPIPS(input, cycle), and input/translation\\nInception-feature cosine. Inputs and cycles are lossless PNG; translations are\\nfixed-quality RGB JPG. FID references are the fixed held-out target-domain images.\\nKID is unscaled unbiased MMD^2 and can be negative. Its subset standard deviation\\nis NOT a confidence interval. Report real/generated sample counts; small-sample\\nFID is noisy and biased. Density/coverage uses k=5 and requires >5 real images.\\n\\nCycle L1 uses tensors on [0,1] before encoding, not arbitrary unpaired targets.\\nTraining cycle/identity losses in logs use [-1,1] and are unweighted components;\\nthe configured weights are separately recorded. LPIPS input/translation can favor\\nunder-stylization and must be interpreted with style evidence. Inception cosine\\nis a semantic proxy, not proof of content fidelity. Training throughput counts\\nBOTH real-domain images per optimizer step, includes data wait and update time,\\nand excludes checkpoint/preview time. Session wall time is separately recorded.\\nParameter totals exclude EMA duplicate storage. Peak GPU memory is allocated\\nPyTorch GPU memory (not total reserved GPU memory); CPU-only reports use null.\\n\\nThe local metrics use a different Inception implementation from the supplied instructor\\nevaluator. For Kaggle, use `src/evaluate_official.py` and its confirmed `ID,FID,MiFID`\\noutput. The original instructor notebook is preserved in `reference/`. Do not compare\\nthese FID implementations as if they were interchangeable. Follow any separate instructor\\nrequirements for the source input set, count and direction. Unavailable leaderboard scores\\nremain pending. No script uploads to Kaggle automatically.\\n\\n## Blinded human audit\\n\\nAfter checkpoint selection, `audit.py prepare` deterministically samples 15 examples\\nper direction. Give BOTH independent raters only `rater_packet/`, never the private\\nkey, architecture or model score. All three criteria use 1-5, with higher better;\\nartifact score means freedom from artifacts. After each rater fills their own sheet,\\n`audit.py summarize` validates all 30 ratings and computes pooled means and exact\\npercentage agreement per criterion, both directions and overall. This satisfies the\\nrubric\'s allowed percentage-agreement alternative. Rerun the report to include it.\\nActual people must rate actual generated images; smoke-test ratings are synthetic.\\n\\n## Incremental commits and transfer\\n\\nCommit real working milestones on the laptop. Capture `git rev-parse HEAD` before\\ntraining and enter it as `CODE_VERSION`; the run additionally hashes Python source.\\nFor no-Git lab transfer: `python src/package_code.py --output task3-code.zip`.\\nThe ZIP deliberately excludes datasets, secrets, checkpoints and generated results.\\nIt is only a CODE bundle, never a training backup. Save completed raw logs unchanged.\\nRecord checkpoint SHA-256 and persistent retrieval instructions; keep large binaries\\noutside Git. Review executed notebook output for private paths or tokens before push.\\n\\n## References\\n\\n- Zhu et al., CycleGAN: https://arxiv.org/abs/1703.10593\\n- Author reference: https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix\\n- Resize-convolution: https://distill.pub/2016/deconv-checkerboard/\\n- EMA: https://arxiv.org/abs/1806.04498\\n- KID: https://arxiv.org/abs/1801.01401\\n- Density/coverage: https://arxiv.org/abs/2002.09797\\n- LPIPS: https://github.com/richzhang/PerceptualSimilarity\\n- Inception implementation: https://github.com/toshas/torch-fidelity\\n- Competition: https://www.kaggle.com/competitions/data-266-fall-2026-gan-image-style-transfer/overview\\n\\n\\n## Instructor evaluator now available\\nThe supplied original is preserved at `reference/Part3_Evaluation_Script.ipynb`.\\n`src/evaluate_official.py` preserves its torchvision Inception weights, normalization,\\nfirst-300 sorted file selection, matched counts and index-wise cosine-distance formula.\\nIt writes the confirmed `ID,FID,MiFID` schema. Use the local metrics for the broader\\nrubric and fixed held-out validation; report official-protocol metrics separately.\\nThe official reference folders include training images; this is not held-out evaluation.\\nDo not reorder or cherry-pick generated images to influence its index-based score.\\n\\n```bash\\n# Official scoring only: same fixed first 300 predictions as a complete all-input export.\\npython src/infer.py --checkpoint checkpoints/evaluation/YOUR_SNAPSHOT.pt --data-root ../data --splits data_processed/splits.json --output outputs/official/YOUR_EXPORT --split all --weights ema --max-per-domain 300 --predictions-only --device cuda\\npython src/evaluate_official.py --data-root ../data --export-dir outputs/official/YOUR_EXPORT --output submission.csv\\n```\\nThis command does not upload anything. The evaluator itself does not specify which\\nsource input set must be translated; follow instructor input-set instructions. Keep\\ninference ordering and sample selection fixed before evaluating.\\n\\nThe fast official export skips input/cycle PNGs and reconstruction inference. It records\\nthe fixed prefix and hashes in the export manifest and prints progress every 100 images.\\nDo not use it for full local validation metrics, which require the complete validation\\nexport with input and cycle images. If an older full export is already running, let it\\nfinish if progressing; the evaluator uses the same first-300 prediction prefix.\\n", "task3_gan/Pramod_Dindukurthi/requirements.txt": "# Install a matching torch/torchvision pair for your machine FIRST.\\n# Preserve the lab\'s working CUDA PyTorch installation.\\nnumpy>=1.24,<3\\nPillow>=10,<13\\nPyYAML>=6,<7\\nscipy>=1.11,<2\\nmatplotlib>=3.7,<4\\ntorch-fidelity==0.3.0\\nlpips==0.1.4\\ntqdm>=4.65,<5\\n", "task3_gan/Pramod_Dindukurthi/evaluate_local.py": "\\"\\"\\"Full local Task 3 evaluation. Never labels a diagnostic as official MiFID.\\"\\"\\"\\nimport argparse\\nimport csv\\nimport json\\nfrom pathlib import Path\\nimport sys\\nimport warnings\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parent/\\"src\\"))\\nimport numpy as np\\nfrom common import device_for, json_write, sha256, utc_now\\nfrom metrics import InceptionFeatures, cosine, density_coverage, fid, kid, lpips_pairs\\n\\n\\ndef evaluate(export_dir, run_dir=None, device=\\"auto\\", batch_size=16, seed=42, subsets=50, k=5):\\n    export_dir = Path(export_dir)\\n    meta = json.loads((export_dir/\\"export_manifest.json\\").read_text())\\n    if meta.get(\\"predictions_only\\"):\\n        raise ValueError(\\"Predictions-only exports are for official scoring. Full local metrics require a validation export with inputs and cycle reconstructions.\\")\\n    if meta[\\"split\\"] != \\"val\\":\\n        warnings.warn(\\"This is NOT held-out validation; report its split honestly and do not compare with validation runs.\\")\\n    records = meta[\\"records\\"]\\n    for row in records:\\n        for field in [\\"input\\", \\"prediction\\", \\"cycle\\"]:\\n            if sha256(export_dir/row[field]) != row[field+\\"_sha256\\"]:\\n                raise ValueError(\\"Export was modified after inference: \\" + row[field])\\n    device = device_for(device)\\n    cache = Path(run_dir)/\\"data_processed\\"/\\"feature_cache\\" if run_dir else export_dir/\\"feature_cache\\"\\n    extractor = InceptionFeatures(device, cache)\\n    feats = {}\\n    grouped = {d: [r for r in records if r[\\"domain\\"] == d] for d in [\\"A\\", \\"B\\"]}\\n    for d, group in grouped.items():\\n        for field in [\\"input\\", \\"prediction\\"]:\\n            print(\\"Extracting\\", d, field, len(group), flush=True)\\n            feats[d, field] = extractor([export_dir/r[field] for r in group], batch_size)\\n    results, lpips_model = [], None\\n    for d, target in [(\\"A\\", \\"B\\"), (\\"B\\", \\"A\\")]:\\n        group = grouped[d]\\n        real, fake = feats[target, \\"input\\"], feats[d, \\"prediction\\"]\\n        if min(len(real), len(fake)) < 2048:\\n            warnings.warn(\\"Small-sample FID is biased/unstable. Report sample counts; use KID and visuals too.\\")\\n        paths = {field: [export_dir/r[field] for r in group] for field in [\\"input\\", \\"prediction\\", \\"cycle\\"]}\\n        lp, lpips_model = lpips_pairs(paths[\\"input\\"], paths[\\"prediction\\"], device, batch_size, lpips_model)\\n        lpc, _ = lpips_pairs(paths[\\"input\\"], paths[\\"cycle\\"], device, batch_size, lpips_model)\\n        row = {\\"direction\\": \\"A2B\\" if d == \\"A\\" else \\"B2A\\", \\"source\\": \\"Monet\\" if d == \\"A\\" else \\"Photo\\",\\n               \\"target\\": \\"Photo\\" if d == \\"A\\" else \\"Monet\\", \\"step\\": meta[\\"step\\"], \\"weights\\": meta[\\"weights\\"],\\n               \\"split\\": meta[\\"split\\"], \\"checkpoint_sha256\\": meta[\\"checkpoint_sha256\\"],\\n               \\"n_real\\": len(real), \\"n_generated\\": len(fake), \\"FID\\": fid(real, fake),\\n               **kid(real, fake, seed, subsets), **density_coverage(real, fake, k),\\n               \\"cycle_L1_tensor_0_1\\": float(np.mean([r[\\"cycle_L1_tensor_0_1\\"] for r in group])),\\n               \\"LPIPS_input_translation\\": float(lp.mean()), \\"LPIPS_input_cycle\\": float(lpc.mean()),\\n               \\"content_cosine_inception2048\\": float(cosine(feats[d, \\"input\\"], fake).mean()),\\n               \\"human_style_mean\\": \\"pending\\", \\"human_content_mean\\": \\"pending\\", \\"human_artifacts_mean\\": \\"pending\\",\\n               \\"human_agreement\\": \\"pending\\", \\"kaggle_public_score\\": \\"pending\\",\\n               \\"kaggle_private_score\\": \\"pending\\", \\"kaggle_rank\\": \\"pending\\"}\\n        if run_dir and (Path(run_dir)/\\"training_summary.json\\").exists():\\n            summary = json.loads((Path(run_dir)/\\"training_summary.json\\").read_text())\\n            # Training totals may describe a later step; identify the summary step explicitly.\\n            row.update(training_summary_step=summary[\\"end_step\\"], training_seconds=summary[\\"training_seconds\\"],\\n                       images_per_second=summary[\\"source_images_per_second\\"],\\n                       peak_gpu_memory_bytes=summary[\\"peak_gpu_memory_bytes\\"],\\n                       parameter_count=sum(summary[\\"parameters\\"].values()),\\n                       nan_count=summary[\\"nan_count\\"], inf_count=summary[\\"inf_count\\"])\\n        results.append(row)\\n        print(json.dumps(row), flush=True)\\n    output = export_dir/\\"full_metrics_report.csv\\"\\n    with open(output, \\"w\\", newline=\\"\\") as f:\\n        w = csv.DictWriter(f, fieldnames=list(results[0])); w.writeheader(); w.writerows(results)\\n    protocol = {\\"created_utc\\": utc_now(), \\"inception\\": \\"torch-fidelity inception-v3-compat 2048\\",\\n                \\"torch_fidelity\\": extractor.version, \\"feature_weights_sha256\\": extractor.weights_hash,\\n                \\"LPIPS\\": \\"alex v0.1; input/translation and input/cycle separately; [-1,1]\\",\\n                \\"FID\\": \\"symmetric covariance PSD square root; uncorrected finite-sample estimate\\",\\n                \\"KID\\": \\"unscaled unbiased MMD^2; subset std is not a confidence interval\\",\\n                \\"density_coverage_k\\": k, \\"seed\\": seed, \\"export_manifest_sha256\\": sha256(export_dir/\\"export_manifest.json\\"),\\n                \\"official_kaggle_equivalent\\": False,\\n                \\"selection_diagnostic_mean_FID\\": float(np.mean([r[\\"FID\\"] for r in results]))}\\n    json_write(export_dir/\\"evaluation_protocol.json\\", protocol)\\n    return results\\n\\n\\nif __name__ == \\"__main__\\":\\n    p = argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\\"--export-dir\\", required=True)\\n    p.add_argument(\\"--run-dir\\")\\n    p.add_argument(\\"--device\\", default=\\"auto\\")\\n    p.add_argument(\\"--batch-size\\", type=int, default=16)\\n    p.add_argument(\\"--seed\\", type=int, default=42)\\n    p.add_argument(\\"--subsets\\", type=int, default=50)\\n    p.add_argument(\\"--k\\", type=int, default=5)\\n    a = p.parse_args()\\n    evaluate(a.export_dir, a.run_dir, a.device, a.batch_size, a.seed, a.subsets, a.k)\\n", "task3_gan/Pramod_Dindukurthi/results.md": "# Task 3 results\\n\\nStatus: no assignment training run has been completed by the code authoring workflow.\\n\\nRun `src/report.py` after inference/evaluation to create `results_draft.md`, then use actual\\nevidence to complete this file. Synthetic smoke-test numbers are not assignment results.\\n\\nDocument architecture, hyperparameters, preprocessing, dataset/split, hardware, code version,\\ncheckpoint hash, all metrics, human audit, Kaggle score/rank, limitations, and teammate comparison.\\n\\nThe supplied instructor evaluator is available as src/evaluate_official.py. Its CSV schema is ID,FID,MiFID. Scores remain pending actual inference/evaluation.\\n", "task3_gan/Pramod_Dindukurthi/failure_analysis.md": "# Task 3 failure analysis\\n\\nComplete after inspecting the fixed validation grids and independently collected human ratings.\\n\\n| Sample ID / direction | Observed failure | Evidence file | Hypothesis | Testable fix |\\n|---|---|---|---|---|\\n\\nLook for insufficient style transfer, distorted objects, checkerboard artifacts, halos,\\ncolor shifts and repeated textures. Include representative weaknesses, not just attractive images.\\nExplain why low cycle loss does not prove semantic preservation or target-domain realism.\\n", "task3_gan/Pramod_Dindukurthi/src/audit.py": "\\"\\"\\"Fixed 30-image audit, independent rater sheets, and exact percentage agreement.\\"\\"\\"\\nimport argparse\\nimport csv\\nimport json\\nfrom pathlib import Path\\nimport random\\nimport shutil\\n\\nfrom PIL import Image\\nimport numpy as np\\n\\nfrom common import json_write, sha256\\nfrom data import rgb\\n\\n\\nCRITERIA = [\\"style\\", \\"content\\", \\"artifacts\\"]\\n\\n\\ndef prepare(export_dir, output, seed=266):\\n    export_dir, output = Path(export_dir), Path(output)\\n    if output.exists():\\n        raise FileExistsError(\\"Audit exists; keep fixed samples and do not overwrite ratings\\")\\n    meta = json.loads((export_dir/\\"export_manifest.json\\").read_text())\\n    if meta[\\"split\\"] != \\"val\\":\\n        raise ValueError(\\"Use held-out validation images for the human audit\\")\\n    rng, chosen = random.Random(seed), []\\n    for direction in [\\"A2B\\", \\"B2A\\"]:\\n        group = [r for r in meta[\\"records\\"] if r[\\"direction\\"] == direction]\\n        if len(group) < 15:\\n            raise ValueError(f\\"Need at least 15 fixed validation samples in {direction} for the planned audit\\")\\n        chosen.extend(rng.sample(group, 15))\\n    rng.shuffle(chosen)\\n    (output/\\"rater_packet\\").mkdir(parents=True)\\n    key, rows = [], []\\n    for i, row in enumerate(chosen, 1):\\n        item = f\\"sample_{i:02d}\\"\\n        source, pred = rgb(export_dir/row[\\"input\\"]), rgb(export_dir/row[\\"prediction\\"])\\n        canvas = Image.new(\\"RGB\\", (source.width*2, source.height))\\n        canvas.paste(source, (0, 0)); canvas.paste(pred, (source.width, 0))\\n        canvas.save(output/\\"rater_packet\\"/(item+\\".png\\"))\\n        rows.append({\\"sample_id\\": item, \\"target_domain\\": \\"Photo\\" if row[\\"domain\\"] == \\"A\\" else \\"Monet\\",\\n                     \\"style\\": \\"\\", \\"content\\": \\"\\", \\"artifacts\\": \\"\\", \\"notes\\": \\"\\"})\\n        key.append({\\"sample_id\\": item, **row})\\n    for rater in [1, 2]:\\n        with open(output/\\"rater_packet\\"/f\\"rater_{rater}.csv\\", \\"w\\", newline=\\"\\") as f:\\n            w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)\\n    (output/\\"rater_packet\\"/\\"INSTRUCTIONS.md\\").write_text(\\n        \\"# Independent blinded audit\\\\n\\\\nLeft: input. Right: translation. Target domain is in the CSV.\\\\n\\"\\n        \\"Rate independently before discussing with the other rater. Do not disclose architecture, \\"\\n        \\"checkpoint, model scores or model identity. Each rater fills their own file.\\\\n\\\\n\\"\\n        \\"Use integers 1-5. Style: 1=no target style, 3=partial, 5=convincing target style. \\"\\n        \\"Content: 1=major scene loss, 3=partly retained, 5=structure and objects retained. \\"\\n        \\"Artifacts: 1=severe defects, 3=noticeable defects, 5=no visible defects. \\"\\n        \\"For ALL criteria higher is better. 2 and 4 are intermediate. Add observations in notes.\\\\n\\")\\n    json_write(output/\\"private_key.json\\", {\\"seed\\": seed, \\"checkpoint_sha256\\": meta[\\"checkpoint_sha256\\"],\\n                                         \\"export_manifest_sha256\\": sha256(export_dir/\\"export_manifest.json\\"),\\n                                         \\"samples\\": key})\\n    print(\\"Give raters only\\", output/\\"rater_packet\\", \\"Keep private_key.json away from raters.\\")\\n\\n\\ndef summarize(output):\\n    output = Path(output)\\n    key = json.loads((output/\\"private_key.json\\").read_text())\\n    ids = {r[\\"sample_id\\"] for r in key[\\"samples\\"]}\\n    ratings = []\\n    for n in [1, 2]:\\n        with open(output/\\"rater_packet\\"/f\\"rater_{n}.csv\\") as f:\\n            rows = list(csv.DictReader(f))\\n        mapping = {r[\\"sample_id\\"]: r for r in rows}\\n        if set(mapping) != ids or len(rows) != len(ids):\\n            raise ValueError(\\"Each rater must rate exactly the 30 unique sample IDs\\")\\n        for row in rows:\\n            for c in CRITERIA:\\n                if row[c] not in {\\"1\\", \\"2\\", \\"3\\", \\"4\\", \\"5\\"}:\\n                    raise ValueError(f\\"Missing/invalid rating: rater {n}, {row[\'sample_id\']}, {c}\\")\\n        ratings.append(mapping)\\n    result = []\\n    for direction in [\\"A2B\\", \\"B2A\\", \\"both\\"]:\\n        group = [r[\\"sample_id\\"] for r in key[\\"samples\\"] if direction == \\"both\\" or r[\\"direction\\"] == direction]\\n        for c in CRITERIA:\\n            a = np.array([int(ratings[0][i][c]) for i in group])\\n            b = np.array([int(ratings[1][i][c]) for i in group])\\n            result.append({\\"direction\\": direction, \\"criterion\\": c, \\"n_samples\\": len(group),\\n                           \\"rater1_mean\\": float(a.mean()), \\"rater2_mean\\": float(b.mean()),\\n                           \\"pooled_mean\\": float(np.concatenate([a, b]).mean()),\\n                           \\"exact_agreement_percent\\": float(100*np.mean(a == b))})\\n    json_write(output/\\"audit_summary.json\\", {\\"checkpoint_sha256\\": key[\\"checkpoint_sha256\\"], \\"results\\": result})\\n    print(json.dumps(result, indent=2))\\n    return result\\n\\n\\nif __name__ == \\"__main__\\":\\n    p = argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\\"action\\", choices=[\\"prepare\\", \\"summarize\\"])\\n    p.add_argument(\\"--export-dir\\")\\n    p.add_argument(\\"--output\\", required=True)\\n    p.add_argument(\\"--seed\\", type=int, default=266)\\n    a = p.parse_args()\\n    if a.action == \\"prepare\\":\\n        if not a.export_dir:\\n            p.error(\\"prepare requires --export-dir\\")\\n        prepare(a.export_dir, a.output, a.seed)\\n    else:\\n        summarize(a.output)\\n", "task3_gan/Pramod_Dindukurthi/src/background.py": "\\"\\"\\"Detach a training command from the notebook; retain logs and exit status.\\"\\"\\"\\nimport argparse\\nfrom contextlib import contextmanager\\nfrom datetime import datetime, timezone\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport subprocess\\nimport sys\\n\\n\\n@contextmanager\\ndef run_lock(path):\\n    # Kernel-managed lock is released even if the supervisor crashes.\\n    with open(path, \\"a+b\\") as handle:\\n        if os.name == \\"nt\\":\\n            import msvcrt\\n            handle.seek(0)\\n            if not handle.read(1):\\n                handle.write(b\\"0\\")\\n                handle.flush()\\n            handle.seek(0)\\n            msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)\\n        else:\\n            import fcntl\\n            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)\\n        try:\\n            yield\\n        finally:\\n            if os.name == \\"nt\\":\\n                handle.seek(0)\\n                msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)\\n            else:\\n                fcntl.flock(handle, fcntl.LOCK_UN)\\n\\n\\ndef write_status(path, value):\\n    tmp = path.with_suffix(\\".tmp\\")\\n    tmp.write_text(json.dumps(value, indent=2))\\n    os.replace(tmp, path)\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"action\\", choices=[\\"launch\\", \\"status\\", \\"worker\\"])\\n    parser.add_argument(\\"--run-dir\\", required=True)\\n    args, command = parser.parse_known_args()\\n    if command[:1] == [\\"--\\"]:\\n        command = command[1:]\\n    folder = Path(args.run_dir).resolve()/\\"reproducibility\\"/\\"background\\"\\n    folder.mkdir(parents=True, exist_ok=True)\\n    lock, status, log = folder/\\"run.lock\\", folder/\\"status.json\\", folder/\\"console.log\\"\\n    if args.action == \\"status\\":\\n        try:\\n            with run_lock(lock):\\n                active = False\\n        except OSError:\\n            active = True\\n        print(\\"Background supervisor active:\\", active)\\n        print(status.read_text() if status.exists() else \\"No recorded job.\\")\\n        if not active and status.exists() and json.loads(status.read_text()).get(\\"state\\") == \\"running\\":\\n            print(\\"Supervisor exited unexpectedly. Inspect logs/checkpoint before resuming.\\")\\n        if log.exists():\\n            with open(log, \\"rb\\") as f:\\n                f.seek(max(0, log.stat().st_size - 12000))\\n                print(f.read().decode(errors=\\"replace\\"))\\n        return 0\\n    if not command:\\n        parser.error(\\"Provide the training command after --\\")\\n    if args.action == \\"launch\\":\\n        options = {\\"start_new_session\\": True} if os.name != \\"nt\\" else {\\n            \\"creationflags\\": subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP}\\n        with open(log, \\"ab\\", buffering=0) as output:\\n            process = subprocess.Popen(\\n                [sys.executable, \\"-u\\", str(Path(__file__).resolve()), \\"worker\\",\\n                 \\"--run-dir\\", str(Path(args.run_dir).resolve()), \\"--\\", *command],\\n                stdin=subprocess.DEVNULL, stdout=output, stderr=subprocess.STDOUT,\\n                close_fds=True, **options)\\n        try:\\n            result = process.wait(timeout=1)\\n        except subprocess.TimeoutExpired:\\n            result = 0\\n        print(\\"Supervisor PID:\\", process.pid, \\"Log:\\", log)\\n        print(\\"Check status and confirm checkpoint progress before leaving.\\")\\n        return result\\n    try:\\n        with run_lock(lock):\\n            info = {\\"state\\": \\"running\\", \\"supervisor_pid\\": os.getpid(),\\n                    \\"started_utc\\": datetime.now(timezone.utc).isoformat(), \\"command\\": command}\\n            write_status(status, info)\\n            try:\\n                child = subprocess.Popen(command, stdin=subprocess.DEVNULL)\\n                info[\\"training_pid\\"] = child.pid\\n                write_status(status, info)\\n                result = child.wait()\\n            except Exception as exc:\\n                info[\\"error\\"] = repr(exc)\\n                result = 1\\n            info.update(state=\\"finished\\" if result == 0 else \\"failed\\", exit_code=result,\\n                        ended_utc=datetime.now(timezone.utc).isoformat())\\n            write_status(status, info)\\n            return result\\n    except OSError as exc:\\n        print(\\"Cannot acquire background run lock; another job may be running:\\", exc, flush=True)\\n        return 2\\n\\n\\nif __name__ == \\"__main__\\":\\n    sys.exit(main())\\n", "task3_gan/Pramod_Dindukurthi/src/checkpointing.py": "\\"\\"\\"Atomic recovery checkpoints. Load only checkpoints you trust.\\"\\"\\"\\nimport copy\\nimport os\\nfrom pathlib import Path\\nimport random\\nimport shutil\\n\\nimport numpy as np\\nimport torch\\n\\n\\nclass ReplayPool:\\n    def __init__(self, capacity):\\n        self.capacity, self.images = capacity, []\\n\\n    def query(self, batch):\\n        result = []\\n        for image in batch.detach():\\n            image = image.unsqueeze(0)\\n            if self.capacity == 0:\\n                result.append(image)\\n            elif len(self.images) < self.capacity:\\n                self.images.append(image.cpu().clone())\\n                result.append(image)\\n            elif random.random() < 0.5:\\n                i = random.randrange(self.capacity)\\n                result.append(self.images[i].to(image.device).clone())\\n                self.images[i] = image.cpu().clone()\\n            else:\\n                result.append(image)\\n        return torch.cat(result)\\n\\n\\ndef make_ema(model):\\n    ema = copy.deepcopy(model).eval()\\n    ema.requires_grad_(False)\\n    return ema\\n\\n\\n@torch.no_grad()\\ndef update_ema(ema, model, decay):\\n    for target, source in zip(ema.parameters(), model.parameters()):\\n        target.lerp_(source, 1 - decay)\\n    for target, source in zip(ema.buffers(), model.buffers()):\\n        target.copy_(source)\\n\\n\\ndef rng_state():\\n    return {\\"python\\": random.getstate(), \\"numpy\\": np.random.get_state(),\\n            \\"torch\\": torch.get_rng_state(),\\n            \\"cuda\\": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}\\n\\n\\ndef restore_rng(state):\\n    random.setstate(state[\\"python\\"])\\n    np.random.set_state(state[\\"numpy\\"])\\n    torch.set_rng_state(state[\\"torch\\"].cpu())\\n    if state[\\"cuda\\"] is not None and torch.cuda.is_available():\\n        if len(state[\\"cuda\\"]) != torch.cuda.device_count():\\n            raise ValueError(\\"CUDA device count changed; exact resume cannot restore all RNG states\\")\\n        torch.cuda.set_rng_state_all([s.cpu() for s in state[\\"cuda\\"]])\\n\\n\\ndef atomic_save(state, path):\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    tmp = path.with_suffix(\\".tmp\\")\\n    with open(tmp, \\"wb\\") as f:\\n        torch.save(state, f)\\n        f.flush()\\n        os.fsync(f.fileno())\\n    os.replace(tmp, path)\\n\\n\\ndef backup_file(source, destination):\\n    source, destination = Path(source), Path(destination)\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    tmp = destination.with_suffix(destination.suffix + \\".tmp\\")\\n    shutil.copy2(source, tmp)\\n    os.replace(tmp, destination)\\n\\n\\ndef load_checkpoint(path, device=\\"cpu\\"):\\n    # Full recovery includes Python/NumPy RNG state; weights_only cannot deserialize it.\\n    return torch.load(path, map_location=device, weights_only=False)\\n", "task3_gan/Pramod_Dindukurthi/src/collect_evidence.py": "\\"\\"\\"Copy immutable run evidence into the team\'s required top-level folders.\\"\\"\\"\\nimport argparse\\nfrom pathlib import Path\\nimport re\\nimport shutil\\n\\nfrom common import sha256\\n\\n\\ndef collect(run_dir, repo_root, run_id):\\n    if not re.fullmatch(r\\"[A-Za-z0-9_-]+\\", run_id):\\n        raise ValueError(\\"run-id must contain only letters, digits, underscores or hyphens\\")\\n    run_dir, repo_root = Path(run_dir), Path(repo_root)\\n    for kind in [\\"raw_logs\\", \\"manifests\\"]:\\n        for source in (run_dir/\\"reproducibility\\"/kind).glob(\\"*\\"):\\n            target = repo_root/\\"reproducibility\\"/kind/(\\"task3_\\"+run_id)/source.name\\n            target.parent.mkdir(parents=True, exist_ok=True)\\n            if target.exists() and sha256(target) != sha256(source):\\n                raise ValueError(f\\"Refusing to overwrite different evidence: {target}\\")\\n            if not target.exists():\\n                shutil.copy2(source, target)\\n    splits = run_dir/\\"data_processed\\"/\\"splits.json\\"\\n    if splits.exists():\\n        target = repo_root/\\"reproducibility\\"/\\"manifests\\"/(\\"task3_\\"+run_id)/\\"splits.json\\"\\n        if target.exists() and sha256(target) != sha256(splits):\\n            raise ValueError(\\"Existing split evidence differs\\")\\n        shutil.copy2(splits, target)\\n    print(\\"Collected run evidence; checkpoint files must be backed up separately.\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    p = argparse.ArgumentParser(description=__doc__)\\n    for key in [\\"run-dir\\", \\"repo-root\\", \\"run-id\\"]:\\n        p.add_argument(\\"--\\"+key, required=True)\\n    a = p.parse_args()\\n    collect(a.run_dir, a.repo_root, a.run_id)\\n", "task3_gan/Pramod_Dindukurthi/src/common.py": "\\"\\"\\"Portable paths, provenance, atomic writes and hardware disclosure.\\"\\"\\"\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport platform\\nimport random\\nimport subprocess\\nimport sys\\nfrom datetime import datetime, timezone\\n\\nimport numpy as np\\nimport torch\\nimport yaml\\n\\n\\ndef utc_now():\\n    return datetime.now(timezone.utc).isoformat()\\n\\n\\ndef sha256(path):\\n    h = hashlib.sha256()\\n    with open(path, \\"rb\\") as f:\\n        for chunk in iter(lambda: f.read(1024 * 1024), b\\"\\"):\\n            h.update(chunk)\\n    return h.hexdigest()\\n\\n\\ndef json_write(path, obj):\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    tmp = path.with_suffix(path.suffix + \\".tmp\\")\\n    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False) + \\"\\\\n\\")\\n    os.replace(tmp, path)\\n\\n\\ndef read_config(path):\\n    config = yaml.safe_load(Path(path).read_text())\\n    t = config[\\"train\\"]\\n    if not 0 <= t[\\"decay_start\\"] < t[\\"max_steps\\"]:\\n        raise ValueError(\\"Require 0 <= decay_start < max_steps\\")\\n    if config[\\"image_size\\"] % 4 or config[\\"image_size\\"] < 32:\\n        raise ValueError(\\"image_size must be a multiple of 4 and >=32\\")\\n    if config[\\"load_size\\"] < config[\\"image_size\\"]:\\n        raise ValueError(\\"load_size must be >= image_size\\")\\n    if not 0 <= t[\\"ema_decay\\"] < 1 or t[\\"batch_size\\"] < 1:\\n        raise ValueError(\\"Invalid EMA decay or batch size\\")\\n    return config\\n\\n\\ndef seed_all(seed, deterministic=True):\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    if torch.cuda.is_available():\\n        torch.cuda.manual_seed_all(seed)\\n    torch.backends.cudnn.benchmark = not deterministic\\n    torch.backends.cudnn.deterministic = deterministic\\n    torch.use_deterministic_algorithms(deterministic, warn_only=True)\\n\\n\\ndef device_for(value=\\"auto\\"):\\n    if value == \\"auto\\":\\n        value = \\"cuda\\" if torch.cuda.is_available() else \\"cpu\\"\\n    device = torch.device(value)\\n    if device.type == \\"cuda\\" and not torch.cuda.is_available():\\n        raise RuntimeError(\\"CUDA unavailable in this kernel. Select the lab\'s CUDA PyTorch environment.\\")\\n    return device\\n\\n\\ndef source_hash():\\n    root = Path(__file__).resolve().parents[1]\\n    h = hashlib.sha256()\\n    for p in sorted(root.rglob(\\"*.py\\")):\\n        h.update(str(p.relative_to(root)).encode())\\n        h.update(p.read_bytes())\\n    return h.hexdigest()\\n\\n\\ndef manifest(config, code_version):\\n    versions = {}\\n    for name in [\\"torch\\", \\"torchvision\\", \\"numpy\\", \\"Pillow\\", \\"scipy\\", \\"PyYAML\\",\\n                 \\"matplotlib\\", \\"torch-fidelity\\", \\"lpips\\"]:\\n        try:\\n            versions[name] = importlib.metadata.version(name)\\n        except importlib.metadata.PackageNotFoundError:\\n            versions[name] = \\"not installed\\"\\n    gpu = []\\n    if torch.cuda.is_available():\\n        gpu = [{\\"name\\": torch.cuda.get_device_name(i),\\n                \\"memory_bytes\\": torch.cuda.get_device_properties(i).total_memory}\\n               for i in range(torch.cuda.device_count())]\\n    cpu = platform.processor() or platform.machine()\\n    if Path(\\"/proc/cpuinfo\\").exists():\\n        cpu = next((s.split(\\":\\", 1)[1].strip() for s in Path(\\"/proc/cpuinfo\\").read_text().splitlines()\\n                    if s.startswith(\\"model name\\")), cpu)\\n    return {\\"created_utc\\": utc_now(), \\"python\\": sys.version, \\"platform\\": platform.platform(),\\n            \\"cpu\\": cpu, \\"cpu_count\\": os.cpu_count(), \\"gpu\\": gpu,\\n            \\"cuda\\": torch.version.cuda, \\"cudnn\\": torch.backends.cudnn.version(),\\n            \\"packages\\": versions, \\"config\\": config, \\"code_version\\": code_version,\\n            \\"source_sha256\\": source_hash(), \\"domain_A\\": \\"Monet\\", \\"domain_B\\": \\"Photo\\"}\\n\\n\\nclass Tee:\\n    def __init__(self, stream, file):\\n        self.stream, self.file = stream, file\\n\\n    def write(self, text):\\n        self.stream.write(text)\\n        self.file.write(text)\\n        self.file.flush()\\n\\n    def flush(self):\\n        self.stream.flush()\\n        self.file.flush()\\n\\n\\ndef save_environment(path):\\n    result = subprocess.run([sys.executable, \\"-m\\", \\"pip\\", \\"freeze\\"],\\n                            capture_output=True, text=True)\\n    Path(path).write_text(result.stdout if result.returncode == 0 else result.stderr)\\n", "task3_gan/Pramod_Dindukurthi/src/data.py": "\\"\\"\\"Data audit and stateless, reproducible unpaired sampling.\\n\\nPreparation never uses a hidden test set. Supply only permitted development data.\\nExact decoded-pixel duplicates are grouped before splitting; near duplicates require review.\\n\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nimport random\\nfrom pathlib import Path\\n\\nimport numpy as np\\nfrom PIL import Image, ImageOps\\nimport torch\\nfrom torch.utils.data import Dataset\\n\\nfrom common import json_write, sha256\\n\\n\\ndef rgb(path):\\n    with Image.open(path) as image:\\n        return ImageOps.exif_transpose(image).convert(\\"RGB\\")\\n\\n\\ndef transform(image, size, load_size=None, rng=None):\\n    side = load_size if rng else size\\n    image = image.resize((side, side), Image.Resampling.BICUBIC)\\n    if rng:\\n        x, y = rng.randint(0, side-size), rng.randint(0, side-size)\\n        image = image.crop((x, y, x+size, y+size))\\n        if rng.random() < 0.5:\\n            image = ImageOps.mirror(image)\\n    a = np.array(image, dtype=np.float32).transpose(2, 0, 1)\\n    return torch.from_numpy(a / 127.5 - 1.0)\\n\\n\\ndef to_image(tensor):\\n    a = ((tensor.detach().cpu().float().clamp(-1, 1) + 1) * 127.5).round()\\n    return Image.fromarray(a.to(torch.uint8).permute(1, 2, 0).numpy())\\n\\n\\ndef prepare(root, output, seed=42, val_fraction=0.15):\\n    root, output = Path(root), Path(output)\\n    if output.exists():\\n        raise FileExistsError(f\\"Split manifest already exists: {output}. Reuse it or choose a new path.\\")\\n    if not 0 < val_fraction < 0.5:\\n        raise ValueError(\\"val_fraction must lie between 0 and 0.5\\")\\n    result = {\\"seed\\": seed, \\"val_fraction\\": val_fraction, \\"domains\\": {},\\n              \\"preprocessing\\": \\"EXIF orientation -> RGB -> bicubic square resize; no pairing\\"}\\n    cross_domain = set()\\n    for domain, folder in [(\\"A\\", \\"monet_jpg\\"), (\\"B\\", \\"photo_jpg\\")]:\\n        files = sorted(p for p in (root / folder).rglob(\\"*\\")\\n                       if p.suffix.lower() in {\\".jpg\\", \\".jpeg\\", \\".png\\"})\\n        unique, seen, duplicates, invalid, sizes = [], {}, [], [], {}\\n        hashes = {}\\n        for p in files:\\n            rel = p.relative_to(root).as_posix()\\n            try:\\n                im = rgb(p)\\n                digest = hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()\\n                if digest in cross_domain:\\n                    raise ValueError(f\\"Identical image occurs in both domains: {rel}\\")\\n                if digest in seen:\\n                    duplicates.append({\\"excluded\\": rel, \\"retained\\": seen[digest]})\\n                    continue\\n                seen[digest] = rel\\n                unique.append(rel)\\n                hashes[rel] = sha256(p)\\n                key = f\\"{im.width}x{im.height}\\"\\n                sizes[key] = sizes.get(key, 0) + 1\\n            except (OSError, SyntaxError) as e:\\n                invalid.append({\\"file\\": rel, \\"error\\": str(e)})\\n        if invalid:\\n            raise ValueError(f\\"Corrupt images in {folder}; resolve before training: {invalid[:10]}\\")\\n        if len(unique) < 8:\\n            raise ValueError(f\\"Need >=8 valid unique images in {root / folder}; found {len(unique)}\\")\\n        cross_domain.update(seen)\\n        random.Random(seed + (0 if domain == \\"A\\" else 1)).shuffle(unique)\\n        nval = min(len(unique)-2, max(2, round(len(unique)*val_fraction)))\\n        result[\\"domains\\"][domain] = {\\"train\\": unique[nval:], \\"val\\": unique[:nval],\\n                                       \\"sha256\\": hashes, \\"duplicates\\": duplicates,\\n                                       \\"sizes\\": sizes, \\"total_unique\\": len(unique)}\\n    json_write(output, result)\\n    print(json.dumps({d: {\\"train\\": len(v[\\"train\\"]), \\"val\\": len(v[\\"val\\"]),\\n                           \\"duplicates_excluded\\": len(v[\\"duplicates\\"])}\\n                      for d, v in result[\\"domains\\"].items()}, indent=2))\\n    return result\\n\\n\\ndef read_splits(path, root=None, verify=False):\\n    splits = json.loads(Path(path).read_text())\\n    for domain in [\\"A\\", \\"B\\"]:\\n        part = splits[\\"domains\\"][domain]\\n        if set(part[\\"train\\"]) & set(part[\\"val\\"]):\\n            raise ValueError(\\"Training/validation overlap\\")\\n        if verify:\\n            for rel in part[\\"train\\"] + part[\\"val\\"]:\\n                if sha256(Path(root) / rel) != part[\\"sha256\\"][rel]:\\n                    raise ValueError(f\\"Dataset changed since preparation: {rel}\\")\\n    return splits\\n\\n\\nclass UnpairedDataset(Dataset):\\n    def __init__(self, root, splits, size, load_size, seed):\\n        self.root, self.size, self.load_size, self.seed = Path(root), size, load_size, seed\\n        self.paths = {d: splits[\\"domains\\"][d][\\"train\\"] for d in [\\"A\\", \\"B\\"]}\\n        self.epoch_size = max(map(len, self.paths.values()))\\n        self.cached_epoch, self.orders = None, {}\\n\\n    def __len__(self):\\n        return self.epoch_size\\n\\n    def __getitem__(self, index):\\n        epoch, offset = divmod(index, self.epoch_size)\\n        if epoch != self.cached_epoch:\\n            self.orders = {}\\n            for i, d in enumerate([\\"A\\", \\"B\\"]):\\n                order = list(range(len(self.paths[d])))\\n                random.Random(self.seed + epoch*7919 + i*31).shuffle(order)\\n                self.orders[d] = order\\n            self.cached_epoch = epoch\\n        batch = {}\\n        for i, d in enumerate([\\"A\\", \\"B\\"]):\\n            path = self.paths[d][self.orders[d][offset % len(self.paths[d])]]\\n            rng = random.Random(self.seed + index*104729 + i*997)\\n            batch[d] = transform(rgb(self.root/path), self.size, self.load_size, rng)\\n        return batch\\n\\n\\nif __name__ == \\"__main__\\":\\n    p = argparse.ArgumentParser()\\n    p.add_argument(\\"--data-root\\", required=True)\\n    p.add_argument(\\"--output\\", required=True)\\n    p.add_argument(\\"--seed\\", type=int, default=42)\\n    p.add_argument(\\"--val-fraction\\", type=float, default=0.15)\\n    a = p.parse_args()\\n    prepare(a.data_root, a.output, a.seed, a.val_fraction)\\n", "task3_gan/Pramod_Dindukurthi/src/evaluate_official.py": "\\"\\"\\"Instructor evaluator adapted to a CLI; same features, sorting, count caps and formulas.\\nReference: reference/Part3_Evaluation_Script.ipynb, retained unchanged.\\nDifferences: CLI paths, CSV writer instead of pandas, input checks, SciPy compatibility.\\nThese official-protocol scores are distinct from torch-fidelity local diagnostics.\\n\\"\\"\\"\\nimport os, glob\\nimport numpy as np\\nfrom PIL import Image\\nfrom tqdm import tqdm\\n\\nimport torch\\nimport torch.nn as nn\\nimport torchvision.transforms as T\\nimport torchvision.models as models\\n\\nimport scipy.linalg\\nfrom scipy.spatial.distance import cosine\\n\\ndef list_images(folder):\\n    exts = (\\".jpg\\", \\".jpeg\\", \\".png\\")\\n    paths = []\\n    for ext in exts:\\n        paths.extend(glob.glob(os.path.join(folder, f\\"*{ext}\\")))\\n        paths.extend(glob.glob(os.path.join(folder, f\\"*{ext.upper()}\\")))\\n    paths = sorted(list(set(paths)))\\n    return paths\\n\\ndef take_n(paths, n):\\n    if n is None:\\n        return paths\\n    return paths[:min(n, len(paths))]\\n\\n# Inception feature extractor\\ndef get_inception_model():\\n    inception = models.inception_v3(\\n        weights=models.Inception_V3_Weights.IMAGENET1K_V1,\\n        transform_input=False\\n    )\\n    inception.fc = nn.Identity()\\n    inception.to(device)\\n    inception.eval()\\n    return inception\\n\\nINCEPTION_TF = T.Compose([\\n    T.Resize(299),\\n    T.CenterCrop(299),\\n    T.ToTensor(),\\n    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))\\n])\\n\\ndef load_batch(paths):\\n    imgs = []\\n    for p in paths:\\n        img = Image.open(p).convert(\\"RGB\\")\\n        imgs.append(INCEPTION_TF(img))\\n    return torch.stack(imgs, dim=0)\\n\\n@torch.no_grad()\\ndef get_activations(model, image_paths, batch_size=32):\\n    feats = []\\n    for i in tqdm(range(0, len(image_paths), batch_size), desc=\\"Inception activations\\"):\\n        batch_paths = image_paths[i:i+batch_size]\\n        x = load_batch(batch_paths).to(device)\\n        f = model(x).detach().cpu().numpy()\\n        feats.append(f)\\n    return np.concatenate(feats, axis=0)\\n\\ndef frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):\\n    # Support SciPy versions with and without the deprecated disp parameter.\\n    try:\\n        covmean, _ = scipy.linalg.sqrtm(sigma1.dot(sigma2), disp=False)\\n    except TypeError:\\n        covmean = scipy.linalg.sqrtm(sigma1.dot(sigma2))\\n    if not np.isfinite(covmean).all():\\n        offset = np.eye(sigma1.shape[0]) * eps\\n        covmean = scipy.linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))\\n\\n    if np.iscomplexobj(covmean):\\n        covmean = covmean.real\\n\\n    diff = mu1 - mu2\\n    return float(diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean))\\n\\ndef calculate_fid_mifid(real_paths, gen_paths, batch_size=32, subsample_to_match=True):\\n    # For fair comparison, match counts\\n    real_paths = sorted(real_paths)\\n    gen_paths  = sorted(gen_paths)\\n\\n    if subsample_to_match:\\n        n = min(len(real_paths), len(gen_paths))\\n        real_paths = real_paths[:n]\\n        gen_paths  = gen_paths[:n]\\n\\n    model = get_inception_model()\\n\\n    real_act = get_activations(model, real_paths, batch_size=batch_size)\\n    gen_act  = get_activations(model, gen_paths,  batch_size=batch_size)\\n\\n    mu_r, sig_r = real_act.mean(axis=0), np.cov(real_act, rowvar=False)\\n    mu_g, sig_g = gen_act.mean(axis=0),  np.cov(gen_act,  rowvar=False)\\n\\n    fid = frechet_distance(mu_r, sig_r, mu_g, sig_g)\\n\\n    # mean cosine distance between feature vectors,\\n    # paired by index after subsampling/matching\\n    m = min(len(real_act), len(gen_act))\\n    cos_dists = [cosine(real_act[i], gen_act[i]) for i in range(m)]\\n    mifid = float(np.mean(cos_dists))\\n\\n    return fid, mifid\\n\\ndef main():\\n    import argparse, csv, json, hashlib\\n    from pathlib import Path\\n    global device\\n    p = argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\\"--data-root\\", required=True)\\n    p.add_argument(\\"--export-dir\\", required=True)\\n    p.add_argument(\\"--output\\", required=True)\\n    p.add_argument(\\"--n-eval\\", type=int, default=300)\\n    p.add_argument(\\"--batch-size\\", type=int, default=32)\\n    p.add_argument(\\"--device\\", default=\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n    a = p.parse_args()\\n    if a.n_eval < 2: p.error(\\"n-eval must be >=2\\")\\n    device = torch.device(a.device)\\n    data, export = Path(a.data_root), Path(a.export_dir)\\n    result = {}\\n    counts = {}\\n    for direction, target in [(\\"A2B\\", \\"photo_jpg\\"), (\\"B2A\\", \\"monet_jpg\\")]:\\n        real = take_n(list_images(str(data/target)), a.n_eval)\\n        generated = take_n(list_images(str(export/(\\"pred_\\"+direction))), a.n_eval)\\n        if min(len(real), len(generated)) < 2:\\n            raise ValueError(\\"Need >=2 real/generated images in \\" + direction)\\n        counts[direction] = {\\"real_capped\\": len(real), \\"generated_capped\\": len(generated), \\"matched\\": min(len(real), len(generated))}\\n        print(direction, counts[direction], flush=True)\\n        fid, mifid = calculate_fid_mifid(real, generated, batch_size=a.batch_size)\\n        if not np.isfinite([fid, mifid]).all(): raise ValueError(\\"Non-finite official metrics\\")\\n        result[direction] = {\\"FID\\": fid, \\"MiFID\\": mifid}\\n    row = {\\"ID\\": 1, \\"FID\\": (result[\\"A2B\\"][\\"FID\\"]+result[\\"B2A\\"][\\"FID\\"])/2,\\n           \\"MiFID\\": (result[\\"A2B\\"][\\"MiFID\\"]+result[\\"B2A\\"][\\"MiFID\\"])/2}\\n    output = Path(a.output); output.parent.mkdir(parents=True, exist_ok=True)\\n    if output.exists(): raise FileExistsError(\\"Preserve previous submission; choose a new output filename\\")\\n    with output.open(\\"w\\", newline=\\"\\") as f:\\n        writer = csv.DictWriter(f, fieldnames=[\\"ID\\", \\"FID\\", \\"MiFID\\"])\\n        writer.writeheader(); writer.writerow(row)\\n    provenance = {\\"per_direction\\": result, \\"submission\\": row, \\"counts\\": counts,\\n                  \\"n_eval\\": a.n_eval, \\"protocol\\": \\"instructor torchvision Inception_V3_Weights.IMAGENET1K_V1; sorted first-N; index cosine\\",\\n                  \\"evaluation_script_sha256\\": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\\n                  \\"note\\": \\"Real references follow instructor folders, including training images; not a held-out generalization score.\\"}\\n    manifest = export/\\"export_manifest.json\\"\\n    if manifest.exists(): provenance[\\"export_manifest_sha256\\"] = hashlib.sha256(manifest.read_bytes()).hexdigest()\\n    output.with_suffix(\\".metrics.json\\").write_text(json.dumps(provenance, indent=2))\\n    print(row, flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "task3_gan/Pramod_Dindukurthi/src/infer.py": "\\"\\"\\"Export a fixed input list or its declared prefix; never hand-pick outputs.\\"\\"\\"\\nimport argparse\\nimport json\\nfrom pathlib import Path\\nimport time\\n\\nimport torch\\n\\nfrom checkpointing import load_checkpoint\\nfrom common import device_for, json_write, sha256, utc_now\\nfrom data import read_splits, rgb, to_image, transform\\nfrom models import Generator\\n\\n\\n@torch.inference_mode()\\ndef export(checkpoint, data_root, splits_path, output, weights=\\"ema\\", split=\\"val\\", device=\\"auto\\",\\n           max_per_domain=None, predictions_only=False):\\n    if max_per_domain is not None and max_per_domain < 1:\\n        raise ValueError(\\"max_per_domain must be positive\\")\\n    device = device_for(device)\\n    out, root = Path(output), Path(data_root)\\n    if (out/\\"export_manifest.json\\").exists() or (out/\\"pred_A2B\\").exists():\\n        raise FileExistsError(\\"Export exists. Use a new output directory to preserve prior evidence.\\")\\n    state = load_checkpoint(checkpoint)\\n    if sha256(splits_path) != state[\\"split_sha256\\"]:\\n        raise ValueError(\\"Split manifest differs from the trained model\'s manifest\\")\\n    splits = read_splits(splits_path, root, verify=True)\\n    size = state[\\"config\\"][\\"image_size\\"]\\n    models = {}\\n    for k in [\\"G_A2B\\", \\"G_B2A\\"]:\\n        models[k] = Generator(**state[\\"config\\"][\\"model\\"]).to(device).eval()\\n        models[k].load_state_dict(state[\\"ema\\" if weights == \\"ema\\" else \\"models\\"][k])\\n    folders = [\\"pred_A2B\\", \\"pred_B2A\\"]\\n    if not predictions_only:\\n        folders += [\\"input_A\\", \\"input_B\\", \\"cycle_A\\", \\"cycle_B\\"]\\n    for folder in folders:\\n        (out/folder).mkdir(parents=True, exist_ok=True)\\n    records = []\\n    started = time.monotonic()\\n    for d, gkey, fkey in [(\\"A\\", \\"G_A2B\\", \\"G_B2A\\"), (\\"B\\", \\"G_B2A\\", \\"G_A2B\\")]:\\n        direction = \\"A2B\\" if d == \\"A\\" else \\"B2A\\"\\n        paths = splits[\\"domains\\"][d][\\"train\\"] + splits[\\"domains\\"][d][\\"val\\"] if split == \\"all\\" else splits[\\"domains\\"][d][split]\\n        if max_per_domain is not None:\\n            paths = paths[:max_per_domain]\\n        for i, rel in enumerate(paths):\\n            x = transform(rgb(root/rel), size).unsqueeze(0).to(device)\\n            y = models[gkey](x)\\n            filename = f\\"{i:06d}\\"\\n            input_path, pred_path, cycle_path = f\\"input_{d}/{filename}.png\\", f\\"pred_{direction}/{filename}.jpg\\", f\\"cycle_{d}/{filename}.png\\"\\n            # Fixed encoding for all candidates; no image selection/postprocessing.\\n            to_image(y[0]).save(out/pred_path, quality=95, subsampling=0)\\n            record = {\\"domain\\": d, \\"direction\\": direction, \\"source\\": rel,\\n                      \\"prediction\\": pred_path, \\"prediction_sha256\\": sha256(out/pred_path)}\\n            if not predictions_only:\\n                rec = models[fkey](y)\\n                to_image(x[0]).save(out/input_path)\\n                to_image(rec[0]).save(out/cycle_path)\\n                record.update(input=input_path, cycle=cycle_path,\\n                              cycle_L1_tensor_0_1=float((rec-x).abs().mean()/2),\\n                              input_sha256=sha256(out/input_path), cycle_sha256=sha256(out/cycle_path))\\n            records.append(record)\\n            if (i+1) % 100 == 0:\\n                print(\\"Generating\\", direction, f\\"{i+1}/{len(paths)}\\", flush=True)\\n        print(\\"Exported\\", direction, len(paths), \\"images\\", flush=True)\\n    meta = {\\"created_utc\\": utc_now(), \\"checkpoint_sha256\\": sha256(checkpoint),\\n            \\"step\\": state[\\"step\\"], \\"weights\\": weights, \\"split\\": split,\\n            \\"split_sha256\\": sha256(splits_path), \\"image_size\\": size,\\n            \\"jpeg_quality\\": 95, \\"jpeg_subsampling\\": 0,\\n            \\"max_per_domain\\": max_per_domain, \\"predictions_only\\": predictions_only,\\n            \\"input_order\\": \\"fixed split manifest order; all uses train followed by val\\",\\n            \\"inference_seconds_including_io\\": time.monotonic()-started,\\n            \\"preprocessing\\": \\"EXIF orientation, RGB, bicubic square resize\\",\\n            \\"records\\": records}\\n    json_write(out/\\"export_manifest.json\\", meta)\\n    return meta\\n\\n\\nif __name__ == \\"__main__\\":\\n    p = argparse.ArgumentParser(description=__doc__)\\n    for key in [\\"checkpoint\\", \\"data-root\\", \\"splits\\", \\"output\\"]:\\n        p.add_argument(\\"--\\"+key, required=True)\\n    p.add_argument(\\"--weights\\", choices=[\\"raw\\", \\"ema\\"], default=\\"ema\\")\\n    p.add_argument(\\"--split\\", choices=[\\"val\\", \\"train\\", \\"all\\"], default=\\"val\\")\\n    p.add_argument(\\"--device\\", default=\\"auto\\")\\n    p.add_argument(\\"--max-per-domain\\", type=int,\\n                   help=\\"Fixed prefix length per domain; use 300 for the instructor\'s first-300 scoring protocol\\")\\n    p.add_argument(\\"--predictions-only\\", action=\\"store_true\\",\\n                   help=\\"Skip reconstruction inference and input/cycle PNGs; official scoring only, not full local metrics\\")\\n    a = p.parse_args()\\n    export(a.checkpoint, a.data_root, a.splits, a.output, a.weights, a.split, a.device,\\n           a.max_per_domain, a.predictions_only)\\n", "task3_gan/Pramod_Dindukurthi/src/metrics.py": "\\"\\"\\"Documented local diagnostics, NOT the unknown official Kaggle evaluator.\\n\\nInception: torch-fidelity TF-compatible 2048-D features, uint8 RGB input.\\nKID: unbiased polynomial MMD^2 (negative estimates are valid), repeated subsets.\\nDensity/coverage: Naeem et al. https://arxiv.org/abs/2002.09797.\\n\\"\\"\\"\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\nfrom scipy.spatial.distance import cdist\\nimport torch\\n\\nfrom common import sha256\\nfrom data import rgb\\n\\n\\ndef feature_checks(real, fake):\\n    real, fake = np.asarray(real, dtype=np.float64), np.asarray(fake, dtype=np.float64)\\n    if real.ndim != 2 or fake.ndim != 2 or real.shape[1] != fake.shape[1]:\\n        raise ValueError(\\"Features must be Nxd and Mxd with matching dimensions\\")\\n    if min(len(real), len(fake)) < 2 or not np.isfinite(real).all() or not np.isfinite(fake).all():\\n        raise ValueError(\\"Need >=2 finite features in both sets\\")\\n    return real, fake\\n\\n\\ndef fid(real, fake):\\n    real, fake = feature_checks(real, fake)\\n    mean_distance = np.sum((real.mean(0)-fake.mean(0))**2)\\n    cr, cf = np.cov(real, rowvar=False), np.cov(fake, rowvar=False)\\n    # Symmetric PSD formulation avoids complex-valued sqrtm artifacts.\\n    values, vectors = np.linalg.eigh(cr)\\n    root = (vectors * np.sqrt(np.maximum(values, 0))) @ vectors.T\\n    product = root @ cf @ root\\n    middle = np.linalg.eigvalsh((product+product.T)*0.5)\\n    value = mean_distance + np.trace(cr) + np.trace(cf) - 2*np.sqrt(np.maximum(middle, 0)).sum()\\n    return float(max(value, 0))\\n\\n\\ndef kid(real, fake, seed=42, subsets=50, subset_size=1000):\\n    real, fake = feature_checks(real, fake)\\n    n = min(subset_size, len(real), len(fake))\\n    if n < 2 or subsets < 1:\\n        raise ValueError(\\"KID needs subset size >=2 and >=1 subsets\\")\\n    rng, values = np.random.default_rng(seed), []\\n    for _ in range(subsets):\\n        x, y = real[rng.choice(len(real), n, replace=False)], fake[rng.choice(len(fake), n, replace=False)]\\n        xx, yy, xy = (x@x.T/x.shape[1]+1)**3, (y@y.T/y.shape[1]+1)**3, (x@y.T/x.shape[1]+1)**3\\n        values.append((xx.sum()-np.trace(xx)+yy.sum()-np.trace(yy))/(n*(n-1))-2*xy.mean())\\n    return {\\"KID_mean\\": float(np.mean(values)), \\"KID_subset_std\\": float(np.std(values)),\\n            \\"KID_subset_size\\": n, \\"KID_subsets\\": subsets}\\n\\n\\ndef density_coverage(real, fake, k=5, block=256):\\n    real, fake = feature_checks(real, fake)\\n    if len(real) <= k:\\n        raise ValueError(f\\"Density/coverage k={k} requires >{k} real images\\")\\n    radii = []\\n    for start in range(0, len(real), block):\\n        distances = cdist(real[start:start+block], real, metric=\\"sqeuclidean\\")\\n        # Self distance occupies position zero; position k is kth non-self neighbor.\\n        radii.extend(np.partition(distances, k, axis=1)[:, k])\\n    radii = np.asarray(radii)\\n    counts, nearest = 0, np.full(len(real), np.inf)\\n    for start in range(0, len(fake), block):\\n        distances = cdist(real, fake[start:start+block], metric=\\"sqeuclidean\\")\\n        counts += int((distances <= radii[:, None]).sum())\\n        nearest = np.minimum(nearest, distances.min(1))\\n    return {\\"density\\": counts/(k*len(fake)), \\"coverage\\": float(np.mean(nearest <= radii)), \\"density_k\\": k}\\n\\n\\ndef cosine(x, y):\\n    if x.shape != y.shape:\\n        raise ValueError(\\"Paired cosine requires identical feature shapes\\")\\n    return np.sum(x*y, axis=1) / np.maximum(np.linalg.norm(x, axis=1)*np.linalg.norm(y, axis=1), 1e-12)\\n\\n\\nclass InceptionFeatures:\\n    def __init__(self, device, cache_dir):\\n        from torch_fidelity.feature_extractor_inceptionv3 import FeatureExtractorInceptionV3\\n        self.device, self.cache = device, Path(cache_dir)\\n        self.cache.mkdir(parents=True, exist_ok=True)\\n        self.model = FeatureExtractorInceptionV3(\\"inception-v3-compat\\", [\\"2048\\"]).to(device).eval()\\n        self.model.requires_grad_(False)\\n        h = hashlib.sha256()\\n        for key, value in self.model.state_dict().items():\\n            h.update(key.encode()); h.update(value.cpu().numpy().tobytes())\\n        self.weights_hash = h.hexdigest()\\n        self.version = importlib.metadata.version(\\"torch-fidelity\\")\\n\\n    @torch.inference_mode()\\n    def __call__(self, paths, batch_size=16):\\n        identity = {\\"sha256\\": [sha256(p) for p in paths], \\"extractor\\": self.version,\\n                    \\"weights_hash\\": self.weights_hash, \\"feature\\": \\"2048\\", \\"protocol\\": \\"uint8 RGB v1\\"}\\n        key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()\\n        cache = self.cache/(key+\\".npy\\")\\n        if cache.exists():\\n            return np.load(cache)\\n        features = []\\n        for start in range(0, len(paths), batch_size):\\n            arrays = [np.array(rgb(p), dtype=np.uint8).transpose(2, 0, 1) for p in paths[start:start+batch_size]]\\n            tensor = torch.from_numpy(np.stack(arrays)).to(self.device)\\n            features.append(self.model(tensor)[0].cpu().numpy())\\n        result = np.concatenate(features)\\n        np.save(cache, result)\\n        return result\\n\\n\\n@torch.inference_mode()\\ndef lpips_pairs(input_paths, output_paths, device, batch_size=16, model=None):\\n    if model is None:\\n        import lpips\\n        model = lpips.LPIPS(net=\\"alex\\").to(device).eval()\\n    model.requires_grad_(False)\\n    values = []\\n    for start in range(0, len(input_paths), batch_size):\\n        def batch(paths):\\n            arrays = [np.array(rgb(p), dtype=np.float32).transpose(2, 0, 1)/127.5-1\\n                      for p in paths[start:start+batch_size]]\\n            return torch.from_numpy(np.stack(arrays)).to(device)\\n        values.extend(model(batch(input_paths), batch(output_paths)).flatten().cpu().tolist())\\n    return np.asarray(values), model\\n", "task3_gan/Pramod_Dindukurthi/src/models.py": "\\"\\"\\"From-scratch CycleGAN networks. No pretrained generation models.\\n\\nArchitecture reference: Zhu et al., ICCV 2017 (https://arxiv.org/abs/1703.10593).\\nA = Monet; B = Photo. D_A judges Monet and D_B judges Photo.\\n\\"\\"\\"\\nimport torch\\nfrom torch import nn\\n\\n\\nclass ResidualBlock(nn.Module):\\n    def __init__(self, channels):\\n        super().__init__()\\n        self.body = nn.Sequential(\\n            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3),\\n            nn.InstanceNorm2d(channels), nn.ReLU(True),\\n            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3),\\n            nn.InstanceNorm2d(channels),\\n        )\\n\\n    def forward(self, x):\\n        return x + self.body(x)\\n\\n\\nclass Generator(nn.Module):\\n    def __init__(self, channels=64, blocks=9, upsample=\\"transpose\\"):\\n        super().__init__()\\n        if upsample not in {\\"transpose\\", \\"resize\\"}:\\n            raise ValueError(\\"upsample must be transpose or resize\\")\\n        layers = [nn.ReflectionPad2d(3), nn.Conv2d(3, channels, 7),\\n                  nn.InstanceNorm2d(channels), nn.ReLU(True)]\\n        c = channels\\n        for _ in range(2):\\n            layers += [nn.Conv2d(c, c * 2, 3, 2, 1),\\n                       nn.InstanceNorm2d(c * 2), nn.ReLU(True)]\\n            c *= 2\\n        layers += [ResidualBlock(c) for _ in range(blocks)]\\n        for _ in range(2):\\n            if upsample == \\"transpose\\":\\n                layers += [nn.ConvTranspose2d(c, c // 2, 3, 2, 1, output_padding=1)]\\n            else:\\n                layers += [nn.Upsample(scale_factor=2, mode=\\"nearest\\"),\\n                           nn.ReflectionPad2d(1), nn.Conv2d(c, c // 2, 3)]\\n            c //= 2\\n            layers += [nn.InstanceNorm2d(c), nn.ReLU(True)]\\n        layers += [nn.ReflectionPad2d(3), nn.Conv2d(c, 3, 7), nn.Tanh()]\\n        self.body = nn.Sequential(*layers)\\n\\n    def forward(self, x):\\n        return self.body(x)\\n\\n\\nclass PatchDiscriminator(nn.Module):\\n    \\"\\"\\"70x70 receptive field; logits (no sigmoid) for least-squares GAN.\\"\\"\\"\\n    def __init__(self, channels=64):\\n        super().__init__()\\n        layers = [nn.Conv2d(3, channels, 4, 2, 1), nn.LeakyReLU(0.2, True)]\\n        for stride in [2, 2, 1]:\\n            layers += [nn.Conv2d(channels, channels * 2, 4, stride, 1),\\n                       nn.InstanceNorm2d(channels * 2), nn.LeakyReLU(0.2, True)]\\n            channels *= 2\\n        layers += [nn.Conv2d(channels, 1, 4, 1, 1)]\\n        self.body = nn.Sequential(*layers)\\n\\n    def forward(self, x):\\n        return self.body(x)\\n\\n\\ndef initialize(module):\\n    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):\\n        nn.init.normal_(module.weight, 0.0, 0.02)\\n        if module.bias is not None:\\n            nn.init.zeros_(module.bias)\\n\\n\\ndef build_models(config, device):\\n    spec = config[\\"model\\"]\\n    models = {\\"G_A2B\\": Generator(**spec), \\"G_B2A\\": Generator(**spec),\\n              \\"D_A\\": PatchDiscriminator(spec[\\"channels\\"]),\\n              \\"D_B\\": PatchDiscriminator(spec[\\"channels\\"])}\\n    for model in models.values():\\n        model.apply(initialize)\\n        model.to(device)\\n    return models\\n", "task3_gan/Pramod_Dindukurthi/src/package_code.py": "\\"\\"\\"Package runnable source without Git, secrets, data, checkpoints or results.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport zipfile\\n\\n\\ndef package(output):\\n    member = Path(__file__).resolve().parents[1]\\n    repo = member.parents[1]\\n    files = [repo/\\"README.md\\", repo/\\".gitignore\\", member/\\"README.md\\", member/\\"requirements.txt\\",\\n             member/\\"evaluate_local.py\\", member/\\"results.md\\", member/\\"failure_analysis.md\\"]\\n    files += sorted((member/\\"src\\").glob(\\"*.py\\")) + sorted((member/\\"src\\").glob(\\"*.ipynb\\"))\\n    files += sorted((member/\\"configs\\").glob(\\"*.yaml\\"))\\n    files += sorted((member/\\"reference\\").glob(\\"*.ipynb\\"))\\n    files = [p for p in files if p.exists()]\\n    manifest = {p.relative_to(repo).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}\\n    output = Path(output)\\n    output.parent.mkdir(parents=True, exist_ok=True)\\n    with zipfile.ZipFile(output, \\"w\\", zipfile.ZIP_DEFLATED) as z:\\n        for p in files:\\n            z.write(p, p.relative_to(repo).as_posix())\\n        z.writestr(\\"CODE_BUNDLE_MANIFEST.json\\", json.dumps(manifest, indent=2))\\n    print(\\"Created\\", output, \\"with\\", len(files), \\"source/config/notebook files\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    p = argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\\"--output\\", default=\\"artifacts/task3-code.zip\\")\\n    package(p.parse_args().output)\\n", "task3_gan/Pramod_Dindukurthi/src/package_completed_run.py": "\\"\\"\\"Package a completed phase without reading the active training checkpoint/logs.\\n\\nStandard library only. Run on the GPU machine; no email is sent.\\n\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport re\\nimport time\\nimport zipfile\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'--run-dir\', required=True)\\n    parser.add_argument(\'--step\', type=int,\\n                        help=\'Completed checkpoint step; default: highest completed step in this run.\')\\n    parser.add_argument(\'--weights\', choices=[\'ema\', \'raw\'], default=\'ema\')\\n    parser.add_argument(\'--notebook\', help=\'Optional path to the saved, executed lab notebook.\')\\n    parser.add_argument(\'--output-dir\')\\n    parser.add_argument(\'--full\', action=\'store_true\',\\n                        help=\'Include frozen checkpoint, evaluation images and all completed previews.\')\\n    args = parser.parse_args()\\n    run = Path(args.run_dir).resolve()\\n    completed = []\\n    for p in (run / \'reproducibility/manifests\').glob(\'completed_*.json\'):\\n        obj = json.loads(p.read_text())\\n        if isinstance(obj.get(\'end_step\'), int):\\n            completed.append((p, obj))\\n    if not completed:\\n        raise RuntimeError(\'No completed training manifest; do not package an active phase.\')\\n    if args.step is None:\\n        args.step = max(obj[\'end_step\'] for _, obj in completed)\\n    selected = [(p, obj) for p, obj in completed if obj[\'end_step\'] == args.step]\\n    if not selected:\\n        raise RuntimeError(f\'No completed training manifest for step {args.step}.\')\\n    if len(selected) != 1:\\n        raise RuntimeError(\'Multiple completed phases match this step; select a unique run directory.\')\\n    completed_path, summary = selected[0]\\n    checkpoint = run / f\'checkpoints/evaluation/step_{args.step:08d}.pt\'\\n    if not checkpoint.is_file():\\n        raise RuntimeError(\'Frozen evaluation checkpoint is missing. Do not substitute latest_checkpoint.pt.\')\\n    output = Path(args.output_dir).resolve() if args.output_dir else run.parent / \'transfer\'\\n    if output == run or run in output.parents:\\n        raise ValueError(\'Put the archive outside the training run directory.\')\\n    output.mkdir(parents=True, exist_ok=True)\\n    mode = \'full\' if args.full else \'email\'\\n    target = output / f\'task3_step_{args.step:08d}_{mode}_{time.time_ns()}.zip\'\\n    partial = target.with_suffix(\'.zip.partial\')\\n    files = {}\\n\\n    def add(p, alias=None):\\n        if p.is_file():\\n            name = alias or f\'task3_gan/{run.name}/{p.relative_to(run).as_posix()}\'\\n            if name in files and files[name] != p:\\n                raise ValueError(f\'Conflicting archive path: {name}\')\\n            files[name] = p\\n\\n    history_phases = [(p, obj) for p, obj in completed\\n                      if obj[\'end_step\'] <= args.step\\n                      and obj.get(\'config\') == summary.get(\'config\')\\n                      and obj.get(\'split_sha256\') == summary.get(\'split_sha256\')]\\n    for phase_path, phase in history_phases:\\n        segment = phase[\'segment\']\\n        add(phase_path)\\n        add(run / f\'reproducibility/manifests/run_{segment}.json\')\\n        add(run / f\'reproducibility/manifests/run_{segment}.requirements.txt\')\\n        for name in (f\'history_{segment}.csv\', f\'train_{segment}.log\'):\\n            p = run / \'reproducibility/raw_logs\' / name\\n            if not p.is_file():\\n                raise RuntimeError(f\'Completed phase log is missing: {p}\')\\n            add(p)\\n    for name in (\'data_processed/splits.json\', \'data_processed/runtime_config.yaml\',\\n                 \'README.md\', \'requirements.txt\', \'evaluate_local.py\',\\n                 \'results.md\', \'failure_analysis.md\'):\\n        add(run / name)\\n    for folder in (\'src\', \'configs\', \'reference\'):\\n        for p in (run / folder).rglob(\'*\'):\\n            if \'__pycache__\' not in p.parts and p.suffix in {\'.py\', \'.yaml\', \'.yml\', \'.ipynb\', \'.md\'}:\\n                add(p)\\n    for p in run.glob(\'*.ipynb\'):\\n        add(p)\\n    if args.notebook:\\n        notebook = Path(args.notebook).resolve()\\n        if not notebook.is_file() or notebook.suffix != \'.ipynb\':\\n            raise FileNotFoundError(\'The --notebook argument must name a saved .ipynb file.\')\\n        add(notebook, f\'task3_gan/{run.name}/notebooks/executed_{notebook.name}\')\\n    exports = []\\n    manifest_hashes = {}\\n    local_metrics_present = False\\n    for folder in (\'evaluations\', \'official\'):\\n        for manifest in (run / \'outputs\' / folder).glob(\'*/export_manifest.json\'):\\n            meta = json.loads(manifest.read_text())\\n            if meta.get(\'step\') != args.step:\\n                continue\\n            if meta.get(\'checkpoint_sha256\') != summary[\'checkpoint_sha256\']:\\n                raise RuntimeError(f\'Export checkpoint differs from completed phase: {manifest}\')\\n            exports.append(str(manifest.parent.relative_to(run)))\\n            manifest_hashes[hashlib.sha256(manifest.read_bytes()).hexdigest()] = meta\\n            if meta.get(\'split\') == \'val\' and (manifest.parent / \'full_metrics_report.csv\').is_file():\\n                local_metrics_present = True\\n            for p in manifest.parent.rglob(\'*\'):\\n                if args.full or p.suffix in {\'.json\', \'.csv\', \'.md\', \'.txt\'}:\\n                    add(p)\\n    # Historical candidate summaries support comparison without copying old images.\\n    for folder in (\'evaluations\', \'official\'):\\n        for manifest in (run / \'outputs\' / folder).glob(\'*/export_manifest.json\'):\\n            meta = json.loads(manifest.read_text())\\n            if not isinstance(meta.get(\'step\'), int) or meta[\'step\'] >= args.step:\\n                continue\\n            for p in manifest.parent.rglob(\'*\'):\\n                if p.suffix in {\'.json\', \'.csv\', \'.md\', \'.txt\'}:\\n                    add(p)\\n    # Use per-candidate submissions; never collect another segment\'s live logs.\\n    submission_candidates = []\\n    for p in run.glob(f\'submission_step_{args.step:08d}_*\'):\\n        if p.suffix in {\'.csv\', \'.json\'}:\\n            add(p)\\n        if p.name.endswith(\'.metrics.json\'):\\n            payload = json.loads(p.read_text())\\n            meta = manifest_hashes.get(payload.get(\'export_manifest_sha256\'))\\n            csv_path = p.with_name(p.name.removesuffix(\'.metrics.json\') + \'.csv\')\\n            if meta and meta.get(\'weights\') == args.weights and csv_path.is_file():\\n                submission_candidates.append((csv_path, p, payload))\\n    # Preserve exact evaluator output under the instructor\'s canonical CSV name.\\n    submission_candidates.sort(key=lambda item: item[0].name)\\n    if submission_candidates:\\n        csv_path, metrics_path, payload = submission_candidates[0]\\n        import csv\\n        import math\\n        with csv_path.open(newline=\'\') as f:\\n            reader = csv.DictReader(f)\\n            rows = list(reader)\\n            if reader.fieldnames != [\'ID\', \'FID\', \'MiFID\'] or len(rows) != 1:\\n                raise ValueError(\'Official submission schema is invalid.\')\\n        if any(not math.isfinite(float(rows[0][key])) or\\n               float(rows[0][key]) != float(payload[\'submission\'][key])\\n               for key in (\'FID\', \'MiFID\')):\\n            raise ValueError(\'Submission CSV differs from its metric provenance.\')\\n        add(csv_path, f\'task3_gan/{run.name}/submission.csv\')\\n        add(metrics_path, f\'task3_gan/{run.name}/submission_provenance.json\')\\n    draft = run / \'results_draft.md\'\\n    if draft.is_file() and summary[\'checkpoint_sha256\'] in draft.read_text():\\n        for name in (\'results_draft.md\', \'full_metrics_report.csv\', \'metrics_report.csv\'):\\n            add(run / name)\\n        for name in (\'adversarial.png\', \'cycle_identity.png\', \'gradients.png\', \'learning_rate.png\'):\\n            add(run / \'outputs/plots\' / name)\\n    audit = run / \'outputs/human_audit\'\\n    audit_matches = False\\n    if (audit / \'private_key.json\').is_file():\\n        key = json.loads((audit / \'private_key.json\').read_text())\\n        audit_matches = key.get(\'checkpoint_sha256\') == summary[\'checkpoint_sha256\']\\n        if audit_matches:\\n            for p in audit.rglob(\'*\'):\\n                if args.full or p.suffix in {\'.json\', \'.csv\', \'.md\', \'.txt\'}:\\n                    add(p)\\n    selection = run / \'selection.json\'\\n    if selection.is_file() and json.loads(selection.read_text()).get(\'checkpoint_sha256\') == summary[\'checkpoint_sha256\']:\\n        add(selection)\\n    add(completed_path, f\'task3_gan/{run.name}/training_summary.json\')\\n    previews = []\\n    for p in (run / \'outputs/plots\').glob(\'preview_*.png\'):\\n        match = re.fullmatch(r\'preview_(\\\\d+)\\\\.png\', p.name)\\n        if match and int(match[1]) <= args.step:\\n            previews.append((int(match[1]), p))\\n    previews.sort()\\n    chosen = previews if args.full else previews[:1] + previews[-3:]\\n    for _, p in chosen:\\n        add(p)\\n    if args.full:\\n        add(checkpoint)\\n    evidence = {\\n        \'step\': args.step, \'archive_mode\': mode,\\n        \'submission_weights\': args.weights,\\n        \'completed_training_summary\': summary,\\n        \'completed_history_segments\': [obj[\'segment\'] for _, obj in history_phases],\\n        \'frozen_checkpoint_relative_path\': str(checkpoint.relative_to(run)),\\n        \'checkpoint_binary_included\': args.full,\\n        \'evaluation_exports\': exports,\\n        \'preview_policy\': \'all through completed step\' if args.full else \'first and final three through completed step\',\\n        \'exclusions\': [\'active training logs and manifests\', \'latest_checkpoint.pt\',\\n                       \'periodic checkpoint collection\', \'shared original dataset\',\\n                       \'background console/status files\', \'reports for a different checkpoint\'],\\n        \'notes\': [\'This archive describes a completed phase, not the current overnight run.\',\\n                  \'Source files are the versions present when packaging; the completed manifest records training source provenance.\',\\n                  \'Save the executed notebook and Kaggle screenshot separately if absent from this folder.\',\\n                  \'Email mode is evidence only, not a checkpoint backup. Full mode contains the frozen resumable checkpoint.\'],\\n        \'completion_status\': {\\n            \'official_submission_csv_present\': bool(submission_candidates),\\n            \'local_validation_metrics_present\': local_metrics_present,\\n            \'human_audit_summary_present\': audit_matches and (audit / \'audit_summary.json\').is_file(),\\n            \'note\': \'Missing rubric evidence still needs completion; creating a ZIP does not complete the report or human audit.\'},\\n        \'files\': [],\\n    }\\n    try:\\n        with zipfile.ZipFile(partial, \'x\', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:\\n            for name, p in sorted(files.items()):\\n                before = p.stat()\\n                info = zipfile.ZipInfo.from_file(p, name)\\n                info.compress_type = zipfile.ZIP_STORED if p.suffix in {\'.pt\', \'.jpg\', \'.jpeg\', \'.png\'} else zipfile.ZIP_DEFLATED\\n                digest = hashlib.sha256()\\n                with p.open(\'rb\') as source, archive.open(info, \'w\', force_zip64=True) as dest:\\n                    while chunk := source.read(1024 * 1024):\\n                        digest.update(chunk)\\n                        dest.write(chunk)\\n                after = p.stat()\\n                if (before.st_ino, before.st_size, before.st_mtime_ns) != (after.st_ino, after.st_size, after.st_mtime_ns):\\n                    raise RuntimeError(f\'File changed while packaging; archive discarded: {p}\')\\n                if p == checkpoint and digest.hexdigest() != summary[\'checkpoint_sha256\']:\\n                    raise RuntimeError(\'Frozen checkpoint checksum differs from the completed manifest.\')\\n                evidence[\'files\'].append({\'path\': name, \'bytes\': before.st_size, \'sha256\': digest.hexdigest()})\\n            archive.writestr(\'ARCHIVE_EVIDENCE.json\', json.dumps(evidence, indent=2))\\n            archive.writestr(\'PACKAGE_README.md\',\\n                f\'# Completed Task 3 snapshot\\\\n\\\\nCheckpoint step: {args.step}. Submission weights: {args.weights}.\\\\n\\\\n\'\\n                \'This package contains completed-phase evidence and excludes the active training job. \'\\n                \'The archive manifest records checksums and which rubric evidence is present.\\\\n\\\\n\'\\n                \'To run on another computer, restore the original dataset to \'\\n                \'`task3_gan/data/monet_jpg/` and `task3_gan/data/photo_jpg/`, install a matching \'\\n                \'CUDA PyTorch/torchvision pair and the member requirements, and update notebook paths. \'\\n                \'Keep the provided split manifest and runtime configuration unchanged when resuming.\\\\n\\\\n\'\\n                \'The shared original dataset is not included. Save your executed notebook and \'\\n                \'Kaggle screenshot separately, or supply `--notebook` while packaging. \'\\n                \'The written report, all requested metrics, and independent human ratings still need \'\\n                \'completion if absent. No submission or email is sent by this packager.\\\\n\')\\n        partial.rename(target)\\n    except BaseException:\\n        partial.unlink(missing_ok=True)\\n        raise\\n    print(f\'Created: {target}\')\\n    print(f\'Size: {target.stat().st_size / 1024**2:.1f} MiB; files: {len(files)}\')\\n    print(f\'Checkpoint step: {args.step}; completed history phases: {len(history_phases)}\')\\n    print(\'Evidence status:\', json.dumps(evidence[\'completion_status\']))\\n    print(\'Training was not stopped. No email or upload was performed.\')\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "task3_gan/Pramod_Dindukurthi/src/preflight.py": "\\"\\"\\"Run on the TRAINING machine; does not reserve a GPU or launch training.\\"\\"\\"\\nimport argparse\\nimport json\\nfrom pathlib import Path\\nimport shutil\\nimport subprocess\\nimport sys\\n\\nfrom common import manifest\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\\"--data-root\\")\\n    a = p.parse_args()\\n    print(json.dumps(manifest({}, \\"preflight\\"), indent=2))\\n    print(\\"Python executable:\\", sys.executable)\\n    print(\\"Working directory:\\", Path.cwd())\\n    print(\\"Disk free GiB:\\", round(shutil.disk_usage(Path.cwd()).free/2**30, 1))\\n    if shutil.which(\\"nvidia-smi\\"):\\n        subprocess.run([\\"nvidia-smi\\"], check=False)\\n    if a.data_root:\\n        for folder in [\\"monet_jpg\\", \\"photo_jpg\\"]:\\n            root = Path(a.data_root)/folder\\n            print(folder, \\"exists:\\", root.exists(), \\"images:\\",\\n                  sum(p.suffix.lower() in {\\".jpg\\", \\".jpeg\\", \\".png\\"} for p in root.rglob(\\"*\\")))\\n    print(\\"Confirm a persistent backup destination before long training. A saved notebook does not back up checkpoints.\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "task3_gan/Pramod_Dindukurthi/src/quick_kaggle_score.py": "from pathlib import Path\\nimport csv, json, sys\\nimport torch\\n\\n# Uses a FROZEN checkpoint. For tomorrow\'s model, first run only Section 9\'s\\n# snapshot cell, then run this cell; no full image-export sections are needed.\\ndefault_run = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / \\"src/train.py\\").is_file()),\\n                   Path(\\"/app/task3_gan/Pramod_Dindukurthi\\"))\\nRUN_DIR = Path(globals().get(\\"RUN_DIR\\", default_run))\\nDATA_ROOT = Path(globals().get(\\"DATA_ROOT\\", RUN_DIR.parent / \\"data\\"))\\nfrozen = sorted(p for p in (RUN_DIR / \\"checkpoints/evaluation\\").glob(\\"step_*.pt\\")\\n                if p.stem.removeprefix(\\"step_\\").isdigit())\\ndefault_checkpoint = frozen[-1] if frozen else RUN_DIR / \\"checkpoints/evaluation/step_00112805.pt\\"\\nCHECKPOINT = Path(globals().get(\\"SNAPSHOT\\", default_checkpoint))\\nWEIGHTS = globals().get(\\"WEIGHTS\\", \\"ema\\")  # set \\"raw\\" for a separately recorded comparison\\nassert WEIGHTS in {\\"ema\\", \\"raw\\"}\\nN_EVAL = 300\\nBATCH_SIZE = 16\\nDEVICE = \\"cuda\\" if torch.cuda.is_available() else \\"cpu\\"\\nSRC = RUN_DIR / \\"src\\"\\nif str(SRC) not in sys.path:\\n    sys.path.insert(0, str(SRC))\\nfrom checkpointing import load_checkpoint\\nfrom common import sha256, json_write, utc_now\\nfrom data import read_splits, rgb, transform, to_image\\nfrom models import Generator\\nimport evaluate_official as official\\n\\nif not CHECKPOINT.is_file():\\n    raise FileNotFoundError(f\\"Freeze the checkpoint using Section 9\'s first cell: {CHECKPOINT}\\")\\nif CHECKPOINT.name == \\"latest_checkpoint.pt\\":\\n    raise ValueError(\\"Use a frozen evaluation checkpoint, not the actively replaced latest_checkpoint.pt.\\")\\nSPLITS = RUN_DIR / \\"data_processed/splits.json\\"\\nprint(\\"Scoring frozen checkpoint:\\", CHECKPOINT, \\"| weights:\\", WEIGHTS, flush=True)\\ncheckpoint_hash = sha256(CHECKPOINT)\\nstate = load_checkpoint(CHECKPOINT)\\nstep = state[\\"step\\"]\\nassert state[\\"config\\"][\\"image_size\\"] == 256, \\"Official predictions must be 256 x 256.\\"\\nassert sha256(SPLITS) == state[\\"split_sha256\\"], \\"Training split manifest differs.\\"\\ntag = f\\"step_{step:08d}_{WEIGHTS}\\"\\nOFFICIAL_DIR = RUN_DIR / \\"outputs/official\\" / f\\"{tag}_quick_{checkpoint_hash[:12]}\\"\\nOFFICIAL_CSV = RUN_DIR / f\\"submission_{tag}_quick_{checkpoint_hash[:12]}.csv\\"\\nSCORE_JSON = OFFICIAL_CSV.with_suffix(\\".metrics.json\\")\\n\\n# Reuse a prior score ONLY if its export manifest matches this checkpoint/weights,\\n# the evaluator source is identical, and both directions used 300 images.\\nprior = []\\nfor candidate in RUN_DIR.glob(f\\"submission_{tag}*.metrics.json\\"):\\n    payload = json.loads(candidate.read_text())\\n    if payload.get(\\"evaluation_script_sha256\\") != sha256(Path(official.__file__)):\\n        continue\\n    if any(payload.get(\\"counts\\", {}).get(d, {}).get(\\"matched\\") != N_EVAL for d in (\\"A2B\\", \\"B2A\\")):\\n        continue\\n    for manifest in (RUN_DIR / \\"outputs/official\\").glob(\\"*/export_manifest.json\\"):\\n        if sha256(manifest) != payload.get(\\"export_manifest_sha256\\"):\\n            continue\\n        metadata = json.loads(manifest.read_text())\\n        if metadata.get(\\"checkpoint_sha256\\") == checkpoint_hash and metadata.get(\\"weights\\") == WEIGHTS:\\n            csv_path = candidate.with_name(candidate.name.removesuffix(\\".metrics.json\\") + \\".csv\\")\\n            if csv_path.is_file():\\n                prior.append((csv_path, payload))\\n            break\\n\\nif prior:\\n    OFFICIAL_CSV, result = prior[0]\\n    row = result[\\"submission\\"]\\n    del state\\n    print(\\"Reusing previously computed, matching official metrics.\\")\\nelse:\\n    manifest_path = OFFICIAL_DIR / \\"export_manifest.json\\"\\n    if not manifest_path.exists():\\n        if OFFICIAL_DIR.exists():\\n            raise RuntimeError(f\\"Incomplete earlier export exists; preserve it and choose a new output name: {OFFICIAL_DIR}\\")\\n        splits = read_splits(SPLITS)\\n        records = []\\n        for domain, direction, generator_key in ((\\"A\\", \\"A2B\\", \\"G_A2B\\"), (\\"B\\", \\"B2A\\", \\"G_B2A\\")):\\n            sources = (splits[\\"domains\\"][domain][\\"train\\"] + splits[\\"domains\\"][domain][\\"val\\"])[:N_EVAL]\\n            assert len(sources) == N_EVAL, \\"Need 300 permitted source images per direction.\\"\\n            folder = OFFICIAL_DIR / f\\"pred_{direction}\\"\\n            folder.mkdir(parents=True)\\n            generator = Generator(**state[\\"config\\"][\\"model\\"]).to(DEVICE).eval()\\n            generator.load_state_dict(state[\\"ema\\" if WEIGHTS == \\"ema\\" else \\"models\\"][generator_key])\\n            with torch.inference_mode():\\n                for i, rel in enumerate(sources):\\n                    assert sha256(DATA_ROOT / rel) == splits[\\"domains\\"][domain][\\"sha256\\"][rel], f\\"Source changed: {rel}\\"\\n                    prediction = generator(transform(rgb(DATA_ROOT / rel), 256).unsqueeze(0).to(DEVICE))\\n                    path = folder / f\\"{i:06d}.jpg\\"\\n                    to_image(prediction[0]).save(path, quality=95, subsampling=0)\\n                    records.append({\\"domain\\": domain, \\"direction\\": direction, \\"source\\": rel,\\n                                    \\"prediction\\": path.relative_to(OFFICIAL_DIR).as_posix(),\\n                                    \\"prediction_sha256\\": sha256(path)})\\n                    if (i + 1) % 100 == 0:\\n                        print(f\\"Generated {direction}: {i + 1}/{N_EVAL}\\", flush=True)\\n            del generator, prediction\\n        json_write(manifest_path, {\\"created_utc\\": utc_now(), \\"checkpoint_sha256\\": checkpoint_hash,\\n            \\"step\\": step, \\"weights\\": WEIGHTS, \\"split\\": \\"all\\", \\"split_sha256\\": sha256(SPLITS),\\n            \\"image_size\\": 256, \\"jpeg_quality\\": 95, \\"jpeg_subsampling\\": 0,\\n            \\"predictions_only\\": True, \\"max_per_domain\\": N_EVAL,\\n            \\"input_order\\": \\"fixed split manifest order; train followed by val; first 300\\",\\n            \\"records\\": records})\\n    meta = json.loads(manifest_path.read_text())\\n    assert meta[\\"checkpoint_sha256\\"] == checkpoint_hash and meta[\\"weights\\"] == WEIGHTS\\n    for record in meta[\\"records\\"]:\\n        assert sha256(OFFICIAL_DIR / record[\\"prediction\\"]) == record[\\"prediction_sha256\\"], \\"Prediction changed.\\"\\n    del state\\n    if DEVICE == \\"cuda\\":\\n        torch.cuda.empty_cache()\\n    if OFFICIAL_CSV.exists() or SCORE_JSON.exists():\\n        raise FileExistsError(\\"Preserve the earlier score files; use a different output name.\\")\\n    official.device = torch.device(DEVICE)\\n    per_direction, counts = {}, {}\\n    for direction, target in ((\\"A2B\\", \\"photo_jpg\\"), (\\"B2A\\", \\"monet_jpg\\")):\\n        real = official.take_n(official.list_images(str(DATA_ROOT / target)), N_EVAL)\\n        generated = official.take_n(official.list_images(str(OFFICIAL_DIR / f\\"pred_{direction}\\")), N_EVAL)\\n        assert len(real) == len(generated) == N_EVAL\\n        fid, mifid = official.calculate_fid_mifid(real, generated, batch_size=BATCH_SIZE)\\n        assert torch.isfinite(torch.tensor([fid, mifid], dtype=torch.float64)).all(), \\"Non-finite metric.\\"\\n        per_direction[direction] = {\\"FID\\": fid, \\"MiFID\\": mifid}\\n        counts[direction] = {\\"real_capped\\": len(real), \\"generated_capped\\": len(generated), \\"matched\\": N_EVAL}\\n    row = {\\"ID\\": 1,\\n           \\"FID\\": sum(x[\\"FID\\"] for x in per_direction.values()) / 2,\\n           \\"MiFID\\": sum(x[\\"MiFID\\"] for x in per_direction.values()) / 2}\\n    with OFFICIAL_CSV.open(\\"x\\", newline=\\"\\") as f:\\n        writer = csv.DictWriter(f, fieldnames=[\\"ID\\", \\"FID\\", \\"MiFID\\"])\\n        writer.writeheader()\\n        writer.writerow(row)\\n    result = {\\"per_direction\\": per_direction, \\"submission\\": row, \\"counts\\": counts, \\"n_eval\\": N_EVAL,\\n              \\"evaluation_script_sha256\\": sha256(Path(official.__file__)),\\n              \\"export_manifest_sha256\\": sha256(manifest_path),\\n              \\"protocol\\": \\"instructor torchvision Inception IMAGENET1K_V1; sorted first-300; index cosine\\"}\\n    json_write(SCORE_JSON, result)\\n\\nestimated_score = -(row[\\"FID\\"] + row[\\"MiFID\\"]) / 2\\nprint(json.dumps(result[\\"per_direction\\"], indent=2))\\nprint(f\\"\\\\nCheckpoint step: {step:,} | Weights: {WEIGHTS}\\")\\nprint(f\\"FID: {row[\'FID\']:.6f} | MiFID: {row[\'MiFID\']:.6f}\\")\\nprint(f\\"Expected Kaggle displayed score: {estimated_score:.6f} (closer to zero is better)\\")\\nprint(f\\"Submission CSV: {OFFICIAL_CSV}\\")\\nprint(\\"No Kaggle upload was performed. This cell evaluates the chosen frozen checkpoint only.\\")\\n", "task3_gan/Pramod_Dindukurthi/src/record_submission.py": "\\"\\"\\"Archive an externally produced OFFICIAL evaluator CSV after validating its schema.\\n\\nThis does not calculate, guess, or replace the competition\'s metric implementation.\\n\\"\\"\\"\\nimport argparse\\nimport csv\\nfrom pathlib import Path\\nimport shutil\\n\\nfrom common import json_write, sha256, utc_now\\n\\n\\ndef record(official_csv, sample_csv, evaluator, official_log, export_dir, run_dir):\\n    official_csv, sample_csv, evaluator, official_log = map(Path, [official_csv, sample_csv, evaluator, official_log])\\n    export_dir, run_dir = Path(export_dir), Path(run_dir)\\n    for path in [official_csv, sample_csv, evaluator, official_log, export_dir/\\"export_manifest.json\\"]:\\n        if not path.is_file():\\n            raise FileNotFoundError(path)\\n    with open(sample_csv) as f:\\n        sample = list(csv.reader(f))\\n    with open(official_csv) as f:\\n        rows = list(csv.reader(f))\\n    if not rows or not sample or rows[0] != sample[0] or len(rows) != len(sample):\\n        raise ValueError(\\"Official output headers/row count do not match sample submission\\")\\n    if any(len(r) != len(rows[0]) or any(not c.strip() for c in r) for r in rows[1:]):\\n        raise ValueError(\\"Submission has missing fields\\")\\n    dest = run_dir/\\"submission.csv\\"\\n    if dest.exists():\\n        raise FileExistsError(\\"Archive the previous submission before recording a new one\\")\\n    shutil.copy2(official_csv, dest)\\n    json_write(run_dir/\\"submission_provenance.json\\", {\\n        \\"recorded_utc\\": utc_now(), \\"submission_sha256\\": sha256(dest),\\n        \\"official_evaluator_sha256\\": sha256(evaluator), \\"official_log_sha256\\": sha256(official_log),\\n        \\"sample_csv_sha256\\": sha256(sample_csv), \\"export_manifest_sha256\\": sha256(export_dir/\\"export_manifest.json\\"),\\n        \\"status\\": \\"schema checked; user must verify evaluator was run on this export; not uploaded\\",\\n        \\"public_score\\": None, \\"private_score\\": None, \\"rank\\": None})\\n    print(\\"Recorded submission.csv and provenance. No values were recomputed and nothing was uploaded.\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    p = argparse.ArgumentParser(description=__doc__)\\n    for name in [\\"official-csv\\", \\"sample-csv\\", \\"evaluator\\", \\"official-log\\", \\"export-dir\\", \\"run-dir\\"]:\\n        p.add_argument(\\"--\\"+name, required=True)\\n    a = p.parse_args()\\n    record(a.official_csv, a.sample_csv, a.evaluator, a.official_log, a.export_dir, a.run_dir)\\n", "task3_gan/Pramod_Dindukurthi/src/report.py": "\\"\\"\\"Produce plots and a report draft from actual artifacts, preserving raw logs.\\"\\"\\"\\nimport argparse\\nimport csv\\nimport json\\nfrom pathlib import Path\\n\\nimport matplotlib\\nmatplotlib.use(\\"Agg\\")\\nimport matplotlib.pyplot as plt\\nimport numpy as np\\n\\n\\ndef build(run_dir, export_dir):\\n    run_dir, export_dir = Path(run_dir), Path(export_dir)\\n    history = {}\\n    for path in sorted((run_dir/\\"reproducibility\\"/\\"raw_logs\\").glob(\\"history_*.csv\\")):\\n        with open(path) as f:\\n            for row in csv.DictReader(f):\\n                history[int(row[\\"step\\"])] = {k: float(v) for k, v in row.items()}\\n    meta = json.loads((export_dir/\\"export_manifest.json\\").read_text())\\n    # Resumed repeated steps use the last log segment; never edit original logs.\\n    rows = [history[k] for k in sorted(history) if k <= meta[\\"step\\"]]\\n    plots = run_dir/\\"outputs\\"/\\"plots\\"\\n    plots.mkdir(parents=True, exist_ok=True)\\n    panels = {\\"adversarial\\": [\\"adv_A2B\\", \\"adv_B2A\\", \\"D_A\\", \\"D_B\\"],\\n              \\"cycle_identity\\": [\\"cycle_A\\", \\"cycle_B\\", \\"identity_A\\", \\"identity_B\\"],\\n              \\"gradients\\": [\\"grad_G_A2B\\", \\"grad_G_B2A\\", \\"grad_D_A\\", \\"grad_D_B\\"],\\n              \\"learning_rate\\": [\\"lr\\"]}\\n    for name, fields in panels.items():\\n        fig, ax = plt.subplots(figsize=(9, 4))\\n        for field in fields:\\n            ax.plot([r[\\"step\\"] for r in rows], [r[field] for r in rows], label=field, alpha=0.8)\\n        ax.set(xlabel=\\"Optimizer step\\", ylabel=name.replace(\\"_\\", \\" \\"))\\n        ax.legend(); fig.tight_layout(); fig.savefig(plots/(name+\\".png\\"), dpi=150); plt.close(fig)\\n    with open(export_dir/\\"full_metrics_report.csv\\") as f:\\n        metrics = list(csv.DictReader(f))\\n    if rows:\\n        window = rows[-min(100, len(rows)):]\\n        for r in metrics:\\n            for key in [\\"G_total\\", \\"D_total\\", \\"adv_A2B\\", \\"adv_B2A\\", \\"D_A\\", \\"D_B\\",\\n                        \\"cycle_A\\", \\"cycle_B\\", \\"identity_A\\", \\"identity_B\\",\\n                        \\"grad_G_A2B\\", \\"grad_G_B2A\\", \\"grad_D_A\\", \\"grad_D_B\\"]:\\n                r[key+\\"_last100_mean\\"] = float(np.mean([x[key] for x in window]))\\n            r[\\"training_seconds_to_checkpoint\\"] = rows[-1][\\"training_seconds\\"]\\n            r[\\"images_per_second_to_checkpoint\\"] = rows[-1][\\"source_images_per_second\\"]\\n            r[\\"nan_count\\"] = rows[-1][\\"nan_count\\"]\\n            r[\\"inf_count\\"] = rows[-1][\\"inf_count\\"]\\n            r[\\"peak_gpu_memory_bytes_through_checkpoint\\"] = max(x[\\"peak_memory_bytes\\"] for x in rows)\\n    audit = run_dir/\\"outputs\\"/\\"human_audit\\"/\\"audit_summary.json\\"\\n    if audit.exists():\\n        audit_data = json.loads(audit.read_text())\\n        if audit_data[\\"checkpoint_sha256\\"] != meta[\\"checkpoint_sha256\\"]:\\n            raise ValueError(\\"Human audit belongs to another checkpoint\\")\\n        for r in metrics:\\n            for a in audit_data[\\"results\\"]:\\n                if a[\\"direction\\"] == r[\\"direction\\"]:\\n                    r[\\"human_\\"+a[\\"criterion\\"]+\\"_mean\\"] = a[\\"pooled_mean\\"]\\n                    r[\\"human_\\"+a[\\"criterion\\"]+\\"_agreement_percent\\"] = a[\\"exact_agreement_percent\\"]\\n            r[\\"human_agreement\\"] = \\"exact agreement per criterion; see dedicated columns\\"\\n    for name in [\\"full_metrics_report.csv\\", \\"metrics_report.csv\\"]:\\n        with open(run_dir/name, \\"w\\", newline=\\"\\") as f:\\n            w = csv.DictWriter(f, fieldnames=list(metrics[0])); w.writeheader(); w.writerows(metrics)\\n    lines = [\\"# Task 3 results draft\\", \\"\\", \\"A = Monet; B = Photo.\\", \\"\\",\\n             f\\"Checkpoint SHA-256: `{meta[\'checkpoint_sha256\']}`; step {meta[\'step\']}; {meta[\'weights\']} weights.\\",\\n             f\\"Evaluation split: {meta[\'split\']}. Local metrics are not verified Kaggle scores.\\", \\"\\",\\n             \\"| Direction | Real/generated n | FID | KID | Content cosine |\\", \\"|---|---|---|---|---|\\"]\\n    for r in metrics:\\n        lines.append(f\\"| {r[\'direction\']} | {r[\'n_real\']}/{r[\'n_generated\']} | {r[\'FID\']} | {r[\'KID_mean\']} | {r[\'content_cosine_inception2048\']} |\\")\\n    lines += [\\"\\", \\"## Interpretation to complete from evidence\\", \\"\\",\\n              \\"- Explain architecture, preprocessing, schedule and loss weights using the run manifest.\\",\\n              \\"- Discuss style/content trade-offs, small-reference-set FID uncertainty and KID subset variation.\\",\\n              \\"- Inspect the fixed input/translation/cycle grids; document specific failures and testable fixes.\\",\\n              \\"- Explain generator/discriminator and gradient curves; low cycle loss does not establish good translation.\\",\\n              \\"- Add independently completed human ratings and verified Kaggle scores/rank when available.\\",\\n              \\"- Compare with teammates under the same evaluation protocol.\\", \\"\\",\\n              \\"## Evidence\\", \\"\\", \\"See full_metrics_report.csv, training_summary.json, reproducibility/,\\",\\n              \\"outputs/plots/, evaluation_protocol.json and export_manifest.json in the selected export directory.\\", \\"\\"]\\n    # Draft has a separate name so rerunning never overwrites the student\'s analysis.\\n    (run_dir/\\"results_draft.md\\").write_text(\\"\\\\n\\".join(lines))\\n    print(\\"Wrote metric reports, plots and results_draft.md to\\", run_dir)\\n\\n\\nif __name__ == \\"__main__\\":\\n    p = argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\\"--run-dir\\", required=True)\\n    p.add_argument(\\"--export-dir\\", required=True)\\n    a = p.parse_args()\\n    build(a.run_dir, a.export_dir)\\n", "task3_gan/Pramod_Dindukurthi/src/select_checkpoint.py": "\\"\\"\\"Record a validation-selected checkpoint and publish its COMPLETE image exports locally.\\"\\"\\"\\nimport argparse\\nimport json\\nfrom pathlib import Path\\nimport shutil\\n\\nfrom common import json_write, sha256, utc_now\\nfrom checkpointing import backup_file\\n\\n\\ndef select(checkpoint, export_dir, run_dir, reason):\\n    checkpoint, export_dir, run_dir = Path(checkpoint), Path(export_dir), Path(run_dir)\\n    meta = json.loads((export_dir/\\"export_manifest.json\\").read_text())\\n    if meta[\\"split\\"] != \\"val\\":\\n        raise ValueError(\\"Select checkpoints using the fixed validation protocol\\")\\n    if sha256(checkpoint) != meta[\\"checkpoint_sha256\\"]:\\n        raise ValueError(\\"Checkpoint changed since inference; use the exact archived snapshot\\")\\n    if not (export_dir/\\"full_metrics_report.csv\\").exists():\\n        raise ValueError(\\"Evaluate the candidate before selecting it\\")\\n    target = run_dir/\\"outputs\\"\\n    for name in [\\"pred_A2B\\", \\"pred_B2A\\"]:\\n        if (target/name).exists():\\n            raise FileExistsError(\\"Canonical outputs already exist. Archive the prior selection explicitly first.\\")\\n    backup_file(checkpoint, run_dir/\\"checkpoints\\"/\\"best_model.pt\\")\\n    for name in [\\"pred_A2B\\", \\"pred_B2A\\"]:\\n        shutil.copytree(export_dir/name, target/name)\\n    json_write(run_dir/\\"selection.json\\", {\\"selected_utc\\": utc_now(), \\"reason\\": reason,\\n              \\"checkpoint_sha256\\": meta[\\"checkpoint_sha256\\"], \\"step\\": meta[\\"step\\"],\\n              \\"weights\\": meta[\\"weights\\"], \\"export_directory_name\\": export_dir.name,\\n              \\"note\\": \\"All fixed validation outputs copied; competition input requirements may differ.\\"})\\n    print(\\"Selected checkpoint and complete prediction directories saved. No Kaggle upload performed.\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    p = argparse.ArgumentParser(description=__doc__)\\n    for key in [\\"checkpoint\\", \\"export-dir\\", \\"run-dir\\", \\"reason\\"]:\\n        p.add_argument(\\"--\\"+key, required=True)\\n    a = p.parse_args()\\n    select(a.checkpoint, a.export_dir, a.run_dir, a.reason)\\n", "task3_gan/Pramod_Dindukurthi/src/smoke_test.py": "\\"\\"\\"Synthetic integration tests, not assignment results. No metric weights needed by default.\\"\\"\\"\\nimport argparse\\nimport copy\\nimport csv\\nimport json\\nfrom pathlib import Path\\nimport random\\nimport tempfile\\nimport sys\\n\\nimport numpy as np\\nfrom PIL import Image\\nimport torch\\nimport yaml\\n\\nfrom checkpointing import load_checkpoint\\nfrom common import seed_all, sha256\\nfrom data import prepare, read_splits, UnpairedDataset\\nfrom infer import export\\nfrom metrics import fid, kid, density_coverage, cosine\\nfrom models import Generator, PatchDiscriminator\\nfrom train import Trainer, parser, run\\nfrom audit import prepare as prepare_audit, summarize\\n\\n\\ndef assert_nested(a, b):\\n    if isinstance(a, torch.Tensor):\\n        torch.testing.assert_close(a, b, rtol=0, atol=0)\\n    elif isinstance(a, np.ndarray):\\n        np.testing.assert_array_equal(a, b)\\n    elif isinstance(a, dict):\\n        assert a.keys() == b.keys()\\n        for k in a: assert_nested(a[k], b[k])\\n    elif isinstance(a, (list, tuple)):\\n        assert len(a) == len(b)\\n        for x, y in zip(a, b): assert_nested(x, y)\\n    else:\\n        assert a == b, (a, b)\\n\\n\\ndef main(full_metrics=False):\\n    torch.set_num_threads(1)\\n    root = Path(tempfile.mkdtemp(prefix=\\"data266-task3-smoke-\\"))\\n    data_root = root/\\"data\\"\\n    rng = np.random.default_rng(42)\\n    for folder in [\\"monet_jpg\\", \\"photo_jpg\\"]:\\n        (data_root/folder).mkdir(parents=True)\\n        for i in range(40):\\n            Image.fromarray(rng.integers(0, 256, (64, 64, 3), dtype=np.uint8)).save(data_root/folder/f\\"{i:03}.png\\")\\n    # Exact duplicate must not enter another split.\\n    (data_root/\\"monet_jpg\\"/\\"duplicate.png\\").write_bytes((data_root/\\"monet_jpg\\"/\\"000.png\\").read_bytes())\\n    splits_path = root/\\"splits.json\\"\\n    splits = prepare(data_root, splits_path, val_fraction=0.4)\\n    assert len(splits[\\"domains\\"][\\"A\\"][\\"duplicates\\"]) == 1\\n    assert not set(splits[\\"domains\\"][\\"A\\"][\\"train\\"]) & set(splits[\\"domains\\"][\\"A\\"][\\"val\\"])\\n    member = Path(__file__).resolve().parents[1]\\n    config = yaml.safe_load((member/\\"configs\\"/\\"baseline.yaml\\").read_text())\\n    config.update(image_size=64, load_size=68)\\n    config[\\"model\\"].update(channels=4, blocks=1)\\n    config[\\"train\\"].update(max_steps=4, decay_start=2, num_workers=0, amp=False,\\n                           pool_size=2, save_every=2, preview_every=2, keep_every=2,\\n                           log_every=1, max_hours=1)\\n    config_path = root/\\"smoke.yaml\\"\\n    config_path.write_text(yaml.safe_dump(config))\\n    for variant in [\\"transpose\\", \\"resize\\"]:\\n        g = Generator(4, 1, variant)\\n        x = torch.randn(1, 3, 64, 64)\\n        y = g(x)\\n        assert y.shape == x.shape\\n        assert torch.isfinite(y).all()\\n        d = PatchDiscriminator(4)\\n        assert d(torch.randn(1, 3, 256, 256)).shape == (1, 1, 30, 30)\\n    # Cycle loss alone reaches both generators; correct compositions use both mappings.\\n    seed_all(42)\\n    trainer = Trainer(config, torch.device(\\"cpu\\"))\\n    x = torch.randn(1, 3, 64, 64)\\n    rec = trainer.models[\\"G_B2A\\"](trainer.models[\\"G_A2B\\"](x))\\n    (rec-x).abs().mean().backward()\\n    for k in [\\"G_A2B\\", \\"G_B2A\\"]:\\n        assert sum(float(p.grad.abs().sum()) for p in trainer.models[k].parameters() if p.grad is not None) > 0\\n    # Identical stateful trainer trajectory across a real save, reload and CLI-level resume.\\n    def args(run_dir, extra=None):\\n        return parser().parse_args([\\"--config\\", str(config_path), \\"--data-root\\", str(data_root),\\n                \\"--splits\\", str(splits_path), \\"--run-dir\\", str(run_dir), \\"--device\\", \\"cpu\\"]+(extra or []))\\n    uninterrupted, resumed = root/\\"uninterrupted\\", root/\\"resumed\\"\\n    run(args(uninterrupted))\\n    run(args(resumed, [\\"--stop-after\\", \\"2\\", \\"--backup-dir\\", str(root/\\"backup\\")]))\\n    first_log = sorted((resumed/\\"reproducibility\\"/\\"raw_logs\\").glob(\\"*.log\\"))[0]\\n    first_log_hash = sha256(first_log)\\n    ck = resumed/\\"checkpoints\\"/\\"latest_checkpoint.pt\\"\\n    run(args(resumed, [\\"--resume\\", str(ck)]))\\n    assert sha256(first_log) == first_log_hash, \\"Resume modified an earlier raw log\\"\\n    a, b = load_checkpoint(uninterrupted/\\"checkpoints\\"/\\"latest_checkpoint.pt\\"), load_checkpoint(ck)\\n    for key in [\\"models\\", \\"ema\\", \\"optimizers\\", \\"schedulers\\", \\"scaler\\", \\"pools\\", \\"rng\\", \\"step\\", \\"examples\\"]:\\n        assert_nested(a[key], b[key])\\n    assert b[\\"step\\"] == 4\\n    # A wall-time pause must preserve the same full schedule and resume trajectory.\\n    timed = root/\\"timed_pause\\"\\n    run(args(timed, [\\"--session-hours\\", \\"1e-12\\"]))\\n    timed_ck = timed/\\"checkpoints\\"/\\"latest_checkpoint.pt\\"\\n    paused = load_checkpoint(timed_ck)\\n    assert paused[\\"step\\"] == 1 and paused[\\"config\\"] == config\\n    run(args(timed, [\\"--resume\\", str(timed_ck), \\"--session-hours\\", \\"1\\"]))\\n    continued = load_checkpoint(timed_ck)\\n    for key in [\\"models\\", \\"ema\\", \\"optimizers\\", \\"schedulers\\", \\"scaler\\", \\"pools\\", \\"rng\\", \\"step\\", \\"examples\\"]:\\n        assert_nested(a[key], continued[key])\\n    bad = copy.deepcopy(config); bad[\\"train\\"][\\"lr\\"] *= 2\\n    try:\\n        Trainer(bad, torch.device(\\"cpu\\")).restore(b, sha256(splits_path))\\n    except ValueError: pass\\n    else: raise AssertionError(\\"Changed config was accepted for exact resume\\")\\n    # Independent mathematical checks (not pretrained-model scoring).\\n    features = rng.normal(size=(30, 5))\\n    assert fid(features, features) < 1e-8\\n    np.testing.assert_allclose(fid(features, features+2), 20, rtol=1e-7)\\n    np.testing.assert_allclose(cosine(features, features), 1, atol=1e-7)\\n    dc = density_coverage(features, features, k=3)\\n    assert dc[\\"coverage\\"] == 1 and dc[\\"density\\"] >= 1\\n    assert np.isfinite(kid(features, features, subsets=3)[\\"KID_mean\\"])\\n    exports = resumed/\\"outputs\\"/\\"eval_smoke\\"\\n    meta = export(ck, data_root, splits_path, exports, device=\\"cpu\\")\\n    assert len(meta[\\"records\\"]) == 32\\n    for row in meta[\\"records\\"]:\\n        with Image.open(exports/row[\\"prediction\\"]) as im:\\n            assert im.mode == \\"RGB\\" and im.size == (64, 64) and im.format == \\"JPEG\\"\\n    # The faster official export must preserve the exact fixed-prefix predictions.\\n    fast_dir = resumed/\\"outputs\\"/\\"fast_prefix\\"\\n    fast = export(ck, data_root, splits_path, fast_dir, device=\\"cpu\\",\\n                  max_per_domain=8, predictions_only=True)\\n    assert len(fast[\\"records\\"]) == 16 and fast[\\"predictions_only\\"]\\n    for domain in [\\"A\\", \\"B\\"]:\\n        original = [r for r in meta[\\"records\\"] if r[\\"domain\\"] == domain][:8]\\n        prefix = [r for r in fast[\\"records\\"] if r[\\"domain\\"] == domain]\\n        assert [r[\\"source\\"] for r in prefix] == [r[\\"source\\"] for r in original]\\n        assert [r[\\"prediction_sha256\\"] for r in prefix] == [r[\\"prediction_sha256\\"] for r in original]\\n    assert not (fast_dir/\\"input_A\\").exists() and not (fast_dir/\\"cycle_B\\").exists()\\n    all_fast = export(ck, data_root, splits_path, resumed/\\"outputs\\"/\\"all_prefix\\",\\n                      device=\\"cpu\\", split=\\"all\\", max_per_domain=3, predictions_only=True)\\n    for domain in [\\"A\\", \\"B\\"]:\\n        expected = (splits[\\"domains\\"][domain][\\"train\\"]+splits[\\"domains\\"][domain][\\"val\\"])[:3]\\n        assert [r[\\"source\\"] for r in all_fast[\\"records\\"] if r[\\"domain\\"] == domain] == expected\\n    audit_dir = resumed/\\"outputs\\"/\\"human_audit\\"\\n    prepare_audit(exports, audit_dir)\\n    # Synthetic ratings exercise validation and agreement; never assignment evidence.\\n    for n in [1, 2]:\\n        path = audit_dir/\\"rater_packet\\"/f\\"rater_{n}.csv\\"\\n        with open(path) as f: rows = list(csv.DictReader(f))\\n        for row in rows:\\n            row.update(style=\\"3\\", content=\\"4\\", artifacts=\\"2\\")\\n        with open(path, \\"w\\", newline=\\"\\") as f:\\n            w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)\\n    assert all(r[\\"exact_agreement_percent\\"] == 100 for r in summarize(audit_dir))\\n    if full_metrics:\\n        sys.path.insert(0, str(member))\\n        from evaluate_local import evaluate\\n        from report import build\\n        from select_checkpoint import select\\n        evaluate(exports, resumed, \\"cpu\\", batch_size=4, subsets=3, k=3)\\n        build(resumed, exports)\\n        select(ck, exports, resumed, \\"Synthetic integration test only\\")\\n    result = {\\"status\\": \\"PASS\\", \\"synthetic_only\\": True, \\"temporary_artifacts\\": str(root),\\n              \\"exact_cpu_resume\\": True, \\"full_pretrained_metrics\\": full_metrics}\\n    print(json.dumps(result, indent=2))\\n    return result\\n\\n\\nif __name__ == \\"__main__\\":\\n    p = argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\\"--full-metrics\\", action=\\"store_true\\", help=\\"Downloads Inception/LPIPS weights on first use\\")\\n    main(p.parse_args().full_metrics)\\n", "task3_gan/Pramod_Dindukurthi/src/train.py": "\\"\\"\\"Step-budgeted CycleGAN training. CLI and notebook use the same entry point.\\"\\"\\"\\nimport argparse\\nimport contextlib\\nimport csv\\nimport json\\nimport math\\nfrom pathlib import Path\\nimport shutil\\nimport signal\\nimport sys\\nimport time\\nimport traceback\\nimport uuid\\n\\nimport torch\\nfrom torch import nn\\nfrom torch.utils.data import DataLoader\\nfrom PIL import Image\\n\\nfrom common import (Tee, device_for, json_write, manifest, read_config, save_environment,\\n                    seed_all, sha256, utc_now)\\nfrom data import UnpairedDataset, read_splits, rgb, to_image, transform\\nfrom models import build_models\\nfrom checkpointing import (ReplayPool, atomic_save, backup_file, load_checkpoint,\\n                           make_ema, restore_rng, rng_state, update_ema)\\n\\nSTOP_REQUESTED = False\\n\\n\\ndef request_stop(signum, frame):\\n    global STOP_REQUESTED\\n    STOP_REQUESTED = True\\n    print(\\"Stop requested; finishing the current optimizer step and saving.\\", flush=True)\\n\\n\\ndef grad_norm(model):\\n    grads = [p.grad.detach().float().norm(2) ** 2 for p in model.parameters() if p.grad is not None]\\n    return torch.stack(grads).sum().sqrt() if grads else torch.tensor(0.)\\n\\n\\nclass Trainer:\\n    def __init__(self, config, device):\\n        self.config, self.device = config, device\\n        self.models = build_models(config, device)\\n        self.ema = {k: make_ema(self.models[k]) for k in [\\"G_A2B\\", \\"G_B2A\\"]}\\n        t = config[\\"train\\"]\\n        self.optimizers = {\\n            \\"G\\": torch.optim.Adam(list(self.models[\\"G_A2B\\"].parameters()) +\\n                                  list(self.models[\\"G_B2A\\"].parameters()),\\n                                  lr=t[\\"lr\\"], betas=(t[\\"beta1\\"], t[\\"beta2\\"])),\\n            \\"D\\": torch.optim.Adam(list(self.models[\\"D_A\\"].parameters()) +\\n                                  list(self.models[\\"D_B\\"].parameters()),\\n                                  lr=t[\\"lr\\"], betas=(t[\\"beta1\\"], t[\\"beta2\\"]))}\\n        def schedule(step):\\n            return 1.0 - max(0, step-t[\\"decay_start\\"]) / (t[\\"max_steps\\"]-t[\\"decay_start\\"])\\n        self.schedulers = {k: torch.optim.lr_scheduler.LambdaLR(v, schedule)\\n                           for k, v in self.optimizers.items()}\\n        self.amp = bool(t[\\"amp\\"] and device.type == \\"cuda\\")\\n        if hasattr(torch.amp, \\"GradScaler\\"):\\n            self.scaler = torch.amp.GradScaler(\\"cuda\\", enabled=self.amp)\\n        else:\\n            # PyTorch 2.1/2.2 expose the CUDA scaler through this namespace.\\n            self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)\\n        self.pools = {d: ReplayPool(t[\\"pool_size\\"]) for d in [\\"A\\", \\"B\\"]}\\n        self.step, self.training_seconds, self.examples = 0, 0.0, 0\\n        self.nan_count, self.inf_count = 0, 0\\n\\n    def autocast(self):\\n        return torch.autocast(\\"cuda\\", dtype=torch.float16) if self.amp else contextlib.nullcontext()\\n\\n    def finite(self, values):\\n        for value in values:\\n            self.nan_count += int(torch.isnan(value).sum().item())\\n            self.inf_count += int(torch.isinf(value).sum().item())\\n        if self.nan_count or self.inf_count:\\n            print(\\"NONFINITE_EVENT\\", json.dumps({\\"attempted_step\\": self.step+1,\\n                  \\"nan_count\\": self.nan_count, \\"inf_count\\": self.inf_count}), flush=True)\\n            raise FloatingPointError(\\"Non-finite loss/gradient. Stop; resume the last complete checkpoint with a reviewed configuration.\\")\\n\\n    def update(self, batch):\\n        A, B = (batch[d].to(self.device, non_blocking=True) for d in [\\"A\\", \\"B\\"])\\n        G, F = self.models[\\"G_A2B\\"], self.models[\\"G_B2A\\"]\\n        DA, DB = self.models[\\"D_A\\"], self.models[\\"D_B\\"]\\n        t = self.config[\\"train\\"]\\n        DA.requires_grad_(False)\\n        DB.requires_grad_(False)\\n        self.optimizers[\\"G\\"].zero_grad(set_to_none=True)\\n        with self.autocast():\\n            fake_B, fake_A = G(A), F(B)\\n            rec_A, rec_B = F(fake_B), G(fake_A)\\n            losses = {\\"adv_A2B\\": ((DB(fake_B)-1)**2).mean(),\\n                      \\"adv_B2A\\": ((DA(fake_A)-1)**2).mean(),\\n                      \\"cycle_A\\": (rec_A-A).abs().mean(), \\"cycle_B\\": (rec_B-B).abs().mean()}\\n            if t[\\"identity_weight\\"]:\\n                losses.update(identity_A=(F(A)-A).abs().mean(), identity_B=(G(B)-B).abs().mean())\\n            else:\\n                losses.update(identity_A=A.new_tensor(0.), identity_B=B.new_tensor(0.))\\n            loss_G = (losses[\\"adv_A2B\\"] + losses[\\"adv_B2A\\"] +\\n                      t[\\"cycle_weight\\"]*(losses[\\"cycle_A\\"]+losses[\\"cycle_B\\"]) +\\n                      t[\\"identity_weight\\"]*(losses[\\"identity_A\\"]+losses[\\"identity_B\\"]))\\n        self.finite([loss_G])\\n        self.scaler.scale(loss_G).backward()\\n        self.scaler.unscale_(self.optimizers[\\"G\\"])\\n        norms = {\\"grad_\\"+k: grad_norm(self.models[k]) for k in [\\"G_A2B\\", \\"G_B2A\\"]}\\n        self.finite(list(norms.values()))\\n        self.scaler.step(self.optimizers[\\"G\\"])\\n        DA.requires_grad_(True)\\n        DB.requires_grad_(True)\\n        self.optimizers[\\"D\\"].zero_grad(set_to_none=True)\\n        with self.autocast():\\n            old_A, old_B = self.pools[\\"A\\"].query(fake_A), self.pools[\\"B\\"].query(fake_B)\\n            losses[\\"D_A\\"] = 0.5*(((DA(A)-1)**2).mean() + (DA(old_A)**2).mean())\\n            losses[\\"D_B\\"] = 0.5*(((DB(B)-1)**2).mean() + (DB(old_B)**2).mean())\\n            loss_D = losses[\\"D_A\\"] + losses[\\"D_B\\"]\\n        self.finite([loss_D])\\n        self.scaler.scale(loss_D).backward()\\n        self.scaler.unscale_(self.optimizers[\\"D\\"])\\n        norms.update({\\"grad_\\"+k: grad_norm(self.models[k]) for k in [\\"D_A\\", \\"D_B\\"]})\\n        self.finite(list(norms.values()))\\n        self.scaler.step(self.optimizers[\\"D\\"])\\n        self.scaler.update()\\n        for s in self.schedulers.values():\\n            s.step()\\n        for k in self.ema:\\n            update_ema(self.ema[k], self.models[k], t[\\"ema_decay\\"])\\n        self.step += 1\\n        self.examples += len(A) + len(B)\\n        losses.update(G_total=loss_G, D_total=loss_D, **norms)\\n        return {k: float(v.detach()) for k, v in losses.items()}\\n\\n    def state(self, split_hash):\\n        return {\\"schema\\": 1, \\"step\\": self.step, \\"config\\": self.config, \\"split_sha256\\": split_hash,\\n                \\"models\\": {k: m.state_dict() for k, m in self.models.items()},\\n                \\"ema\\": {k: m.state_dict() for k, m in self.ema.items()},\\n                \\"optimizers\\": {k: o.state_dict() for k, o in self.optimizers.items()},\\n                \\"schedulers\\": {k: s.state_dict() for k, s in self.schedulers.items()},\\n                \\"scaler\\": self.scaler.state_dict(), \\"pools\\": {k: p.images for k, p in self.pools.items()},\\n                \\"rng\\": rng_state(), \\"examples\\": self.examples,\\n                \\"training_seconds\\": self.training_seconds,\\n                \\"nan_count\\": self.nan_count, \\"inf_count\\": self.inf_count}\\n\\n    def restore(self, state, split_hash):\\n        if state[\\"config\\"] != self.config or state[\\"split_sha256\\"] != split_hash:\\n            raise ValueError(\\"Resume requires identical config and split. New experiments need a separate run.\\")\\n        for k, m in self.models.items():\\n            m.load_state_dict(state[\\"models\\"][k])\\n        for k, m in self.ema.items():\\n            m.load_state_dict(state[\\"ema\\"][k])\\n        for k, o in self.optimizers.items():\\n            o.load_state_dict(state[\\"optimizers\\"][k])\\n        for k, s in self.schedulers.items():\\n            s.load_state_dict(state[\\"schedulers\\"][k])\\n        self.scaler.load_state_dict(state[\\"scaler\\"])\\n        for k, p in self.pools.items():\\n            p.images = [image.cpu() for image in state[\\"pools\\"][k]]\\n        for k in [\\"step\\", \\"examples\\", \\"training_seconds\\", \\"nan_count\\", \\"inf_count\\"]:\\n            setattr(self, k, state[k])\\n        restore_rng(state[\\"rng\\"])\\n\\n\\n@torch.inference_mode()\\ndef preview(trainer, root, splits, out):\\n    size = trainer.config[\\"image_size\\"]\\n    rows, scores = [], {}\\n    for d, gkey, fkey in [(\\"A\\", \\"G_A2B\\", \\"G_B2A\\"), (\\"B\\", \\"G_B2A\\", \\"G_A2B\\")]:\\n        values = []\\n        for rel in splits[\\"domains\\"][d][\\"val\\"][:4]:\\n            x = transform(rgb(Path(root)/rel), size).unsqueeze(0).to(trainer.device)\\n            y = trainer.ema[gkey](x)\\n            rec = trainer.ema[fkey](y)\\n            values.append(float((rec-x).abs().mean()))\\n            row = Image.new(\\"RGB\\", (size*3, size))\\n            for i, tensor in enumerate([x[0], y[0], rec[0]]):\\n                row.paste(to_image(tensor), (i*size, 0))\\n            rows.append(row)\\n        scores[\\"preview_cycle_\\"+d] = sum(values)/len(values)\\n    grid = Image.new(\\"RGB\\", (size*3, size*len(rows)))\\n    for i, row in enumerate(rows):\\n        grid.paste(row, (0, i*size))\\n    Path(out).parent.mkdir(parents=True, exist_ok=True)\\n    grid.save(out)\\n    return scores\\n\\n\\ndef run(args):\\n    if args.session_hours is not None and (not math.isfinite(args.session_hours) or args.session_hours <= 0):\\n        raise ValueError(\\"--session-hours must be finite and positive\\")\\n    config = read_config(args.config)\\n    device = device_for(args.device)\\n    seed_all(config[\\"seed\\"], config[\\"train\\"][\\"deterministic\\"])\\n    run_dir = Path(args.run_dir).resolve()\\n    ckdir = run_dir/\\"checkpoints\\"\\n    if (ckdir/\\"latest_checkpoint.pt\\").exists() and not args.resume:\\n        raise FileExistsError(\\"Run already has a checkpoint: use --resume or a new --run-dir\\")\\n    ckdir.mkdir(parents=True, exist_ok=True)\\n    segment = utc_now().replace(\\":\\", \\"-\\") + \\"_\\" + uuid.uuid4().hex[:6]\\n    evidence = run_dir/\\"reproducibility\\"\\n    logs = evidence/\\"raw_logs\\"\\n    logs.mkdir(parents=True, exist_ok=True)\\n    with open(logs/f\\"train_{segment}.log\\", \\"x\\", buffering=1) as log:\\n        with contextlib.redirect_stdout(Tee(sys.stdout, log)), contextlib.redirect_stderr(Tee(sys.stderr, log)):\\n            try:\\n                result = train_run(args, config, device, run_dir, segment)\\n            except BaseException:\\n                traceback.print_exc()\\n                print(\\"The last complete atomic checkpoint is retained. Never resume a partially updated step.\\")\\n                raise\\n    if args.backup_dir:\\n        for path in logs.glob(\\"*\\"):\\n            backup_file(path, Path(args.backup_dir)/\\"reproducibility\\"/\\"raw_logs\\"/path.name)\\n    return result\\n\\n\\ndef train_run(args, config, device, run_dir, segment):\\n    started = time.monotonic()\\n    print(\\"Started\\", utc_now(), \\"device\\", device, \\"config\\", json.dumps(config), flush=True)\\n    splits = read_splits(args.splits, args.data_root, verify=True)\\n    split_hash = sha256(args.splits)\\n    split_dest = run_dir/\\"data_processed\\"/\\"splits.json\\"\\n    split_dest.parent.mkdir(parents=True, exist_ok=True)\\n    if Path(args.splits).resolve() != split_dest.resolve():\\n        shutil.copy2(args.splits, split_dest)\\n    t = config[\\"train\\"]\\n    session_hours = args.session_hours if args.session_hours is not None else t[\\"max_hours\\"]\\n    trainer = Trainer(config, device)\\n    if args.resume:\\n        trainer.restore(load_checkpoint(args.resume, device), split_hash)\\n        print(\\"Resumed completed optimizer step\\", trainer.step)\\n    info = manifest(config, args.code_version)\\n    info.update(split_sha256=split_hash, start_step=trainer.step, segment=segment,\\n                session_hours=session_hours,\\n                parameters={k: sum(p.numel() for p in m.parameters()) for k, m in trainer.models.items()})\\n    mpath = run_dir/\\"reproducibility\\"/\\"manifests\\"/f\\"run_{segment}.json\\"\\n    json_write(mpath, info)\\n    save_environment(mpath.with_suffix(\\".requirements.txt\\"))\\n    dataset = UnpairedDataset(args.data_root, splits, config[\\"image_size\\"], config[\\"load_size\\"], config[\\"seed\\"])\\n    stop_step = min(t[\\"max_steps\\"], trainer.step + args.stop_after) if args.stop_after else t[\\"max_steps\\"]\\n    loader = DataLoader(dataset, batch_size=t[\\"batch_size\\"],\\n                        sampler=range(trainer.step*t[\\"batch_size\\"], stop_step*t[\\"batch_size\\"]),\\n                        num_workers=t[\\"num_workers\\"], pin_memory=device.type == \\"cuda\\",\\n                        generator=torch.Generator().manual_seed(config[\\"seed\\"]), drop_last=True)\\n    if device.type == \\"cuda\\":\\n        torch.cuda.reset_peak_memory_stats(device)\\n    checkpoint = run_dir/\\"checkpoints\\"/\\"latest_checkpoint.pt\\"\\n    last_save = time.monotonic()\\n    def save():\\n        nonlocal last_save\\n        atomic_save(trainer.state(split_hash), checkpoint)\\n        if trainer.step % t[\\"keep_every\\"] == 0 and trainer.step:\\n            backup_file(checkpoint, checkpoint.parent/\\"periodic\\"/f\\"step_{trainer.step:08d}.pt\\")\\n        if args.backup_dir:\\n            backup = Path(args.backup_dir)\\n            if backup.resolve() == run_dir:\\n                raise ValueError(\\"Backup directory must differ from run directory\\")\\n            backup_file(checkpoint, backup/\\"checkpoints\\"/checkpoint.name)\\n            backup_file(split_dest, backup/\\"data_processed\\"/\\"splits.json\\")\\n            backup_file(mpath, backup/\\"reproducibility\\"/\\"manifests\\"/mpath.name)\\n            for f in (run_dir/\\"reproducibility\\"/\\"raw_logs\\").glob(\\"*\\"):\\n                backup_file(f, backup/\\"reproducibility\\"/\\"raw_logs\\"/f.name)\\n        last_save = time.monotonic()\\n        print(\\"Checkpoint saved\\", trainer.step, flush=True)\\n    history_path = run_dir/\\"reproducibility\\"/\\"raw_logs\\"/f\\"history_{segment}.csv\\"\\n    with open(history_path, \\"x\\", newline=\\"\\", buffering=1) as f:\\n        writer = None\\n        tick = time.monotonic()\\n        for batch in loader:\\n            # Timer includes data wait and optimizer work, excludes preview/checkpoint overhead.\\n            losses = trainer.update(batch)\\n            if device.type == \\"cuda\\":\\n                torch.cuda.synchronize(device)\\n            elapsed = time.monotonic() - tick\\n            trainer.training_seconds += elapsed\\n            row = {\\"step\\": trainer.step, \\"epoch_fraction\\": trainer.step*t[\\"batch_size\\"]/len(dataset),\\n                   **losses, \\"lr\\": trainer.optimizers[\\"G\\"].param_groups[0][\\"lr\\"],\\n                   \\"step_seconds\\": elapsed, \\"training_seconds\\": trainer.training_seconds,\\n                   \\"source_images_per_second\\": trainer.examples/max(trainer.training_seconds, 1e-9),\\n                   \\"nan_count\\": trainer.nan_count, \\"inf_count\\": trainer.inf_count,\\n                   \\"peak_memory_bytes\\": torch.cuda.max_memory_allocated(device) if device.type == \\"cuda\\" else 0}\\n            if writer is None:\\n                writer = csv.DictWriter(f, fieldnames=list(row)); writer.writeheader()\\n            writer.writerow(row)\\n            if trainer.step % t[\\"log_every\\"] == 0 or trainer.step == 1:\\n                print(json.dumps(row), flush=True)\\n            if trainer.step % t[\\"preview_every\\"] == 0:\\n                vals = preview(trainer, args.data_root, splits,\\n                               run_dir/\\"outputs\\"/\\"plots\\"/f\\"preview_{trainer.step:08d}.png\\")\\n                print(\\"Fixed EMA validation preview (not a selection score):\\", vals, flush=True)\\n            if trainer.step % t[\\"save_every\\"] == 0 or time.monotonic()-last_save >= t[\\"save_minutes\\"]*60:\\n                save()\\n            if STOP_REQUESTED or time.monotonic()-started >= session_hours*3600:\\n                print(\\"Session time budget reached; saving for continuation.\\")\\n                break\\n            tick = time.monotonic()\\n    save()\\n    summary = {**info, \\"ended_utc\\": utc_now(), \\"end_step\\": trainer.step,\\n               \\"session_wall_seconds\\": time.monotonic()-started,\\n               \\"training_seconds\\": trainer.training_seconds, \\"source_images_seen\\": trainer.examples,\\n               \\"source_images_per_second\\": trainer.examples/max(trainer.training_seconds, 1e-9),\\n               \\"nan_count\\": trainer.nan_count, \\"inf_count\\": trainer.inf_count,\\n               \\"peak_gpu_memory_bytes\\": torch.cuda.max_memory_allocated(device) if device.type == \\"cuda\\" else None,\\n               \\"checkpoint\\": \\"checkpoints/latest_checkpoint.pt\\", \\"checkpoint_sha256\\": sha256(checkpoint),\\n               \\"schedule_complete\\": trainer.step >= t[\\"max_steps\\"]}\\n    json_write(run_dir/\\"training_summary.json\\", summary)\\n    json_write(run_dir/\\"reproducibility\\"/\\"manifests\\"/f\\"completed_{segment}.json\\", summary)\\n    if args.backup_dir:\\n        backup_file(run_dir/\\"training_summary.json\\", Path(args.backup_dir)/\\"training_summary.json\\")\\n    print(\\"Completed\\", json.dumps(summary), flush=True)\\n    return summary\\n\\n\\ndef parser():\\n    p = argparse.ArgumentParser(description=__doc__)\\n    for name in [\\"config\\", \\"data-root\\", \\"splits\\", \\"run-dir\\"]:\\n        p.add_argument(\\"--\\"+name, required=True)\\n    p.add_argument(\\"--device\\", default=\\"auto\\")\\n    p.add_argument(\\"--resume\\")\\n    p.add_argument(\\"--backup-dir\\")\\n    p.add_argument(\\"--code-version\\", default=\\"uncommitted-source-hash-recorded\\")\\n    p.add_argument(\\"--stop-after\\", type=int, help=\\"Stop after N additional steps; preserves full LR schedule\\")\\n    p.add_argument(\\"--session-hours\\", type=float,\\n                   help=\\"Pause and checkpoint after this invocation\'s wall-clock budget; preserves full LR schedule\\")\\n    return p\\n\\n\\nif __name__ == \\"__main__\\":\\n    signal.signal(signal.SIGINT, request_stop)\\n    signal.signal(signal.SIGTERM, request_stop)\\n    run(parser().parse_args())\\n", "task3_gan/Pramod_Dindukurthi/src/task3_lab.ipynb": "{\\n \\"nbformat\\": 4,\\n \\"nbformat_minor\\": 5,\\n \\"metadata\\": {\\n  \\"kernelspec\\": {\\n   \\"display_name\\": \\"Python 3\\",\\n   \\"language\\": \\"python\\",\\n   \\"name\\": \\"python3\\"\\n  },\\n  \\"language_info\\": {\\n   \\"name\\": \\"python\\",\\n   \\"version\\": \\"3.10\\"\\n  }\\n },\\n \\"cells\\": [\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"# DATA266 Task 3 \\\\u2014 CycleGAN lab runner\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"**A = Monet, B = Photo.** This notebook orchestrates the sibling Python scripts. It is not a standalone replacement for the project ZIP.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"Run in lab Jupyter, VS Code, or Colab connected to the **lab machine\'s local runtime**. Keep the entire extracted repository on that machine. Git is not required there.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"Read the member README before full training. No assignment results exist until you run this notebook on the permitted dataset. Smoke-test data and ratings are synthetic and must never appear as lab results.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"## 1. Locate the extracted project\\\\n\\",\\n    \\"Edit `PROJECT_ROOT` if automatic detection fails. Paths belong to this runtime, not your laptop. For Google-hosted Colab, first copy/extract the project into its runtime. Opening the notebook alone does not copy sibling scripts.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-000\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"from pathlib import Path\\\\n\\",\\n    \\"import sys, os, json, subprocess, signal, shutil, time\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"# If needed replace None with Path(\\\\\\"the extracted project folder\\\\\\") on THIS machine.\\\\n\\",\\n    \\"PROJECT_ROOT = None\\\\n\\",\\n    \\"if PROJECT_ROOT is None:\\\\n\\",\\n    \\"    candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd()/\\\\\\"data266-lab1-pair17\\\\\\"]\\\\n\\",\\n    \\"    PROJECT_ROOT = next((p for p in candidates if (p/\\\\\\"task3_gan\\\\\\"/\\\\\\"Pramod_Dindukurthi\\\\\\"/\\\\\\"src\\\\\\"/\\\\\\"train.py\\\\\\").exists()), None)\\\\n\\",\\n    \\"if PROJECT_ROOT is None:\\\\n\\",\\n    \\"    raise RuntimeError(\\\\\\"Set PROJECT_ROOT above to the folder containing task3_gan, then rerun this cell.\\\\\\")\\\\n\\",\\n    \\"PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()\\\\n\\",\\n    \\"MEMBER = PROJECT_ROOT/\\\\\\"task3_gan\\\\\\"/\\\\\\"Pramod_Dindukurthi\\\\\\"\\\\n\\",\\n    \\"SRC = MEMBER/\\\\\\"src\\\\\\"\\\\n\\",\\n    \\"os.chdir(MEMBER)\\\\n\\",\\n    \\"print(\\\\\\"Project found:\\\\\\", PROJECT_ROOT)\\\\n\\",\\n    \\"print(\\\\\\"Kernel Python:\\\\\\", sys.executable)\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-001\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 2. Verify PyTorch, then install evaluation dependencies\\\\n\\",\\n    \\"Use the lab\'s matching CUDA PyTorch/torchvision pair. If this cell cannot import them, select the correct lab kernel or follow the lab\'s installation instructions. Installing requirements does not configure the NVIDIA driver.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-002\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"import torch, torchvision\\\\n\\",\\n    \\"print(\\\\\\"PyTorch:\\\\\\", torch.__version__, \\\\\\"torchvision:\\\\\\", torchvision.__version__)\\\\n\\",\\n    \\"print(\\\\\\"CUDA:\\\\\\", torch.version.cuda, \\\\\\"available:\\\\\\", torch.cuda.is_available())\\\\n\\",\\n    \\"if torch.cuda.is_available():\\\\n\\",\\n    \\"    print(\\\\\\"GPU:\\\\\\", torch.cuda.get_device_name(0))\\\\n\\",\\n    \\"    print(\\\\\\"VRAM GiB:\\\\\\", round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))\\\\n\\",\\n    \\"else:\\\\n\\",\\n    \\"    print(\\\\\\"CPU smoke tests work, but do not start full CycleGAN training until CUDA is available.\\\\\\")\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-003\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"# Run once in a new environment. Restart the kernel if an already-imported package changes.\\\\n\\",\\n    \\"subprocess.run([sys.executable, \\\\\\"-m\\\\\\", \\\\\\"pip\\\\\\", \\\\\\"install\\\\\\", \\\\\\"-r\\\\\\", str(MEMBER/\\\\\\"requirements.txt\\\\\\")], check=True)\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-004\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 3. Runtime paths and provenance\\\\n\\",\\n    \\"Set `DATA_ROOT` to the parent of `monet_jpg/` and `photo_jpg/`. Set `BACKUP_DIR` to a **persistent** location that survives the lab reservation.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"For a second experiment use a new `RUN_DIR`; never overwrite the baseline. Saving the notebook does not save checkpoints. With a local runtime, normal Colab `drive.mount()` is unavailable. Use the lab\'s supported persistent storage. With a Google-hosted runtime, mount Drive separately and set the backup path there.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-005\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"DATA_ROOT = PROJECT_ROOT/\\\\\\"task3_gan\\\\\\"/\\\\\\"data\\\\\\"\\\\n\\",\\n    \\"RUN_DIR = MEMBER                     # matches required member folder layout\\\\n\\",\\n    \\"BACKUP_DIR = None                    # REQUIRED before full training: Path(\\\\\\"persistent destination\\\\\\")\\\\n\\",\\n    \\"CONFIG_NAME = \\\\\\"baseline\\\\\\"             # baseline or resizeconv\\\\n\\",\\n    \\"CODE_VERSION = \\\\\\"uncommitted\\\\\\"         # paste laptop commit SHA or a code-bundle label\\\\n\\",\\n    \\"RUN_ID = \\\\\\"pramod_baseline_seed42\\\\\\"     # unique label for team evidence collection\\\\n\\",\\n    \\"TRAIN_HOURS = 12.0                   # full learning-rate schedule, including later continuation\\\\n\\",\\n    \\"SESSION_HOURS = 4.0                  # this phase only; evaluate after four hours, then use 8.0\\\\n\\",\\n    \\"NUM_WORKERS = 2                      # use 0 if multiprocessing fails on the lab environment\\\\n\\",\\n    \\"BATCH_SIZE = 1\\\\n\\",\\n    \\"RUN_DIR = Path(RUN_DIR).expanduser().resolve()\\\\n\\",\\n    \\"DATA_ROOT = Path(DATA_ROOT).expanduser().resolve()\\\\n\\",\\n    \\"if BACKUP_DIR is not None:\\\\n\\",\\n    \\"    BACKUP_DIR = Path(BACKUP_DIR).expanduser().resolve()\\\\n\\",\\n    \\"SPLITS = RUN_DIR/\\\\\\"data_processed\\\\\\"/\\\\\\"splits.json\\\\\\"\\\\n\\",\\n    \\"for folder in [\\\\\\"monet_jpg\\\\\\", \\\\\\"photo_jpg\\\\\\"]:\\\\n\\",\\n    \\"    assert (DATA_ROOT/folder).is_dir(), f\\\\\\"Missing {DATA_ROOT/folder}\\\\\\"\\\\n\\",\\n    \\"print(\\\\\\"Data:\\\\\\", DATA_ROOT, \\\\\\"Run:\\\\\\", RUN_DIR, \\\\\\"Backup:\\\\\\", BACKUP_DIR)\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-006\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"def launch(script, *arguments):\\\\n\\",\\n    \\"    \\\\\\"\\\\\\"\\\\\\"Run in this kernel\'s Python environment; stream output and propagate failures.\\\\\\"\\\\\\"\\\\\\"\\\\n\\",\\n    \\"    command = [sys.executable, \\\\\\"-u\\\\\\", str(script), *map(str, arguments)]\\\\n\\",\\n    \\"    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\\\\n\\",\\n    \\"                               text=True, bufsize=1, cwd=MEMBER)\\\\n\\",\\n    \\"    try:\\\\n\\",\\n    \\"        for line in process.stdout:\\\\n\\",\\n    \\"            print(line, end=\\\\\\"\\\\\\", flush=True)\\\\n\\",\\n    \\"        status = process.wait()\\\\n\\",\\n    \\"    except KeyboardInterrupt:\\\\n\\",\\n    \\"        process.send_signal(signal.SIGINT)\\\\n\\",\\n    \\"        print(\\\\\\"Stop requested. Trainer will finish a complete step and save; wait for it to exit.\\\\\\")\\\\n\\",\\n    \\"        process.wait()\\\\n\\",\\n    \\"        raise\\\\n\\",\\n    \\"    if status:\\\\n\\",\\n    \\"        raise RuntimeError(f\\\\\\"Command failed with exit code {status}; inspect the output above.\\\\\\")\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"launch(SRC/\\\\\\"preflight.py\\\\\\", \\\\\\"--data-root\\\\\\", DATA_ROOT)\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-007\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 4. Smoke test on CPU\\\\n\\",\\n    \\"Tests both architectures, cycle gradients, exact save/reload/resume against uninterrupted training, image export, metric mathematics and audit-sheet processing. This takes no training dataset and produces only temporary synthetic evidence.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-008\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"launch(SRC/\\\\\\"smoke_test.py\\\\\\")\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-009\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 5. Audit images and freeze splits\\\\n\\",\\n    \\"Run once. Reuse the same split manifest for every comparable experiment. Exact decoded-pixel duplicates are excluded before splitting; visually inspect possible near duplicates separately. Do not put official hidden-test images into the development folders.\\\\n\\",\\n    \\"The default 15% validation split supports the planned 15 examples per direction only if each domain has enough images.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-010\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"if SPLITS.exists():\\\\n\\",\\n    \\"    print(\\\\\\"Reusing existing split manifest:\\\\\\", SPLITS)\\\\n\\",\\n    \\"else:\\\\n\\",\\n    \\"    launch(SRC/\\\\\\"data.py\\\\\\", \\\\\\"--data-root\\\\\\", DATA_ROOT, \\\\\\"--output\\\\\\", SPLITS, \\\\\\"--seed\\\\\\", 42, \\\\\\"--val-fraction\\\\\\", 0.15)\\\\n\\",\\n    \\"split_info = json.loads(SPLITS.read_text())\\\\n\\",\\n    \\"for d in [\\\\\\"A\\\\\\", \\\\\\"B\\\\\\"]:\\\\n\\",\\n    \\"    part = split_info[\\\\\\"domains\\\\\\"][d]\\\\n\\",\\n    \\"    print(d, \\\\\\"train\\\\\\", len(part[\\\\\\"train\\\\\\"]), \\\\\\"validation\\\\\\", len(part[\\\\\\"val\\\\\\"]), \\\\\\"duplicates\\\\\\", len(part[\\\\\\"duplicates\\\\\\"]))\\\\n\\",\\n    \\"    if len(part[\\\\\\"val\\\\\\"]) < 15:\\\\n\\",\\n    \\"        raise RuntimeError(\\\\\\"The planned audit needs 15 validation samples per domain. Decide a suitable split before training.\\\\\\")\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-011\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 6. Short GPU throughput benchmark\\\\n\\",\\n    \\"This is a separate disposable run, never the assignment baseline. It measures 30 updates using your actual model/data. The first few updates include warm-up effects. Do not compare its image quality or scores as if it were a trained model.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-012\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"import yaml, csv, statistics\\\\n\\",\\n    \\"assert torch.cuda.is_available(), \\\\\\"Select a CUDA-capable lab kernel before benchmarking.\\\\\\"\\\\n\\",\\n    \\"base_config = yaml.safe_load((MEMBER/\\\\\\"configs\\\\\\"/(CONFIG_NAME+\\\\\\".yaml\\\\\\")).read_text())\\\\n\\",\\n    \\"base_config[\\\\\\"train\\\\\\"].update(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)\\\\n\\",\\n    \\"BENCH_DIR = RUN_DIR/\\\\\\"data_processed\\\\\\"/(\\\\\\"benchmark_\\\\\\"+CONFIG_NAME)\\\\n\\",\\n    \\"BENCH_DIR.mkdir(parents=True, exist_ok=True)\\\\n\\",\\n    \\"BENCH_CONFIG = BENCH_DIR/\\\\\\"benchmark.yaml\\\\\\"\\\\n\\",\\n    \\"if not (BENCH_DIR/\\\\\\"training_summary.json\\\\\\").exists():\\\\n\\",\\n    \\"    BENCH_CONFIG.write_text(yaml.safe_dump(base_config))\\\\n\\",\\n    \\"    launch(SRC/\\\\\\"train.py\\\\\\", \\\\\\"--config\\\\\\", BENCH_CONFIG, \\\\\\"--data-root\\\\\\", DATA_ROOT,\\\\n\\",\\n    \\"           \\\\\\"--splits\\\\\\", SPLITS, \\\\\\"--run-dir\\\\\\", BENCH_DIR, \\\\\\"--device\\\\\\", \\\\\\"cuda\\\\\\", \\\\\\"--stop-after\\\\\\", 30,\\\\n\\",\\n    \\"           \\\\\\"--code-version\\\\\\", CODE_VERSION)\\\\n\\",\\n    \\"else:\\\\n\\",\\n    \\"    assert yaml.safe_load(BENCH_CONFIG.read_text()) == base_config, \\\\\\"Benchmark config changed; choose a fresh benchmark directory.\\\\\\"\\\\n\\",\\n    \\"    print(\\\\\\"Reusing completed benchmark.\\\\\\")\\\\n\\",\\n    \\"history_file = sorted((BENCH_DIR/\\\\\\"reproducibility\\\\\\"/\\\\\\"raw_logs\\\\\\").glob(\\\\\\"history_*.csv\\\\\\"))[-1]\\\\n\\",\\n    \\"with open(history_file) as f:\\\\n\\",\\n    \\"    benchmark_rows = list(csv.DictReader(f))\\\\n\\",\\n    \\"seconds_per_step = statistics.median(float(r[\\\\\\"step_seconds\\\\\\"]) for r in benchmark_rows[-20:])\\\\n\\",\\n    \\"suggested_steps = max(2, int(TRAIN_HOURS*3600*0.85/seconds_per_step))\\\\n\\",\\n    \\"print(\\\\\\"Median seconds/step:\\\\\\", seconds_per_step)\\\\n\\",\\n    \\"print(\\\\\\"Suggested total steps including 15% overhead allowance:\\\\\\", suggested_steps)\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-013\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 7. Freeze the training configuration and verify backups\\\\n\\",\\n    \\"Review the displayed architecture, objective and estimated duration. The learning rate decays over the planned TOTAL steps. Resume must keep this exact config. Increasing steps after starting would change the learning-rate trajectory and is deliberately rejected.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-014\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"RUNTIME_CONFIG = RUN_DIR/\\\\\\"data_processed\\\\\\"/\\\\\\"runtime_config.yaml\\\\\\"\\\\n\\",\\n    \\"if RUNTIME_CONFIG.exists():\\\\n\\",\\n    \\"    train_config = yaml.safe_load(RUNTIME_CONFIG.read_text())\\\\n\\",\\n    \\"    print(\\\\\\"Existing runtime configuration retained for exact resume.\\\\\\")\\\\n\\",\\n    \\"    if train_config[\\\\\\"train\\\\\\"][\\\\\\"max_hours\\\\\\"] != TRAIN_HOURS + 0.5:\\\\n\\",\\n    \\"        raise RuntimeError(\\\\\\"Existing schedule uses a different time budget. Use a fresh RUN_DIR for the overnight experiment, or restore TRAIN_HOURS to match the existing schedule for exact resume.\\\\\\")\\\\n\\",\\n    \\"else:\\\\n\\",\\n    \\"    train_config = base_config.copy()\\\\n\\",\\n    \\"    train_config[\\\\\\"train\\\\\\"] = base_config[\\\\\\"train\\\\\\"].copy()\\\\n\\",\\n    \\"    train_config[\\\\\\"train\\\\\\"].update(max_steps=suggested_steps, decay_start=suggested_steps//2,\\\\n\\",\\n    \\"                                 max_hours=TRAIN_HOURS+0.5)\\\\n\\",\\n    \\"    RUNTIME_CONFIG.parent.mkdir(parents=True, exist_ok=True)\\\\n\\",\\n    \\"    RUNTIME_CONFIG.write_text(yaml.safe_dump(train_config))\\\\n\\",\\n    \\"print(yaml.safe_dump(train_config))\\\\n\\",\\n    \\"if BACKUP_DIR is None:\\\\n\\",\\n    \\"    raise RuntimeError(\\\\\\"Set BACKUP_DIR to persistent storage in cell 3, then rerun this cell.\\\\\\")\\\\n\\",\\n    \\"assert BACKUP_DIR != RUN_DIR and RUN_DIR not in BACKUP_DIR.parents, \\\\\\"Backup must be outside the run directory to avoid recursive copying.\\\\\\"\\\\n\\",\\n    \\"BACKUP_DIR.mkdir(parents=True, exist_ok=True)\\\\n\\",\\n    \\"probe = BACKUP_DIR/\\\\\\"task3_write_check.txt\\\\\\"\\\\n\\",\\n    \\"probe.write_text(\\\\\\"Persistent backup write check\\\\\\")\\\\n\\",\\n    \\"assert probe.read_text() == \\\\\\"Persistent backup write check\\\\\\"\\\\n\\",\\n    \\"print(\\\\\\"Backup directory is writable. Confirm it survives the lab session:\\\\\\", BACKUP_DIR)\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-015\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 8. Train four hours, evaluate, then resume\\\\n\\",\\n    \\"Set `START_TRAINING=True` after reviewing the benchmark and frozen configuration.\\\\n\\",\\n    \\"The default 12-hour budget is adjustable; it is not a claim about optimal convergence.\\\\n\\",\\n    \\"`BACKGROUND_TRAINING=True` launches a separate process with disk logs and checkpoints.\\\\n\\",\\n    \\"Do not also run foreground training for the same run directory.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"Before leaving, check the status cell, confirm an actual checkpoint and backup exist,\\\\n\\",\\n    \\"and ensure the lab machine stays awake and your session will not be terminated.\\\\n\\",\\n    \\"Closing the notebook tab is not a shutdown request for this background process, but\\\\n\\",\\n    \\"lab policies, machine restarts, or process cleanup can still terminate it.\\\\n\\",\\n    \\"Notebook interruption does **not** stop a detached job.\\\\n\\",\\n    \\"Run sections 9 onward only after training has stopped; do not use Run All.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"**First phase:** keep `TRAIN_HOURS=12.0` for the full schedule and launch with\\\\n\\",\\n    \\"`SESSION_HOURS=4.0`. It pauses after a complete optimizer step around four hours\\\\n\\",\\n    \\"and saves the entire training state. Wait for background status `finished`.\\\\n\\",\\n    \\"Run sections 9-12 to freeze and evaluate this checkpoint and make a fallback\\\\n\\",\\n    \\"submission CSV using the instructor evaluator. Preserve this snapshot and CSV.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"**Continuation:** if the model is healthy, change `SESSION_HOURS` in this launch\\\\n\\",\\n    \\"cell to `8.0` and rerun it. The latest checkpoint is resumed automatically with\\\\n\\",\\n    \\"the same models, optimizer state, replay pools, EMA, RNG and full learning-rate\\\\n\\",\\n    \\"schedule. Do not change `TRAIN_HOURS`, architecture, losses or the frozen YAML.\\\\n\\",\\n    \\"Evaluate the new checkpoint again afterwards; retain the first-phase candidate.\\\\n\\",\\n    \\"A four-hour checkpoint is recoverable, but its quality still needs evaluation.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-016\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"START_TRAINING = False  # set True after reviewing cells 1-7\\\\n\\",\\n    \\"BACKGROUND_TRAINING = True\\\\n\\",\\n    \\"SESSION_HOURS = 4.0  # first phase; change to 8.0 AFTER evaluating to continue\\\\n\\",\\n    \\"if START_TRAINING:\\\\n\\",\\n    \\"    assert BACKUP_DIR is not None, \\\\\\"Persistent backup is required.\\\\\\"\\\\n\\",\\n    \\"    command_args = [\\\\\\"--config\\\\\\", RUNTIME_CONFIG, \\\\\\"--data-root\\\\\\", DATA_ROOT,\\\\n\\",\\n    \\"                    \\\\\\"--splits\\\\\\", SPLITS, \\\\\\"--run-dir\\\\\\", RUN_DIR, \\\\\\"--device\\\\\\", \\\\\\"cuda\\\\\\",\\\\n\\",\\n    \\"                    \\\\\\"--backup-dir\\\\\\", BACKUP_DIR, \\\\\\"--code-version\\\\\\", CODE_VERSION,\\\\n\\",\\n    \\"                    \\\\\\"--session-hours\\\\\\", SESSION_HOURS]\\\\n\\",\\n    \\"    latest = RUN_DIR/\\\\\\"checkpoints\\\\\\"/\\\\\\"latest_checkpoint.pt\\\\\\"\\\\n\\",\\n    \\"    if latest.exists():\\\\n\\",\\n    \\"        command_args += [\\\\\\"--resume\\\\\\", latest]\\\\n\\",\\n    \\"    if BACKGROUND_TRAINING:\\\\n\\",\\n    \\"        launch(SRC/\\\\\\"background.py\\\\\\", \\\\\\"launch\\\\\\", \\\\\\"--run-dir\\\\\\", RUN_DIR, \\\\\\"--\\\\\\",\\\\n\\",\\n    \\"               sys.executable, \\\\\\"-u\\\\\\", str(SRC/\\\\\\"train.py\\\\\\"), *map(str, command_args))\\\\n\\",\\n    \\"    else:\\\\n\\",\\n    \\"        launch(SRC/\\\\\\"train.py\\\\\\", *command_args)\\\\n\\",\\n    \\"else:\\\\n\\",\\n    \\"    print(\\\\\\"Training not started. Set START_TRAINING=True when ready.\\\\\\")\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-017\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"# Rerun while training, and again when you return. This does not start a job.\\\\n\\",\\n    \\"launch(SRC/\\\\\\"background.py\\\\\\", \\\\\\"status\\\\\\", \\\\\\"--run-dir\\\\\\", RUN_DIR)\\\\n\\",\\n    \\"checkpoint = RUN_DIR/\\\\\\"checkpoints\\\\\\"/\\\\\\"latest_checkpoint.pt\\\\\\"\\\\n\\",\\n    \\"print(\\\\\\"Latest checkpoint:\\\\\\", checkpoint, \\\\\\"exists:\\\\\\", checkpoint.exists())\\\\n\\",\\n    \\"if checkpoint.exists():\\\\n\\",\\n    \\"    print(\\\\\\"Checkpoint modified:\\\\\\", time.ctime(checkpoint.stat().st_mtime))\\\\n\\"\\n   ],\\n   \\"id\\": \\"9797afbb\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 9. Freeze an evaluation checkpoint\\\\n\\",\\n    \\"Choose a periodic checkpoint or copy the latest completed checkpoint. Compare candidate checkpoints and raw/EMA weights on the same validation split. Never select the model solely on low cycle loss. Full FID/KID and LPIPS require evaluation-weight downloads on first use.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-018\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"CANDIDATE_SOURCE = RUN_DIR/\\\\\\"checkpoints\\\\\\"/\\\\\\"latest_checkpoint.pt\\\\\\"\\\\n\\",\\n    \\"WEIGHTS = \\\\\\"ema\\\\\\"  # compare with \\\\\\"raw\\\\\\" using another export directory\\\\n\\",\\n    \\"assert CANDIDATE_SOURCE.exists(), \\\\\\"Complete a training segment first.\\\\\\"\\\\n\\",\\n    \\"sys.path.insert(0, str(SRC))\\\\n\\",\\n    \\"from checkpointing import load_checkpoint, backup_file\\\\n\\",\\n    \\"from common import sha256\\\\n\\",\\n    \\"candidate = load_checkpoint(CANDIDATE_SOURCE)\\\\n\\",\\n    \\"step = candidate[\\\\\\"step\\\\\\"]\\\\n\\",\\n    \\"SNAPSHOT = RUN_DIR/\\\\\\"checkpoints\\\\\\"/\\\\\\"evaluation\\\\\\"/f\\\\\\"step_{step:08d}.pt\\\\\\"\\\\n\\",\\n    \\"if SNAPSHOT.exists():\\\\n\\",\\n    \\"    assert sha256(SNAPSHOT) == sha256(CANDIDATE_SOURCE), \\\\\\"Conflicting snapshot; preserve both with distinct names.\\\\\\"\\\\n\\",\\n    \\"else:\\\\n\\",\\n    \\"    backup_file(CANDIDATE_SOURCE, SNAPSHOT)\\\\n\\",\\n    \\"EXPORT_DIR = RUN_DIR/\\\\\\"outputs\\\\\\"/\\\\\\"evaluations\\\\\\"/f\\\\\\"step_{step:08d}_{WEIGHTS}\\\\\\"\\\\n\\",\\n    \\"if BACKUP_DIR is not None:\\\\n\\",\\n    \\"    backup_file(SNAPSHOT, BACKUP_DIR/\\\\\\"checkpoints\\\\\\"/\\\\\\"evaluation\\\\\\"/SNAPSHOT.name)\\\\n\\",\\n    \\"print(\\\\\\"Snapshot:\\\\\\", SNAPSHOT, \\\\\\"Export:\\\\\\", EXPORT_DIR)\\\\n\\",\\n    \\"del candidate\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-019\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"if not (EXPORT_DIR/\\\\\\"export_manifest.json\\\\\\").exists():\\\\n\\",\\n    \\"    launch(SRC/\\\\\\"infer.py\\\\\\", \\\\\\"--checkpoint\\\\\\", SNAPSHOT, \\\\\\"--data-root\\\\\\", DATA_ROOT,\\\\n\\",\\n    \\"           \\\\\\"--splits\\\\\\", SPLITS, \\\\\\"--output\\\\\\", EXPORT_DIR, \\\\\\"--weights\\\\\\", WEIGHTS,\\\\n\\",\\n    \\"           \\\\\\"--split\\\\\\", \\\\\\"val\\\\\\", \\\\\\"--device\\\\\\", \\\\\\"cuda\\\\\\")\\\\n\\",\\n    \\"else:\\\\n\\",\\n    \\"    print(\\\\\\"Export exists; preserve it. Choose another directory for another candidate.\\\\\\")\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-020\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"launch(MEMBER/\\\\\\"evaluate_local.py\\\\\\", \\\\\\"--export-dir\\\\\\", EXPORT_DIR, \\\\\\"--run-dir\\\\\\", RUN_DIR,\\\\n\\",\\n    \\"       \\\\\\"--device\\\\\\", \\\\\\"cuda\\\\\\", \\\\\\"--batch-size\\\\\\", 8)\\\\n\\",\\n    \\"launch(SRC/\\\\\\"report.py\\\\\\", \\\\\\"--run-dir\\\\\\", RUN_DIR, \\\\\\"--export-dir\\\\\\", EXPORT_DIR)\\\\n\\",\\n    \\"print((EXPORT_DIR/\\\\\\"full_metrics_report.csv\\\\\\").read_text())\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-021\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 10. Inspect and select\\\\n\\",\\n    \\"Review BOTH directions, sample counts, KID uncertainty, semantic preservation and failure grids. The local FID is not guaranteed to match Kaggle. Repeat cells 9 and evaluation for raw weights or another archived checkpoint if time allows.\\\\n\\",\\n    \\"Only after comparing candidates set `SELECT_THIS=True` and write a reason. Selection copies the entire fixed export and saves `best_model.pt`; it never chooses only attractive images.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-022\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"from IPython.display import display, Image as NotebookImage\\\\n\\",\\n    \\"preview_files = sorted((RUN_DIR/\\\\\\"outputs\\\\\\"/\\\\\\"plots\\\\\\").glob(\\\\\\"preview_*.png\\\\\\"))\\\\n\\",\\n    \\"if preview_files:\\\\n\\",\\n    \\"    display(NotebookImage(filename=str(preview_files[-1]), width=768))\\\\n\\",\\n    \\"for filename in [\\\\\\"adversarial.png\\\\\\", \\\\\\"cycle_identity.png\\\\\\", \\\\\\"gradients.png\\\\\\", \\\\\\"learning_rate.png\\\\\\"]:\\\\n\\",\\n    \\"    path = RUN_DIR/\\\\\\"outputs\\\\\\"/\\\\\\"plots\\\\\\"/filename\\\\n\\",\\n    \\"    if path.exists(): display(NotebookImage(filename=str(path)))\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-023\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"SELECT_THIS = False\\\\n\\",\\n    \\"SELECTION_REASON = \\\\\\"\\\\\\"  # actual validation comparison and visual findings\\\\n\\",\\n    \\"if SELECT_THIS:\\\\n\\",\\n    \\"    assert SELECTION_REASON.strip(), \\\\\\"Document why this candidate was selected.\\\\\\"\\\\n\\",\\n    \\"    launch(SRC/\\\\\\"select_checkpoint.py\\\\\\", \\\\\\"--checkpoint\\\\\\", SNAPSHOT, \\\\\\"--export-dir\\\\\\", EXPORT_DIR,\\\\n\\",\\n    \\"           \\\\\\"--run-dir\\\\\\", RUN_DIR, \\\\\\"--reason\\\\\\", SELECTION_REASON)\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-024\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 11. Blinded human audit\\\\n\\",\\n    \\"Prepare only after choosing the model. Both raters receive the same packet with 30 fixed examples (15 per direction). Keep `private_key.json` private from raters. Each independently fills `rater_1.csv` or `rater_2.csv`. Higher artifact score means fewer artifacts.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-025\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"AUDIT_DIR = RUN_DIR/\\\\\\"outputs\\\\\\"/\\\\\\"human_audit\\\\\\"\\\\n\\",\\n    \\"if (RUN_DIR/\\\\\\"selection.json\\\\\\").exists() and not AUDIT_DIR.exists():\\\\n\\",\\n    \\"    selection = json.loads((RUN_DIR/\\\\\\"selection.json\\\\\\").read_text())\\\\n\\",\\n    \\"    assert selection[\\\\\\"checkpoint_sha256\\\\\\"] == sha256(SNAPSHOT), \\\\\\"Audit must use the selected checkpoint.\\\\\\"\\\\n\\",\\n    \\"    launch(SRC/\\\\\\"audit.py\\\\\\", \\\\\\"prepare\\\\\\", \\\\\\"--export-dir\\\\\\", EXPORT_DIR, \\\\\\"--output\\\\\\", AUDIT_DIR)\\\\n\\",\\n    \\"else:\\\\n\\",\\n    \\"    print(\\\\\\"Select a checkpoint first, or reuse the already fixed audit packet.\\\\\\")\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-026\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"RATINGS_COMPLETE = False\\\\n\\",\\n    \\"if RATINGS_COMPLETE:\\\\n\\",\\n    \\"    launch(SRC/\\\\\\"audit.py\\\\\\", \\\\\\"summarize\\\\\\", \\\\\\"--output\\\\\\", AUDIT_DIR)\\\\n\\",\\n    \\"    launch(SRC/\\\\\\"report.py\\\\\\", \\\\\\"--run-dir\\\\\\", RUN_DIR, \\\\\\"--export-dir\\\\\\", EXPORT_DIR)\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-027\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 12. Official Kaggle evaluator\\\\n\\",\\n    \\"The instructor evaluator is included as `src/evaluate_official.py`. It uses torchvision Inception (different from local diagnostic FID), the first 300 sorted files per folder, matching counts, and index-wise cosine distance. It writes `ID,FID,MiFID`. Do not rearrange or hand-pick outputs to improve the score. Keep fixed held-out metrics separately for the report.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"After freezing and evaluating a candidate on validation, create a separate official\\\\n\\",\\n    \\"export. Use this only for the supplied first-300 scoring protocol and the approved\\\\n\\",\\n    \\"development source set; follow any separate instructor input-set requirements.\\\\n\\",\\n    \\"The deterministic prefix preserves the first 300 outputs of a full all-input export.\\\\n\\",\\n    \\"Full local validation still uses all validation samples and cycle images.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"Run in a new cell after training has stopped:\\\\n\\",\\n    \\"```python\\\\n\\",\\n    \\"OFFICIAL_DIR = RUN_DIR/\\\\\\"outputs\\\\\\"/\\\\\\"official\\\\\\"/f\\\\\\"{EXPORT_DIR.name}_first300\\\\\\"\\\\n\\",\\n    \\"if not (OFFICIAL_DIR/\\\\\\"export_manifest.json\\\\\\").exists():\\\\n\\",\\n    \\"    launch(SRC/\\\\\\"infer.py\\\\\\", \\\\\\"--checkpoint\\\\\\", SNAPSHOT, \\\\\\"--data-root\\\\\\", DATA_ROOT,\\\\n\\",\\n    \\"           \\\\\\"--splits\\\\\\", SPLITS, \\\\\\"--output\\\\\\", OFFICIAL_DIR, \\\\\\"--weights\\\\\\", WEIGHTS,\\\\n\\",\\n    \\"           \\\\\\"--split\\\\\\", \\\\\\"all\\\\\\", \\\\\\"--max-per-domain\\\\\\", 300, \\\\\\"--predictions-only\\\\\\",\\\\n\\",\\n    \\"           \\\\\\"--device\\\\\\", \\\\\\"cuda\\\\\\")\\\\n\\",\\n    \\"OFFICIAL_CSV = RUN_DIR/f\\\\\\"submission_{EXPORT_DIR.name}.csv\\\\\\"\\\\n\\",\\n    \\"if not OFFICIAL_CSV.exists():\\\\n\\",\\n    \\"    launch(SRC/\\\\\\"evaluate_official.py\\\\\\", \\\\\\"--data-root\\\\\\", DATA_ROOT,\\\\n\\",\\n    \\"           \\\\\\"--export-dir\\\\\\", OFFICIAL_DIR, \\\\\\"--output\\\\\\", OFFICIAL_CSV, \\\\\\"--batch-size\\\\\\", 16)\\\\n\\",\\n    \\"print(OFFICIAL_CSV.read_text())\\\\n\\",\\n    \\"```\\\\n\\",\\n    \\"Each checkpoint gets a distinct CSV so the four-hour fallback stays intact.\\\\n\\",\\n    \\"Copy your chosen candidate CSV to `submission.csv` for the final repository.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"The reference folders follow the instructor script and include training images; this official-protocol score is not held-out generalization evidence. Confirm any separate competition source-image requirements. No automatic upload is performed.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"## 13. Collect evidence and back up the full run\\\\n\\",\\n    \\"Run after training stops. Save the executed notebook separately too.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-028\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"execution_count\\": null,\\n   \\"metadata\\": {},\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"launch(SRC/\\\\\\"collect_evidence.py\\\\\\", \\\\\\"--run-dir\\\\\\", RUN_DIR, \\\\\\"--repo-root\\\\\\", PROJECT_ROOT, \\\\\\"--run-id\\\\\\", RUN_ID)\\\\n\\",\\n    \\"if BACKUP_DIR is not None:\\\\n\\",\\n    \\"    # Explicit include list avoids recursively copying the backup into itself.\\\\n\\",\\n    \\"    for name in [\\\\\\"checkpoints\\\\\\", \\\\\\"outputs\\\\\\", \\\\\\"reproducibility\\\\\\", \\\\\\"data_processed\\\\\\"]:\\\\n\\",\\n    \\"        source = RUN_DIR/name\\\\n\\",\\n    \\"        if source.exists(): shutil.copytree(source, BACKUP_DIR/name, dirs_exist_ok=True)\\\\n\\",\\n    \\"    for name in [\\\\\\"training_summary.json\\\\\\", \\\\\\"selection.json\\\\\\", \\\\\\"full_metrics_report.csv\\\\\\",\\\\n\\",\\n    \\"                 \\\\\\"metrics_report.csv\\\\\\", \\\\\\"results_draft.md\\\\\\", \\\\\\"submission.csv\\\\\\", \\\\\\"submission_provenance.json\\\\\\"]:\\\\n\\",\\n    \\"        source = RUN_DIR/name\\\\n\\",\\n    \\"        if source.exists(): shutil.copy2(source, BACKUP_DIR/name)\\\\n\\",\\n    \\"    latest = RUN_DIR/\\\\\\"checkpoints\\\\\\"/\\\\\\"latest_checkpoint.pt\\\\\\"\\\\n\\",\\n    \\"    if latest.exists():\\\\n\\",\\n    \\"        assert sha256(latest) == sha256(BACKUP_DIR/\\\\\\"checkpoints\\\\\\"/latest.name)\\\\n\\",\\n    \\"    print(\\\\\\"Backup complete; latest checkpoint checksum verified. Save the executed notebook separately.\\\\\\")\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-029\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 14. Laptop commits and report\\\\n\\",\\n    \\"Bring back the executed notebook, source changes (if any), raw logs, manifests, metrics, plots and analysis. Keep large binaries in persistent storage with checkpoint hashes and retrieval instructions. Commit code before runs and results after runs; do not manufacture history.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"Complete `results.md` and `failure_analysis.md` from real evidence. The generated `results_draft.md` is a scaffold, not finished scientific analysis. Coordinate evaluation settings with your teammate. Explain architecture, objectives, metric limitations and observed failures in the viva.\\\\n\\"\\n   ],\\n   \\"id\\": \\"task3-030\\"\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"id\\": \\"quick-score-section\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 15. Quick official score for the chosen frozen snapshot\\\\n\\",\\n    \\"After training stops, rerun Section 9\'s **first cell** to update `SNAPSHOT`. This cell uses that snapshot, not the active checkpoint. It reuses matching official results or generates only 300 JPGs per direction and computes the instructor FID/MiFID and expected displayed Kaggle score. It does not replace the full local metrics or human audit.\\\\n\\"\\n   ]\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"id\\": \\"quick-score-launch\\",\\n   \\"metadata\\": {},\\n   \\"execution_count\\": null,\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"import runpy\\\\n\\",\\n    \\"quick_result = runpy.run_path(str(SRC/\\\\\\"quick_kaggle_score.py\\\\\\"),\\\\n\\",\\n    \\"    init_globals={\\\\\\"RUN_DIR\\\\\\": RUN_DIR, \\\\\\"DATA_ROOT\\\\\\": DATA_ROOT, \\\\\\"SNAPSHOT\\\\\\": SNAPSHOT, \\\\\\"WEIGHTS\\\\\\": WEIGHTS})\\\\n\\",\\n    \\"OFFICIAL_CSV = Path(quick_result[\\\\\\"OFFICIAL_CSV\\\\\\"])\\\\n\\"\\n   ]\\n  },\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"id\\": \\"completed-package-section\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"## 16. Downloadable completed-snapshot package\\\\n\\",\\n    \\"Run after the selected training segment has finished and Section 9\'s first cell has frozen it. Set `PACKAGE_STEP=None` to select the highest completed step, or set it to the printed step to archive that exact snapshot. The ZIP includes completed matching training-history segments, exact score CSVs, generated images, source notebooks and the frozen checkpoint. The manifest lists missing rubric evidence. The dataset is excluded; restore the same original dataset separately. No upload or email occurs.\\\\n\\",\\n    \\"\\\\nDownload the resulting file from `task3_gan/transfer/`. An executed notebook outside the member folder can be included by setting `EXECUTED_NOTEBOOK` to its saved path. The starter file in source is a clean runnable notebook, not a substitute for the executed notebook.\\\\n\\"\\n   ]\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"id\\": \\"completed-package-launch\\",\\n   \\"metadata\\": {},\\n   \\"execution_count\\": null,\\n   \\"outputs\\": [],\\n   \\"source\\": [\\n    \\"PACKAGE_STEP = None  # default: latest completed phase; or enter the exact snapshot step\\\\n\\",\\n    \\"EXECUTED_NOTEBOOK = None  # optional Path to the SAVED executed notebook\\\\n\\",\\n    \\"package_args = [\\\\\\"--run-dir\\\\\\", RUN_DIR, \\\\\\"--full\\\\\\", \\\\\\"--weights\\\\\\", WEIGHTS]\\\\n\\",\\n    \\"if PACKAGE_STEP is not None:\\\\n\\",\\n    \\"    package_args += [\\\\\\"--step\\\\\\", PACKAGE_STEP]\\\\n\\",\\n    \\"if EXECUTED_NOTEBOOK is not None:\\\\n\\",\\n    \\"    package_args += [\\\\\\"--notebook\\\\\\", EXECUTED_NOTEBOOK]\\\\n\\",\\n    \\"launch(SRC/\\\\\\"package_completed_run.py\\\\\\", *package_args)\\\\n\\"\\n   ]\\n  }\\n ]\\n}\\n", "task3_gan/Pramod_Dindukurthi/src/Quick_Kaggle_Score.ipynb": "{\\n \\"nbformat\\": 4,\\n \\"nbformat_minor\\": 5,\\n \\"metadata\\": {\\n  \\"kernelspec\\": {\\n   \\"display_name\\": \\"Python 3\\",\\n   \\"language\\": \\"python\\",\\n   \\"name\\": \\"python3\\"\\n  },\\n  \\"language_info\\": {\\n   \\"name\\": \\"python\\"\\n  }\\n },\\n \\"cells\\": [\\n  {\\n   \\"cell_type\\": \\"markdown\\",\\n   \\"id\\": \\"quick-score-instructions\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"# Quick FID, MiFID and expected Kaggle score\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"Upload this notebook to the GPU lab machine and run its one code cell, or copy the code cell into your existing notebook. It uses the existing Task 3 source files; no packages or model architecture are changed.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"**Current checkpoint:** defaults to the highest-numbered frozen checkpoint in the selected run directory. A matching prior official result is reused immediately. In your existing notebook, `SNAPSHOT` takes precedence.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"**New checkpoint:** after the training segment stops, run only Section 9\'s snapshot/configuration cell. Then run this cell. Alternatively set `CHECKPOINT` explicitly to that frozen `.pt` file. Do not read the live `latest_checkpoint.pt`.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"For new checkpoints this generates only 300 predictions per direction, saves the actual 256 x 256 RGB JPGs and verifies their hashes. It preserves the same source order, JPG encoding and instructor evaluation functions used for your baseline. It skips input/cycle exports and the full report workflow. Official scoring includes training-reference images and does not replace held-out evaluation, all assignment metrics or the human audit.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"When GPU evaluation overlaps background training, the jobs share GPU time. The result describes the selected frozen checkpoint, not the model currently training. FID covariance calculations still take time on CPU.\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"The expected displayed score is `-(FID + MiFID) / 2`, confirmed by the supplied competition description and the user\'s accepted -53.0607 baseline. Actual Kaggle submission remains a separate step.\\\\n\\"\\n   ]\\n  },\\n  {\\n   \\"cell_type\\": \\"code\\",\\n   \\"id\\": \\"quick-score-run\\",\\n   \\"metadata\\": {},\\n   \\"source\\": [\\n    \\"from pathlib import Path\\\\n\\",\\n    \\"import csv, json, sys\\\\n\\",\\n    \\"import torch\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"# Uses a FROZEN checkpoint. For tomorrow\'s model, first run only Section 9\'s\\\\n\\",\\n    \\"# snapshot cell, then run this cell; no full image-export sections are needed.\\\\n\\",\\n    \\"default_run = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / \\\\\\"src/train.py\\\\\\").is_file()),\\\\n\\",\\n    \\"                   Path(\\\\\\"/app/task3_gan/Pramod_Dindukurthi\\\\\\"))\\\\n\\",\\n    \\"RUN_DIR = Path(globals().get(\\\\\\"RUN_DIR\\\\\\", default_run))\\\\n\\",\\n    \\"DATA_ROOT = Path(globals().get(\\\\\\"DATA_ROOT\\\\\\", RUN_DIR.parent / \\\\\\"data\\\\\\"))\\\\n\\",\\n    \\"frozen = sorted(p for p in (RUN_DIR / \\\\\\"checkpoints/evaluation\\\\\\").glob(\\\\\\"step_*.pt\\\\\\")\\\\n\\",\\n    \\"                if p.stem.removeprefix(\\\\\\"step_\\\\\\").isdigit())\\\\n\\",\\n    \\"default_checkpoint = frozen[-1] if frozen else RUN_DIR / \\\\\\"checkpoints/evaluation/step_00112805.pt\\\\\\"\\\\n\\",\\n    \\"CHECKPOINT = Path(globals().get(\\\\\\"SNAPSHOT\\\\\\", default_checkpoint))\\\\n\\",\\n    \\"WEIGHTS = globals().get(\\\\\\"WEIGHTS\\\\\\", \\\\\\"ema\\\\\\")  # set \\\\\\"raw\\\\\\" for a separately recorded comparison\\\\n\\",\\n    \\"assert WEIGHTS in {\\\\\\"ema\\\\\\", \\\\\\"raw\\\\\\"}\\\\n\\",\\n    \\"N_EVAL = 300\\\\n\\",\\n    \\"BATCH_SIZE = 16\\\\n\\",\\n    \\"DEVICE = \\\\\\"cuda\\\\\\" if torch.cuda.is_available() else \\\\\\"cpu\\\\\\"\\\\n\\",\\n    \\"SRC = RUN_DIR / \\\\\\"src\\\\\\"\\\\n\\",\\n    \\"if str(SRC) not in sys.path:\\\\n\\",\\n    \\"    sys.path.insert(0, str(SRC))\\\\n\\",\\n    \\"from checkpointing import load_checkpoint\\\\n\\",\\n    \\"from common import sha256, json_write, utc_now\\\\n\\",\\n    \\"from data import read_splits, rgb, transform, to_image\\\\n\\",\\n    \\"from models import Generator\\\\n\\",\\n    \\"import evaluate_official as official\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"if not CHECKPOINT.is_file():\\\\n\\",\\n    \\"    raise FileNotFoundError(f\\\\\\"Freeze the checkpoint using Section 9\'s first cell: {CHECKPOINT}\\\\\\")\\\\n\\",\\n    \\"if CHECKPOINT.name == \\\\\\"latest_checkpoint.pt\\\\\\":\\\\n\\",\\n    \\"    raise ValueError(\\\\\\"Use a frozen evaluation checkpoint, not the actively replaced latest_checkpoint.pt.\\\\\\")\\\\n\\",\\n    \\"SPLITS = RUN_DIR / \\\\\\"data_processed/splits.json\\\\\\"\\\\n\\",\\n    \\"print(\\\\\\"Scoring frozen checkpoint:\\\\\\", CHECKPOINT, \\\\\\"| weights:\\\\\\", WEIGHTS, flush=True)\\\\n\\",\\n    \\"checkpoint_hash = sha256(CHECKPOINT)\\\\n\\",\\n    \\"state = load_checkpoint(CHECKPOINT)\\\\n\\",\\n    \\"step = state[\\\\\\"step\\\\\\"]\\\\n\\",\\n    \\"assert state[\\\\\\"config\\\\\\"][\\\\\\"image_size\\\\\\"] == 256, \\\\\\"Official predictions must be 256 x 256.\\\\\\"\\\\n\\",\\n    \\"assert sha256(SPLITS) == state[\\\\\\"split_sha256\\\\\\"], \\\\\\"Training split manifest differs.\\\\\\"\\\\n\\",\\n    \\"tag = f\\\\\\"step_{step:08d}_{WEIGHTS}\\\\\\"\\\\n\\",\\n    \\"OFFICIAL_DIR = RUN_DIR / \\\\\\"outputs/official\\\\\\" / f\\\\\\"{tag}_quick_{checkpoint_hash[:12]}\\\\\\"\\\\n\\",\\n    \\"OFFICIAL_CSV = RUN_DIR / f\\\\\\"submission_{tag}_quick_{checkpoint_hash[:12]}.csv\\\\\\"\\\\n\\",\\n    \\"SCORE_JSON = OFFICIAL_CSV.with_suffix(\\\\\\".metrics.json\\\\\\")\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"# Reuse a prior score ONLY if its export manifest matches this checkpoint/weights,\\\\n\\",\\n    \\"# the evaluator source is identical, and both directions used 300 images.\\\\n\\",\\n    \\"prior = []\\\\n\\",\\n    \\"for candidate in RUN_DIR.glob(f\\\\\\"submission_{tag}*.metrics.json\\\\\\"):\\\\n\\",\\n    \\"    payload = json.loads(candidate.read_text())\\\\n\\",\\n    \\"    if payload.get(\\\\\\"evaluation_script_sha256\\\\\\") != sha256(Path(official.__file__)):\\\\n\\",\\n    \\"        continue\\\\n\\",\\n    \\"    if any(payload.get(\\\\\\"counts\\\\\\", {}).get(d, {}).get(\\\\\\"matched\\\\\\") != N_EVAL for d in (\\\\\\"A2B\\\\\\", \\\\\\"B2A\\\\\\")):\\\\n\\",\\n    \\"        continue\\\\n\\",\\n    \\"    for manifest in (RUN_DIR / \\\\\\"outputs/official\\\\\\").glob(\\\\\\"*/export_manifest.json\\\\\\"):\\\\n\\",\\n    \\"        if sha256(manifest) != payload.get(\\\\\\"export_manifest_sha256\\\\\\"):\\\\n\\",\\n    \\"            continue\\\\n\\",\\n    \\"        metadata = json.loads(manifest.read_text())\\\\n\\",\\n    \\"        if metadata.get(\\\\\\"checkpoint_sha256\\\\\\") == checkpoint_hash and metadata.get(\\\\\\"weights\\\\\\") == WEIGHTS:\\\\n\\",\\n    \\"            csv_path = candidate.with_name(candidate.name.removesuffix(\\\\\\".metrics.json\\\\\\") + \\\\\\".csv\\\\\\")\\\\n\\",\\n    \\"            if csv_path.is_file():\\\\n\\",\\n    \\"                prior.append((csv_path, payload))\\\\n\\",\\n    \\"            break\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"if prior:\\\\n\\",\\n    \\"    OFFICIAL_CSV, result = prior[0]\\\\n\\",\\n    \\"    row = result[\\\\\\"submission\\\\\\"]\\\\n\\",\\n    \\"    del state\\\\n\\",\\n    \\"    print(\\\\\\"Reusing previously computed, matching official metrics.\\\\\\")\\\\n\\",\\n    \\"else:\\\\n\\",\\n    \\"    manifest_path = OFFICIAL_DIR / \\\\\\"export_manifest.json\\\\\\"\\\\n\\",\\n    \\"    if not manifest_path.exists():\\\\n\\",\\n    \\"        if OFFICIAL_DIR.exists():\\\\n\\",\\n    \\"            raise RuntimeError(f\\\\\\"Incomplete earlier export exists; preserve it and choose a new output name: {OFFICIAL_DIR}\\\\\\")\\\\n\\",\\n    \\"        splits = read_splits(SPLITS)\\\\n\\",\\n    \\"        records = []\\\\n\\",\\n    \\"        for domain, direction, generator_key in ((\\\\\\"A\\\\\\", \\\\\\"A2B\\\\\\", \\\\\\"G_A2B\\\\\\"), (\\\\\\"B\\\\\\", \\\\\\"B2A\\\\\\", \\\\\\"G_B2A\\\\\\")):\\\\n\\",\\n    \\"            sources = (splits[\\\\\\"domains\\\\\\"][domain][\\\\\\"train\\\\\\"] + splits[\\\\\\"domains\\\\\\"][domain][\\\\\\"val\\\\\\"])[:N_EVAL]\\\\n\\",\\n    \\"            assert len(sources) == N_EVAL, \\\\\\"Need 300 permitted source images per direction.\\\\\\"\\\\n\\",\\n    \\"            folder = OFFICIAL_DIR / f\\\\\\"pred_{direction}\\\\\\"\\\\n\\",\\n    \\"            folder.mkdir(parents=True)\\\\n\\",\\n    \\"            generator = Generator(**state[\\\\\\"config\\\\\\"][\\\\\\"model\\\\\\"]).to(DEVICE).eval()\\\\n\\",\\n    \\"            generator.load_state_dict(state[\\\\\\"ema\\\\\\" if WEIGHTS == \\\\\\"ema\\\\\\" else \\\\\\"models\\\\\\"][generator_key])\\\\n\\",\\n    \\"            with torch.inference_mode():\\\\n\\",\\n    \\"                for i, rel in enumerate(sources):\\\\n\\",\\n    \\"                    assert sha256(DATA_ROOT / rel) == splits[\\\\\\"domains\\\\\\"][domain][\\\\\\"sha256\\\\\\"][rel], f\\\\\\"Source changed: {rel}\\\\\\"\\\\n\\",\\n    \\"                    prediction = generator(transform(rgb(DATA_ROOT / rel), 256).unsqueeze(0).to(DEVICE))\\\\n\\",\\n    \\"                    path = folder / f\\\\\\"{i:06d}.jpg\\\\\\"\\\\n\\",\\n    \\"                    to_image(prediction[0]).save(path, quality=95, subsampling=0)\\\\n\\",\\n    \\"                    records.append({\\\\\\"domain\\\\\\": domain, \\\\\\"direction\\\\\\": direction, \\\\\\"source\\\\\\": rel,\\\\n\\",\\n    \\"                                    \\\\\\"prediction\\\\\\": path.relative_to(OFFICIAL_DIR).as_posix(),\\\\n\\",\\n    \\"                                    \\\\\\"prediction_sha256\\\\\\": sha256(path)})\\\\n\\",\\n    \\"                    if (i + 1) % 100 == 0:\\\\n\\",\\n    \\"                        print(f\\\\\\"Generated {direction}: {i + 1}/{N_EVAL}\\\\\\", flush=True)\\\\n\\",\\n    \\"            del generator, prediction\\\\n\\",\\n    \\"        json_write(manifest_path, {\\\\\\"created_utc\\\\\\": utc_now(), \\\\\\"checkpoint_sha256\\\\\\": checkpoint_hash,\\\\n\\",\\n    \\"            \\\\\\"step\\\\\\": step, \\\\\\"weights\\\\\\": WEIGHTS, \\\\\\"split\\\\\\": \\\\\\"all\\\\\\", \\\\\\"split_sha256\\\\\\": sha256(SPLITS),\\\\n\\",\\n    \\"            \\\\\\"image_size\\\\\\": 256, \\\\\\"jpeg_quality\\\\\\": 95, \\\\\\"jpeg_subsampling\\\\\\": 0,\\\\n\\",\\n    \\"            \\\\\\"predictions_only\\\\\\": True, \\\\\\"max_per_domain\\\\\\": N_EVAL,\\\\n\\",\\n    \\"            \\\\\\"input_order\\\\\\": \\\\\\"fixed split manifest order; train followed by val; first 300\\\\\\",\\\\n\\",\\n    \\"            \\\\\\"records\\\\\\": records})\\\\n\\",\\n    \\"    meta = json.loads(manifest_path.read_text())\\\\n\\",\\n    \\"    assert meta[\\\\\\"checkpoint_sha256\\\\\\"] == checkpoint_hash and meta[\\\\\\"weights\\\\\\"] == WEIGHTS\\\\n\\",\\n    \\"    for record in meta[\\\\\\"records\\\\\\"]:\\\\n\\",\\n    \\"        assert sha256(OFFICIAL_DIR / record[\\\\\\"prediction\\\\\\"]) == record[\\\\\\"prediction_sha256\\\\\\"], \\\\\\"Prediction changed.\\\\\\"\\\\n\\",\\n    \\"    del state\\\\n\\",\\n    \\"    if DEVICE == \\\\\\"cuda\\\\\\":\\\\n\\",\\n    \\"        torch.cuda.empty_cache()\\\\n\\",\\n    \\"    if OFFICIAL_CSV.exists() or SCORE_JSON.exists():\\\\n\\",\\n    \\"        raise FileExistsError(\\\\\\"Preserve the earlier score files; use a different output name.\\\\\\")\\\\n\\",\\n    \\"    official.device = torch.device(DEVICE)\\\\n\\",\\n    \\"    per_direction, counts = {}, {}\\\\n\\",\\n    \\"    for direction, target in ((\\\\\\"A2B\\\\\\", \\\\\\"photo_jpg\\\\\\"), (\\\\\\"B2A\\\\\\", \\\\\\"monet_jpg\\\\\\")):\\\\n\\",\\n    \\"        real = official.take_n(official.list_images(str(DATA_ROOT / target)), N_EVAL)\\\\n\\",\\n    \\"        generated = official.take_n(official.list_images(str(OFFICIAL_DIR / f\\\\\\"pred_{direction}\\\\\\")), N_EVAL)\\\\n\\",\\n    \\"        assert len(real) == len(generated) == N_EVAL\\\\n\\",\\n    \\"        fid, mifid = official.calculate_fid_mifid(real, generated, batch_size=BATCH_SIZE)\\\\n\\",\\n    \\"        assert torch.isfinite(torch.tensor([fid, mifid], dtype=torch.float64)).all(), \\\\\\"Non-finite metric.\\\\\\"\\\\n\\",\\n    \\"        per_direction[direction] = {\\\\\\"FID\\\\\\": fid, \\\\\\"MiFID\\\\\\": mifid}\\\\n\\",\\n    \\"        counts[direction] = {\\\\\\"real_capped\\\\\\": len(real), \\\\\\"generated_capped\\\\\\": len(generated), \\\\\\"matched\\\\\\": N_EVAL}\\\\n\\",\\n    \\"    row = {\\\\\\"ID\\\\\\": 1,\\\\n\\",\\n    \\"           \\\\\\"FID\\\\\\": sum(x[\\\\\\"FID\\\\\\"] for x in per_direction.values()) / 2,\\\\n\\",\\n    \\"           \\\\\\"MiFID\\\\\\": sum(x[\\\\\\"MiFID\\\\\\"] for x in per_direction.values()) / 2}\\\\n\\",\\n    \\"    with OFFICIAL_CSV.open(\\\\\\"x\\\\\\", newline=\\\\\\"\\\\\\") as f:\\\\n\\",\\n    \\"        writer = csv.DictWriter(f, fieldnames=[\\\\\\"ID\\\\\\", \\\\\\"FID\\\\\\", \\\\\\"MiFID\\\\\\"])\\\\n\\",\\n    \\"        writer.writeheader()\\\\n\\",\\n    \\"        writer.writerow(row)\\\\n\\",\\n    \\"    result = {\\\\\\"per_direction\\\\\\": per_direction, \\\\\\"submission\\\\\\": row, \\\\\\"counts\\\\\\": counts, \\\\\\"n_eval\\\\\\": N_EVAL,\\\\n\\",\\n    \\"              \\\\\\"evaluation_script_sha256\\\\\\": sha256(Path(official.__file__)),\\\\n\\",\\n    \\"              \\\\\\"export_manifest_sha256\\\\\\": sha256(manifest_path),\\\\n\\",\\n    \\"              \\\\\\"protocol\\\\\\": \\\\\\"instructor torchvision Inception IMAGENET1K_V1; sorted first-300; index cosine\\\\\\"}\\\\n\\",\\n    \\"    json_write(SCORE_JSON, result)\\\\n\\",\\n    \\"\\\\n\\",\\n    \\"estimated_score = -(row[\\\\\\"FID\\\\\\"] + row[\\\\\\"MiFID\\\\\\"]) / 2\\\\n\\",\\n    \\"print(json.dumps(result[\\\\\\"per_direction\\\\\\"], indent=2))\\\\n\\",\\n    \\"print(f\\\\\\"\\\\\\\\nCheckpoint step: {step:,} | Weights: {WEIGHTS}\\\\\\")\\\\n\\",\\n    \\"print(f\\\\\\"FID: {row[\'FID\']:.6f} | MiFID: {row[\'MiFID\']:.6f}\\\\\\")\\\\n\\",\\n    \\"print(f\\\\\\"Expected Kaggle displayed score: {estimated_score:.6f} (closer to zero is better)\\\\\\")\\\\n\\",\\n    \\"print(f\\\\\\"Submission CSV: {OFFICIAL_CSV}\\\\\\")\\\\n\\",\\n    \\"print(\\\\\\"No Kaggle upload was performed. This cell evaluates the chosen frozen checkpoint only.\\\\\\")\\\\n\\"\\n   ],\\n   \\"execution_count\\": null,\\n   \\"outputs\\": []\\n  }\\n ]\\n}\\n", "task3_gan/Pramod_Dindukurthi/configs/baseline.yaml": "name: baseline\\nseed: 42\\nimage_size: 256\\nload_size: 286\\nmodel:\\n  channels: 64\\n  blocks: 9\\n  upsample: transpose\\ntrain:\\n  batch_size: 1\\n  num_workers: 2\\n  max_steps: 20000\\n  decay_start: 10000\\n  lr: 0.0002\\n  beta1: 0.5\\n  beta2: 0.999\\n  cycle_weight: 10.0\\n  identity_weight: 5.0\\n  pool_size: 50\\n  ema_decay: 0.999\\n  amp: false\\n  deterministic: true\\n  log_every: 25\\n  save_every: 500\\n  save_minutes: 5\\n  preview_every: 500\\n  keep_every: 2000\\n  max_hours: 6.0\\n", "task3_gan/Pramod_Dindukurthi/configs/resizeconv.yaml": "name: resizeconv\\nseed: 42\\nimage_size: 256\\nload_size: 286\\nmodel:\\n  channels: 64\\n  blocks: 9\\n  upsample: resize\\ntrain:\\n  batch_size: 1\\n  num_workers: 2\\n  max_steps: 20000\\n  decay_start: 10000\\n  lr: 0.0002\\n  beta1: 0.5\\n  beta2: 0.999\\n  cycle_weight: 10.0\\n  identity_weight: 5.0\\n  pool_size: 50\\n  ema_decay: 0.999\\n  amp: false\\n  deterministic: true\\n  log_every: 25\\n  save_every: 500\\n  save_minutes: 5\\n  preview_every: 500\\n  keep_every: 2000\\n  max_hours: 6.0\\n", "task3_gan/Pramod_Dindukurthi/reference/Part3_Evaluation_Script.ipynb": "{\\"nbformat\\":4,\\"nbformat_minor\\":0,\\"metadata\\":{\\"colab\\":{\\"provenance\\":[],\\"authorship_tag\\":\\"ABX9TyNLQJa/Plf1LCtmi3z3Fhnu\\"},\\"kernelspec\\":{\\"name\\":\\"python3\\",\\"display_name\\":\\"Python 3\\"},\\"language_info\\":{\\"name\\":\\"python\\"}},\\"cells\\":[{\\"cell_type\\":\\"markdown\\",\\"source\\":[\\"\\\\n\\",\\"# **CycleGAN Evaluation Script**\\\\n\\",\\"\\\\n\\",\\"### Folder structure expected:\\\\n\\",\\"\\\\n\\",\\"```\\\\n\\",\\" Part 3/Data/\\\\n\\",\\"   \\u251c\\u2500\\u2500 monet_jpg/   (REAL Monet)          = domain A real\\\\n\\",\\"   \\u251c\\u2500\\u2500 photo_jpg/   (REAL Photo)          = domain B real\\\\n\\",\\"   \\u251c\\u2500\\u2500 pred_A2B/    (GEN Monet -> Photo)  = generated B\\\\n\\",\\"   \\u2514\\u2500\\u2500 pred_B2A/    (GEN Photo -> Monet)  = generated A\\\\n\\",\\"```\\\\n\\",\\"\\\\n\\",\\"### The below script computes the average in both directions of:\\\\n\\",\\"- `FID_A2B, MiFID_A2B`:  real photos  vs generated photos (pred_A2B)\\\\n\\",\\"- `FID_B2A, MiFID_B2A`:  real monet   vs generated monet  (pred_B2A)\\\\n\\",\\"\\\\n\\"],\\"metadata\\":{\\"id\\":\\"fFHaVTzpf1ug\\"}},{\\"cell_type\\":\\"code\\",\\"source\\":[\\"# !pip -q install scipy pillow tqdm\\"],\\"metadata\\":{\\"id\\":\\"H27ifzWYWg8U\\",\\"executionInfo\\":{\\"status\\":\\"ok\\",\\"timestamp\\":1770675988561,\\"user_tz\\":480,\\"elapsed\\":2,\\"user\\":{\\"displayName\\":\\"Sadhvi Singh\\",\\"userId\\":\\"18140026044236769529\\"}}},\\"execution_count\\":1,\\"outputs\\":[]},{\\"cell_type\\":\\"code\\",\\"source\\":[\\"import os, glob\\\\n\\",\\"import numpy as np\\\\n\\",\\"from PIL import Image\\\\n\\",\\"from tqdm import tqdm\\\\n\\",\\"\\\\n\\",\\"import torch\\\\n\\",\\"import torch.nn as nn\\\\n\\",\\"import torchvision.transforms as T\\\\n\\",\\"import torchvision.models as models\\\\n\\",\\"\\\\n\\",\\"import scipy.linalg\\\\n\\",\\"from scipy.spatial.distance import cosine\\"],\\"metadata\\":{\\"id\\":\\"TSIF3RXaWi87\\",\\"executionInfo\\":{\\"status\\":\\"ok\\",\\"timestamp\\":1770675990470,\\"user_tz\\":480,\\"elapsed\\":1908,\\"user\\":{\\"displayName\\":\\"Sadhvi Singh\\",\\"userId\\":\\"18140026044236769529\\"}}},\\"execution_count\\":2,\\"outputs\\":[]},{\\"cell_type\\":\\"code\\",\\"source\\":[\\"# Config: set your root\\\\n\\",\\"BASE = \\\\\\"Part 3/Data\\\\\\"\\\\n\\",\\"\\\\n\\",\\"REAL_MONET = os.path.join(BASE, \\\\\\"monet_jpg\\\\\\")   # real domain A\\\\n\\",\\"REAL_PHOTO = os.path.join(BASE, \\\\\\"photo_jpg\\\\\\")   # real domain B\\\\n\\",\\"GEN_A2B    = os.path.join(BASE, \\\\\\"pred_A2B\\\\\\")    # Monet -> Photo #You can modify the patha s per your directory\\\\n\\",\\"GEN_B2A    = os.path.join(BASE, \\\\\\"pred_B2A\\\\\\")    # Photo -> Monet #You can modify the patha s per your directory\\\\n\\",\\"\\\\n\\",\\"# Limit to N images per set\\\\n\\",\\"# Set to None to use all images in each folder\\\\n\\",\\"N_EVAL = 300\\\\n\\",\\"\\\\n\\",\\"BATCH_SIZE = 32\\\\n\\",\\"device = torch.device(\\\\\\"cuda\\\\\\" if torch.cuda.is_available() else \\\\\\"cpu\\\\\\")\\\\n\\",\\"print(\\\\\\"Device:\\\\\\", device)\\"],\\"metadata\\":{\\"colab\\":{\\"base_uri\\":\\"https://localhost:8080/\\"},\\"id\\":\\"tJbgErmiWkuL\\",\\"executionInfo\\":{\\"status\\":\\"ok\\",\\"timestamp\\":1770675990488,\\"user_tz\\":480,\\"elapsed\\":6,\\"user\\":{\\"displayName\\":\\"Sadhvi Singh\\",\\"userId\\":\\"18140026044236769529\\"}},\\"outputId\\":\\"ff8c684d-62a0-4a71-ee9f-146d5fe6bd02\\"},\\"execution_count\\":3,\\"outputs\\":[{\\"output_type\\":\\"stream\\",\\"name\\":\\"stdout\\",\\"text\\":[\\"Device: cuda\\\\n\\"]}]},{\\"cell_type\\":\\"code\\",\\"source\\":[\\"def list_images(folder):\\\\n\\",\\"    exts = (\\\\\\".jpg\\\\\\", \\\\\\".jpeg\\\\\\", \\\\\\".png\\\\\\")\\\\n\\",\\"    paths = []\\\\n\\",\\"    for ext in exts:\\\\n\\",\\"        paths.extend(glob.glob(os.path.join(folder, f\\\\\\"*{ext}\\\\\\")))\\\\n\\",\\"        paths.extend(glob.glob(os.path.join(folder, f\\\\\\"*{ext.upper()}\\\\\\")))\\\\n\\",\\"    paths = sorted(list(set(paths)))\\\\n\\",\\"    return paths\\\\n\\",\\"\\\\n\\",\\"def take_n(paths, n):\\\\n\\",\\"    if n is None:\\\\n\\",\\"        return paths\\\\n\\",\\"    return paths[:min(n, len(paths))]\\"],\\"metadata\\":{\\"id\\":\\"5PnAnnkFWtJb\\",\\"executionInfo\\":{\\"status\\":\\"ok\\",\\"timestamp\\":1770675990496,\\"user_tz\\":480,\\"elapsed\\":7,\\"user\\":{\\"displayName\\":\\"Sadhvi Singh\\",\\"userId\\":\\"18140026044236769529\\"}}},\\"execution_count\\":4,\\"outputs\\":[]},{\\"cell_type\\":\\"code\\",\\"source\\":[\\"# Inception feature extractor\\\\n\\",\\"def get_inception_model():\\\\n\\",\\"    inception = models.inception_v3(\\\\n\\",\\"        weights=models.Inception_V3_Weights.IMAGENET1K_V1,\\\\n\\",\\"        transform_input=False\\\\n\\",\\"    )\\\\n\\",\\"    inception.fc = nn.Identity()\\\\n\\",\\"    inception.to(device)\\\\n\\",\\"    inception.eval()\\\\n\\",\\"    return inception\\\\n\\",\\"\\\\n\\",\\"INCEPTION_TF = T.Compose([\\\\n\\",\\"    T.Resize(299),\\\\n\\",\\"    T.CenterCrop(299),\\\\n\\",\\"    T.ToTensor(),\\\\n\\",\\"    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))\\\\n\\",\\"])\\\\n\\",\\"\\\\n\\",\\"def load_batch(paths):\\\\n\\",\\"    imgs = []\\\\n\\",\\"    for p in paths:\\\\n\\",\\"        img = Image.open(p).convert(\\\\\\"RGB\\\\\\")\\\\n\\",\\"        imgs.append(INCEPTION_TF(img))\\\\n\\",\\"    return torch.stack(imgs, dim=0)\\\\n\\",\\"\\\\n\\",\\"@torch.no_grad()\\\\n\\",\\"def get_activations(model, image_paths, batch_size=32):\\\\n\\",\\"    feats = []\\\\n\\",\\"    for i in tqdm(range(0, len(image_paths), batch_size), desc=\\\\\\"Inception activations\\\\\\"):\\\\n\\",\\"        batch_paths = image_paths[i:i+batch_size]\\\\n\\",\\"        x = load_batch(batch_paths).to(device)\\\\n\\",\\"        f = model(x).detach().cpu().numpy()\\\\n\\",\\"        feats.append(f)\\\\n\\",\\"    return np.concatenate(feats, axis=0)\\"],\\"metadata\\":{\\"id\\":\\"QuQbxliWWwYj\\",\\"executionInfo\\":{\\"status\\":\\"ok\\",\\"timestamp\\":1770675990497,\\"user_tz\\":480,\\"elapsed\\":0,\\"user\\":{\\"displayName\\":\\"Sadhvi Singh\\",\\"userId\\":\\"18140026044236769529\\"}}},\\"execution_count\\":5,\\"outputs\\":[]},{\\"cell_type\\":\\"code\\",\\"source\\":[\\"def frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):\\\\n\\",\\"    covmean, _ = scipy.linalg.sqrtm(sigma1.dot(sigma2), disp=False)\\\\n\\",\\"    if not np.isfinite(covmean).all():\\\\n\\",\\"        offset = np.eye(sigma1.shape[0]) * eps\\\\n\\",\\"        covmean = scipy.linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))\\\\n\\",\\"\\\\n\\",\\"    if np.iscomplexobj(covmean):\\\\n\\",\\"        covmean = covmean.real\\\\n\\",\\"\\\\n\\",\\"    diff = mu1 - mu2\\\\n\\",\\"    return float(diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean))\\"],\\"metadata\\":{\\"id\\":\\"OSe-p6syW0CT\\",\\"executionInfo\\":{\\"status\\":\\"ok\\",\\"timestamp\\":1770675990503,\\"user_tz\\":480,\\"elapsed\\":5,\\"user\\":{\\"displayName\\":\\"Sadhvi Singh\\",\\"userId\\":\\"18140026044236769529\\"}}},\\"execution_count\\":6,\\"outputs\\":[]},{\\"cell_type\\":\\"code\\",\\"source\\":[\\"def calculate_fid_mifid(real_paths, gen_paths, batch_size=32, subsample_to_match=True):\\\\n\\",\\"    # For fair comparison, match counts\\\\n\\",\\"    real_paths = sorted(real_paths)\\\\n\\",\\"    gen_paths  = sorted(gen_paths)\\\\n\\",\\"\\\\n\\",\\"    if subsample_to_match:\\\\n\\",\\"        n = min(len(real_paths), len(gen_paths))\\\\n\\",\\"        real_paths = real_paths[:n]\\\\n\\",\\"        gen_paths  = gen_paths[:n]\\\\n\\",\\"\\\\n\\",\\"    model = get_inception_model()\\\\n\\",\\"\\\\n\\",\\"    real_act = get_activations(model, real_paths, batch_size=batch_size)\\\\n\\",\\"    gen_act  = get_activations(model, gen_paths,  batch_size=batch_size)\\\\n\\",\\"\\\\n\\",\\"    mu_r, sig_r = real_act.mean(axis=0), np.cov(real_act, rowvar=False)\\\\n\\",\\"    mu_g, sig_g = gen_act.mean(axis=0),  np.cov(gen_act,  rowvar=False)\\\\n\\",\\"\\\\n\\",\\"    fid = frechet_distance(mu_r, sig_r, mu_g, sig_g)\\\\n\\",\\"\\\\n\\",\\"    # mean cosine distance between feature vectors,\\\\n\\",\\"    # paired by index after subsampling/matching\\\\n\\",\\"    m = min(len(real_act), len(gen_act))\\\\n\\",\\"    cos_dists = [cosine(real_act[i], gen_act[i]) for i in range(m)]\\\\n\\",\\"    mifid = float(np.mean(cos_dists))\\\\n\\",\\"\\\\n\\",\\"    return fid, mifid\\"],\\"metadata\\":{\\"id\\":\\"6nlkYmNnW3J7\\",\\"executionInfo\\":{\\"status\\":\\"ok\\",\\"timestamp\\":1770675990512,\\"user_tz\\":480,\\"elapsed\\":8,\\"user\\":{\\"displayName\\":\\"Sadhvi Singh\\",\\"userId\\":\\"18140026044236769529\\"}}},\\"execution_count\\":7,\\"outputs\\":[]},{\\"cell_type\\":\\"code\\",\\"source\\":[\\"for d in [REAL_MONET, REAL_PHOTO, GEN_A2B, GEN_B2A]:\\\\n\\",\\"    assert os.path.isdir(d), f\\\\\\"Missing folder: {d}\\\\\\"\\\\n\\",\\"\\\\n\\",\\"real_monet = take_n(list_images(REAL_MONET), N_EVAL)\\\\n\\",\\"real_photo = take_n(list_images(REAL_PHOTO), N_EVAL)\\\\n\\",\\"gen_a2b    = take_n(list_images(GEN_A2B),    N_EVAL)  # Monet->Photo (generated photos)\\\\n\\",\\"gen_b2a    = take_n(list_images(GEN_B2A),    N_EVAL)  # Photo->Monet (generated monet)\\\\n\\",\\"\\\\n\\",\\"print(\\\\\\"\\\\\\\\nCounts (after N_EVAL cap):\\\\\\")\\\\n\\",\\"print(\\\\\\"Real Monet:\\\\\\", len(real_monet), \\\\\\" | Gen Monet (B2A):\\\\\\", len(gen_b2a))\\\\n\\",\\"print(\\\\\\"Real Photo:\\\\\\", len(real_photo), \\\\\\" | Gen Photo (A2B):\\\\\\", len(gen_a2b))\\"],\\"metadata\\":{\\"id\\":\\"8jxPwxdnW83r\\"},\\"execution_count\\":null,\\"outputs\\":[]},{\\"cell_type\\":\\"code\\",\\"source\\":[\\"print(\\\\\\"\\\\\\\\n Evaluating Photo -> Monet (B2A) \\\\\\")\\\\n\\",\\"# Ground truth = real Monet; Generated = pred_B2A (generated Monet-like)\\\\n\\",\\"fid_B2A, mifid_B2A = calculate_fid_mifid(real_monet, gen_b2a, batch_size=BATCH_SIZE)\\\\n\\",\\"print(f\\\\\\"[Photo->Monet] FID={fid_B2A:.3f}  MiFID={mifid_B2A:.4f}\\\\\\")\\\\n\\",\\"\\\\n\\",\\"print(\\\\\\"\\\\\\\\n Evaluating Monet -> Photo (A2B) \\\\\\")\\\\n\\",\\"# Ground truth = real Photo; Generated = pred_A2B (generated Photo-like)\\\\n\\",\\"fid_A2B, mifid_A2B = calculate_fid_mifid(real_photo, gen_a2b, batch_size=BATCH_SIZE)\\\\n\\",\\"print(f\\\\\\"[Monet->Photo] FID={fid_A2B:.3f}  MiFID={mifid_A2B:.4f}\\\\\\")\\\\n\\"],\\"metadata\\":{\\"id\\":\\"ahV7SQ2OXFaT\\"},\\"execution_count\\":null,\\"outputs\\":[]},{\\"cell_type\\":\\"code\\",\\"source\\":[\\"import pandas as pd\\\\n\\",\\"\\\\n\\",\\"sub_fid  = (fid_A2B + fid_B2A) / 2\\\\n\\",\\"sub_mifid = (mifid_A2B + mifid_B2A) / 2\\\\n\\",\\"\\\\n\\",\\"submission = pd.DataFrame([{\\\\n\\",\\"    \\\\\\"ID\\\\\\": 1,\\\\n\\",\\"    \\\\\\"FID\\\\\\": float(sub_fid),\\\\n\\",\\"    \\\\\\"MiFID\\\\\\": float(sub_mifid)}])\\\\n\\",\\"\\\\n\\",\\"submission.to_csv(\\\\\\"submission.csv\\\\\\", index=False)\\\\n\\",\\"print(submission)\\"],\\"metadata\\":{\\"id\\":\\"J0zvgKMdYo4r\\"},\\"execution_count\\":null,\\"outputs\\":[]}]}"}')
conflicts = [PROJECT_ROOT/relative for relative, content in FILES.items()
             if (PROJECT_ROOT/relative).exists() and (PROJECT_ROOT/relative).read_text() != content]
if conflicts:
    raise RuntimeError(f"Existing different code: {conflicts[0]}. Preserve this run and use a fresh folder for the updated bootstrap.")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
for relative, content in FILES.items():
    target = PROJECT_ROOT/relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.write_text(content)
print("Project ready:", PROJECT_ROOT)


# DATA266 Task 3 — CycleGAN lab runner

**A = Monet, B = Photo.** This notebook orchestrates the sibling Python scripts. It is not a standalone replacement for the project ZIP.

Run in lab Jupyter, VS Code, or Colab connected to the **lab machine's local runtime**. Keep the entire extracted repository on that machine. Git is not required there.

Read the member README before full training. No assignment results exist until you run this notebook on the permitted dataset. Smoke-test data and ratings are synthetic and must never appear as lab results.

## 1. Locate the extracted project
Edit `PROJECT_ROOT` if automatic detection fails. Paths belong to this runtime, not your laptop. For Google-hosted Colab, first copy/extract the project into its runtime. Opening the notebook alone does not copy sibling scripts.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, signal, shutil, time

# If needed replace None with Path("the extracted project folder") on THIS machine.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd()/"data266-lab1-pair17"]
    PROJECT_ROOT = next((p for p in candidates if (p/"task3_gan"/"Pramod_Dindukurthi"/"src"/"train.py").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Set PROJECT_ROOT above to the folder containing task3_gan, then rerun this cell.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
MEMBER = PROJECT_ROOT/"task3_gan"/"Pramod_Dindukurthi"
SRC = MEMBER/"src"
os.chdir(MEMBER)
print("Project found:", PROJECT_ROOT)
print("Kernel Python:", sys.executable)


## 2. Verify PyTorch, then install evaluation dependencies
Use the lab's matching CUDA PyTorch/torchvision pair. If this cell cannot import them, select the correct lab kernel or follow the lab's installation instructions. Installing requirements does not configure the NVIDIA driver.


In [ ]:
import torch, torchvision
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda, "available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))
else:
    print("CPU smoke tests work, but do not start full CycleGAN training until CUDA is available.")


In [ ]:
# Run once in a new environment. Restart the kernel if an already-imported package changes.
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(MEMBER/"requirements.txt")], check=True)


## 3. Runtime paths and provenance
Set `DATA_ROOT` to the parent of `monet_jpg/` and `photo_jpg/`. Set `BACKUP_DIR` to a **persistent** location that survives the lab reservation.

For a second experiment use a new `RUN_DIR`; never overwrite the baseline. Saving the notebook does not save checkpoints. With a local runtime, normal Colab `drive.mount()` is unavailable. Use the lab's supported persistent storage. With a Google-hosted runtime, mount Drive separately and set the backup path there.


In [ ]:
DATA_ROOT = PROJECT_ROOT/"task3_gan"/"data"
RUN_DIR = MEMBER                     # matches required member folder layout
BACKUP_DIR = None                    # REQUIRED before full training: Path("persistent destination")
CONFIG_NAME = "baseline"             # baseline or resizeconv
CODE_VERSION = "uncommitted"         # paste laptop commit SHA or a code-bundle label
RUN_ID = "pramod_baseline_seed42"     # unique label for team evidence collection
TRAIN_HOURS = 12.0                   # full learning-rate schedule, including later continuation
SESSION_HOURS = 4.0                  # this phase only; evaluate after four hours, then use 8.0
NUM_WORKERS = 2                      # use 0 if multiprocessing fails on the lab environment
BATCH_SIZE = 1
RUN_DIR = Path(RUN_DIR).expanduser().resolve()
DATA_ROOT = Path(DATA_ROOT).expanduser().resolve()
if BACKUP_DIR is not None:
    BACKUP_DIR = Path(BACKUP_DIR).expanduser().resolve()
SPLITS = RUN_DIR/"data_processed"/"splits.json"
for folder in ["monet_jpg", "photo_jpg"]:
    assert (DATA_ROOT/folder).is_dir(), f"Missing {DATA_ROOT/folder}"
print("Data:", DATA_ROOT, "Run:", RUN_DIR, "Backup:", BACKUP_DIR)


In [ ]:
def launch(script, *arguments):
    """Run in this kernel's Python environment; stream output and propagate failures."""
    command = [sys.executable, "-u", str(script), *map(str, arguments)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, cwd=MEMBER)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        print("Stop requested. Trainer will finish a complete step and save; wait for it to exit.")
        process.wait()
        raise
    if status:
        raise RuntimeError(f"Command failed with exit code {status}; inspect the output above.")

launch(SRC/"preflight.py", "--data-root", DATA_ROOT)


## 4. Smoke test on CPU
Tests both architectures, cycle gradients, exact save/reload/resume against uninterrupted training, image export, metric mathematics and audit-sheet processing. This takes no training dataset and produces only temporary synthetic evidence.


In [ ]:
launch(SRC/"smoke_test.py")


## 5. Audit images and freeze splits
Run once. Reuse the same split manifest for every comparable experiment. Exact decoded-pixel duplicates are excluded before splitting; visually inspect possible near duplicates separately. Do not put official hidden-test images into the development folders.
The default 15% validation split supports the planned 15 examples per direction only if each domain has enough images.


In [ ]:
if SPLITS.exists():
    print("Reusing existing split manifest:", SPLITS)
else:
    launch(SRC/"data.py", "--data-root", DATA_ROOT, "--output", SPLITS, "--seed", 42, "--val-fraction", 0.15)
split_info = json.loads(SPLITS.read_text())
for d in ["A", "B"]:
    part = split_info["domains"][d]
    print(d, "train", len(part["train"]), "validation", len(part["val"]), "duplicates", len(part["duplicates"]))
    if len(part["val"]) < 15:
        raise RuntimeError("The planned audit needs 15 validation samples per domain. Decide a suitable split before training.")


## 6. Short GPU throughput benchmark
This is a separate disposable run, never the assignment baseline. It measures 30 updates using your actual model/data. The first few updates include warm-up effects. Do not compare its image quality or scores as if it were a trained model.


In [ ]:
import yaml, csv, statistics
assert torch.cuda.is_available(), "Select a CUDA-capable lab kernel before benchmarking."
base_config = yaml.safe_load((MEMBER/"configs"/(CONFIG_NAME+".yaml")).read_text())
base_config["train"].update(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
BENCH_DIR = RUN_DIR/"data_processed"/("benchmark_"+CONFIG_NAME)
BENCH_DIR.mkdir(parents=True, exist_ok=True)
BENCH_CONFIG = BENCH_DIR/"benchmark.yaml"
if not (BENCH_DIR/"training_summary.json").exists():
    BENCH_CONFIG.write_text(yaml.safe_dump(base_config))
    launch(SRC/"train.py", "--config", BENCH_CONFIG, "--data-root", DATA_ROOT,
           "--splits", SPLITS, "--run-dir", BENCH_DIR, "--device", "cuda", "--stop-after", 30,
           "--code-version", CODE_VERSION)
else:
    assert yaml.safe_load(BENCH_CONFIG.read_text()) == base_config, "Benchmark config changed; choose a fresh benchmark directory."
    print("Reusing completed benchmark.")
history_file = sorted((BENCH_DIR/"reproducibility"/"raw_logs").glob("history_*.csv"))[-1]
with open(history_file) as f:
    benchmark_rows = list(csv.DictReader(f))
seconds_per_step = statistics.median(float(r["step_seconds"]) for r in benchmark_rows[-20:])
suggested_steps = max(2, int(TRAIN_HOURS*3600*0.85/seconds_per_step))
print("Median seconds/step:", seconds_per_step)
print("Suggested total steps including 15% overhead allowance:", suggested_steps)


## 7. Freeze the training configuration and verify backups
Review the displayed architecture, objective and estimated duration. The learning rate decays over the planned TOTAL steps. Resume must keep this exact config. Increasing steps after starting would change the learning-rate trajectory and is deliberately rejected.


In [ ]:
RUNTIME_CONFIG = RUN_DIR/"data_processed"/"runtime_config.yaml"
if RUNTIME_CONFIG.exists():
    train_config = yaml.safe_load(RUNTIME_CONFIG.read_text())
    print("Existing runtime configuration retained for exact resume.")
    if train_config["train"]["max_hours"] != TRAIN_HOURS + 0.5:
        raise RuntimeError("Existing schedule uses a different time budget. Use a fresh RUN_DIR for the overnight experiment, or restore TRAIN_HOURS to match the existing schedule for exact resume.")
else:
    train_config = base_config.copy()
    train_config["train"] = base_config["train"].copy()
    train_config["train"].update(max_steps=suggested_steps, decay_start=suggested_steps//2,
                                 max_hours=TRAIN_HOURS+0.5)
    RUNTIME_CONFIG.parent.mkdir(parents=True, exist_ok=True)
    RUNTIME_CONFIG.write_text(yaml.safe_dump(train_config))
print(yaml.safe_dump(train_config))
if BACKUP_DIR is None:
    raise RuntimeError("Set BACKUP_DIR to persistent storage in cell 3, then rerun this cell.")
assert BACKUP_DIR != RUN_DIR and RUN_DIR not in BACKUP_DIR.parents, "Backup must be outside the run directory to avoid recursive copying."
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
probe = BACKUP_DIR/"task3_write_check.txt"
probe.write_text("Persistent backup write check")
assert probe.read_text() == "Persistent backup write check"
print("Backup directory is writable. Confirm it survives the lab session:", BACKUP_DIR)


## 8. Train four hours, evaluate, then resume
Set `START_TRAINING=True` after reviewing the benchmark and frozen configuration.
The default 12-hour budget is adjustable; it is not a claim about optimal convergence.
`BACKGROUND_TRAINING=True` launches a separate process with disk logs and checkpoints.
Do not also run foreground training for the same run directory.

Before leaving, check the status cell, confirm an actual checkpoint and backup exist,
and ensure the lab machine stays awake and your session will not be terminated.
Closing the notebook tab is not a shutdown request for this background process, but
lab policies, machine restarts, or process cleanup can still terminate it.
Notebook interruption does **not** stop a detached job.
Run sections 9 onward only after training has stopped; do not use Run All.

**First phase:** keep `TRAIN_HOURS=12.0` for the full schedule and launch with
`SESSION_HOURS=4.0`. It pauses after a complete optimizer step around four hours
and saves the entire training state. Wait for background status `finished`.
Run sections 9-12 to freeze and evaluate this checkpoint and make a fallback
submission CSV using the instructor evaluator. Preserve this snapshot and CSV.

**Continuation:** if the model is healthy, change `SESSION_HOURS` in this launch
cell to `8.0` and rerun it. The latest checkpoint is resumed automatically with
the same models, optimizer state, replay pools, EMA, RNG and full learning-rate
schedule. Do not change `TRAIN_HOURS`, architecture, losses or the frozen YAML.
Evaluate the new checkpoint again afterwards; retain the first-phase candidate.
A four-hour checkpoint is recoverable, but its quality still needs evaluation.


In [ ]:
START_TRAINING = False  # set True after reviewing cells 1-7
BACKGROUND_TRAINING = True
SESSION_HOURS = 4.0  # first phase; change to 8.0 AFTER evaluating to continue
if START_TRAINING:
    assert BACKUP_DIR is not None, "Persistent backup is required."
    command_args = ["--config", RUNTIME_CONFIG, "--data-root", DATA_ROOT,
                    "--splits", SPLITS, "--run-dir", RUN_DIR, "--device", "cuda",
                    "--backup-dir", BACKUP_DIR, "--code-version", CODE_VERSION,
                    "--session-hours", SESSION_HOURS]
    latest = RUN_DIR/"checkpoints"/"latest_checkpoint.pt"
    if latest.exists():
        command_args += ["--resume", latest]
    if BACKGROUND_TRAINING:
        launch(SRC/"background.py", "launch", "--run-dir", RUN_DIR, "--",
               sys.executable, "-u", str(SRC/"train.py"), *map(str, command_args))
    else:
        launch(SRC/"train.py", *command_args)
else:
    print("Training not started. Set START_TRAINING=True when ready.")


In [ ]:
# Rerun while training, and again when you return. This does not start a job.
launch(SRC/"background.py", "status", "--run-dir", RUN_DIR)
checkpoint = RUN_DIR/"checkpoints"/"latest_checkpoint.pt"
print("Latest checkpoint:", checkpoint, "exists:", checkpoint.exists())
if checkpoint.exists():
    print("Checkpoint modified:", time.ctime(checkpoint.stat().st_mtime))


## 9. Freeze an evaluation checkpoint
Choose a periodic checkpoint or copy the latest completed checkpoint. Compare candidate checkpoints and raw/EMA weights on the same validation split. Never select the model solely on low cycle loss. Full FID/KID and LPIPS require evaluation-weight downloads on first use.


In [ ]:
CANDIDATE_SOURCE = RUN_DIR/"checkpoints"/"latest_checkpoint.pt"
WEIGHTS = "ema"  # compare with "raw" using another export directory
assert CANDIDATE_SOURCE.exists(), "Complete a training segment first."
sys.path.insert(0, str(SRC))
from checkpointing import load_checkpoint, backup_file
from common import sha256
candidate = load_checkpoint(CANDIDATE_SOURCE)
step = candidate["step"]
SNAPSHOT = RUN_DIR/"checkpoints"/"evaluation"/f"step_{step:08d}.pt"
if SNAPSHOT.exists():
    assert sha256(SNAPSHOT) == sha256(CANDIDATE_SOURCE), "Conflicting snapshot; preserve both with distinct names."
else:
    backup_file(CANDIDATE_SOURCE, SNAPSHOT)
EXPORT_DIR = RUN_DIR/"outputs"/"evaluations"/f"step_{step:08d}_{WEIGHTS}"
if BACKUP_DIR is not None:
    backup_file(SNAPSHOT, BACKUP_DIR/"checkpoints"/"evaluation"/SNAPSHOT.name)
print("Snapshot:", SNAPSHOT, "Export:", EXPORT_DIR)
del candidate


In [ ]:
if not (EXPORT_DIR/"export_manifest.json").exists():
    launch(SRC/"infer.py", "--checkpoint", SNAPSHOT, "--data-root", DATA_ROOT,
           "--splits", SPLITS, "--output", EXPORT_DIR, "--weights", WEIGHTS,
           "--split", "val", "--device", "cuda")
else:
    print("Export exists; preserve it. Choose another directory for another candidate.")


In [ ]:
launch(MEMBER/"evaluate_local.py", "--export-dir", EXPORT_DIR, "--run-dir", RUN_DIR,
       "--device", "cuda", "--batch-size", 8)
launch(SRC/"report.py", "--run-dir", RUN_DIR, "--export-dir", EXPORT_DIR)
print((EXPORT_DIR/"full_metrics_report.csv").read_text())


## 10. Inspect and select
Review BOTH directions, sample counts, KID uncertainty, semantic preservation and failure grids. The local FID is not guaranteed to match Kaggle. Repeat cells 9 and evaluation for raw weights or another archived checkpoint if time allows.
Only after comparing candidates set `SELECT_THIS=True` and write a reason. Selection copies the entire fixed export and saves `best_model.pt`; it never chooses only attractive images.


In [ ]:
from IPython.display import display, Image as NotebookImage
preview_files = sorted((RUN_DIR/"outputs"/"plots").glob("preview_*.png"))
if preview_files:
    display(NotebookImage(filename=str(preview_files[-1]), width=768))
for filename in ["adversarial.png", "cycle_identity.png", "gradients.png", "learning_rate.png"]:
    path = RUN_DIR/"outputs"/"plots"/filename
    if path.exists(): display(NotebookImage(filename=str(path)))


In [ ]:
SELECT_THIS = False
SELECTION_REASON = ""  # actual validation comparison and visual findings
if SELECT_THIS:
    assert SELECTION_REASON.strip(), "Document why this candidate was selected."
    launch(SRC/"select_checkpoint.py", "--checkpoint", SNAPSHOT, "--export-dir", EXPORT_DIR,
           "--run-dir", RUN_DIR, "--reason", SELECTION_REASON)


## 11. Blinded human audit
Prepare only after choosing the model. Both raters receive the same packet with 30 fixed examples (15 per direction). Keep `private_key.json` private from raters. Each independently fills `rater_1.csv` or `rater_2.csv`. Higher artifact score means fewer artifacts.


In [ ]:
AUDIT_DIR = RUN_DIR/"outputs"/"human_audit"
if (RUN_DIR/"selection.json").exists() and not AUDIT_DIR.exists():
    selection = json.loads((RUN_DIR/"selection.json").read_text())
    assert selection["checkpoint_sha256"] == sha256(SNAPSHOT), "Audit must use the selected checkpoint."
    launch(SRC/"audit.py", "prepare", "--export-dir", EXPORT_DIR, "--output", AUDIT_DIR)
else:
    print("Select a checkpoint first, or reuse the already fixed audit packet.")


In [ ]:
RATINGS_COMPLETE = False
if RATINGS_COMPLETE:
    launch(SRC/"audit.py", "summarize", "--output", AUDIT_DIR)
    launch(SRC/"report.py", "--run-dir", RUN_DIR, "--export-dir", EXPORT_DIR)


## 12. Official Kaggle evaluator
The instructor evaluator is included as `src/evaluate_official.py`. It uses torchvision Inception (different from local diagnostic FID), the first 300 sorted files per folder, matching counts, and index-wise cosine distance. It writes `ID,FID,MiFID`. Do not rearrange or hand-pick outputs to improve the score. Keep fixed held-out metrics separately for the report.

After freezing and evaluating a candidate on validation, create a separate official
export. Use this only for the supplied first-300 scoring protocol and the approved
development source set; follow any separate instructor input-set requirements.
The deterministic prefix preserves the first 300 outputs of a full all-input export.
Full local validation still uses all validation samples and cycle images.

Run in a new cell after training has stopped:
```python
OFFICIAL_DIR = RUN_DIR/"outputs"/"official"/f"{EXPORT_DIR.name}_first300"
if not (OFFICIAL_DIR/"export_manifest.json").exists():
    launch(SRC/"infer.py", "--checkpoint", SNAPSHOT, "--data-root", DATA_ROOT,
           "--splits", SPLITS, "--output", OFFICIAL_DIR, "--weights", WEIGHTS,
           "--split", "all", "--max-per-domain", 300, "--predictions-only",
           "--device", "cuda")
OFFICIAL_CSV = RUN_DIR/f"submission_{EXPORT_DIR.name}.csv"
if not OFFICIAL_CSV.exists():
    launch(SRC/"evaluate_official.py", "--data-root", DATA_ROOT,
           "--export-dir", OFFICIAL_DIR, "--output", OFFICIAL_CSV, "--batch-size", 16)
print(OFFICIAL_CSV.read_text())
```
Each checkpoint gets a distinct CSV so the four-hour fallback stays intact.
Copy your chosen candidate CSV to `submission.csv` for the final repository.

The reference folders follow the instructor script and include training images; this official-protocol score is not held-out generalization evidence. Confirm any separate competition source-image requirements. No automatic upload is performed.

## 13. Collect evidence and back up the full run
Run after training stops. Save the executed notebook separately too.


In [ ]:
launch(SRC/"collect_evidence.py", "--run-dir", RUN_DIR, "--repo-root", PROJECT_ROOT, "--run-id", RUN_ID)
if BACKUP_DIR is not None:
    # Explicit include list avoids recursively copying the backup into itself.
    for name in ["checkpoints", "outputs", "reproducibility", "data_processed"]:
        source = RUN_DIR/name
        if source.exists(): shutil.copytree(source, BACKUP_DIR/name, dirs_exist_ok=True)
    for name in ["training_summary.json", "selection.json", "full_metrics_report.csv",
                 "metrics_report.csv", "results_draft.md", "submission.csv", "submission_provenance.json"]:
        source = RUN_DIR/name
        if source.exists(): shutil.copy2(source, BACKUP_DIR/name)
    latest = RUN_DIR/"checkpoints"/"latest_checkpoint.pt"
    if latest.exists():
        assert sha256(latest) == sha256(BACKUP_DIR/"checkpoints"/latest.name)
    print("Backup complete; latest checkpoint checksum verified. Save the executed notebook separately.")


## 14. Laptop commits and report
Bring back the executed notebook, source changes (if any), raw logs, manifests, metrics, plots and analysis. Keep large binaries in persistent storage with checkpoint hashes and retrieval instructions. Commit code before runs and results after runs; do not manufacture history.

Complete `results.md` and `failure_analysis.md` from real evidence. The generated `results_draft.md` is a scaffold, not finished scientific analysis. Coordinate evaluation settings with your teammate. Explain architecture, objectives, metric limitations and observed failures in the viva.


## 15. Quick official score for the chosen frozen snapshot
After training stops, rerun Section 9's **first cell** to update `SNAPSHOT`. This cell uses that snapshot, not the active checkpoint. It reuses matching official results or generates only 300 JPGs per direction and computes the instructor FID/MiFID and expected displayed Kaggle score. It does not replace the full local metrics or human audit.


In [ ]:
import runpy
quick_result = runpy.run_path(str(SRC/"quick_kaggle_score.py"),
    init_globals={"RUN_DIR": RUN_DIR, "DATA_ROOT": DATA_ROOT, "SNAPSHOT": SNAPSHOT, "WEIGHTS": WEIGHTS})
OFFICIAL_CSV = Path(quick_result["OFFICIAL_CSV"])


## 16. Downloadable completed-snapshot package
Run after the selected training segment has finished and Section 9's first cell has frozen it. Set `PACKAGE_STEP=None` to select the highest completed step, or set it to the printed step to archive that exact snapshot. The ZIP includes completed matching training-history segments, exact score CSVs, generated images, source notebooks and the frozen checkpoint. The manifest lists missing rubric evidence. The dataset is excluded; restore the same original dataset separately. No upload or email occurs.

Download the resulting file from `task3_gan/transfer/`. An executed notebook outside the member folder can be included by setting `EXECUTED_NOTEBOOK` to its saved path. The starter file in source is a clean runnable notebook, not a substitute for the executed notebook.


In [ ]:
PACKAGE_STEP = None  # default: latest completed phase; or enter the exact snapshot step
EXECUTED_NOTEBOOK = None  # optional Path to the SAVED executed notebook
package_args = ["--run-dir", RUN_DIR, "--full", "--weights", WEIGHTS]
if PACKAGE_STEP is not None:
    package_args += ["--step", PACKAGE_STEP]
if EXECUTED_NOTEBOOK is not None:
    package_args += ["--notebook", EXECUTED_NOTEBOOK]
launch(SRC/"package_completed_run.py", *package_args)
